# Generador de Dataset Sintético — Chulupi Lang (v1.1)

Notebook autocontenido y listo para **Google Colab** (sin dependencias externas obligatorias; usa solo la biblioteca estándar de Python y `pandas` opcionalmente para inspección).

### Objetivo del proyecto
Generar pares paralelos controlados **(lengua aglutinante artificial → español)** siguiendo fielmente la especificación **`Chulupi Lang v1.1`** basada en `guia_estructural_aymara_v3.md` (Hardman *et al.* 2001 + Gramática Oficial ILCNA–IPELC 2024):
- **Modo B**: raíces inventadas + morfología y palabras gramaticales reales del aymara normalizado (L2).
- **Modo A**: mismas raíces inventadas + morfología y palabras gramaticales inventadas con mapeo fijo desde B (preservando homofonía y morfofonología).
- **Español (ES)**: generado determinísticamente desde la misma **idea JSON**, garantizando coherencia semántica (`CH-406`) y unicidad de traducción (`CH-1013`).

### Contenido del notebook
1. **Alfabeto y Fonotáctica** (`CH-101` a `CH-209`)
2. **Inventario de Morfemas B/A y Palabras Gramaticales** (`CH-501` a `CH-521`, con verificación `CH-503`)
3. **Motor Morfofonológico y Etiquetador `[DERIVADA]`** (`CH-601` a `CH-710`)
4. **Gate de Regresión**: 102 casos con raíces reales (`CH-710`, igualdad B + paralelismo A) y 64 celdas de paradigma (§8)
5. **Generador de Raíces Inventadas** (`CH-301` a `CH-310`) con filtro Trie de palabras reales
6. **Léxico Inicial** (93 entradas semánticamente anotadas) + carga opcional de `lexico_sin_raices.csv`
7. **Conjugador Español y Constructor de Oraciones B/A/ES** (§9 y §10)
8. **Verificación de los 40 Casos Oficiales** (`C01`–`C40`, incluyendo los 17 `[ATESTIGUADO]` de L2)
9. **Validador de Ideas (`R1`–`R20`), Generador Estratificado con Pesos (§11) y Verificador `V01`–`V31` (§13)**
10. **Función Principal `generar_dataset_chulupi(n=1000)`** y generación del dataset inicial de **1.000 pares**
11. **Exportación y Descarga** (`.jsonl` oficial + `.tsv` para Modo B y Modo A + `lexico_chulupi.csv`)

> **Cómo usar en Colab:** haz clic en **Entorno de ejecución → Ejecutar todo** (`Ctrl+F9`). Al final se descargarán o guardarán los archivos del dataset de 1.000 pares.

## 1. Alfabeto y Fonotáctica (`CH-101` a `CH-209`)

In [ ]:
import csv
import hashlib
import io
import json
import math
import os
import random
import re
import unicodedata
from collections import Counter, defaultdict
from copy import deepcopy
from typing import Any, Dict, List, Optional, Set, Tuple

VERSION_SPEC = "1.1"
VERSION_LEXICO = "1.0"

# --- Alfabeto oficial (CH-101 a CH-106) ---
APOSTROFOS_CONVERTIR = ["’", "‘", "ʼ", "´", "`"]
VOCALES_CORTAS = {"a", "i", "u"}
VOCALES_LARGAS_MAP = {"a": "ä", "i": "ï", "u": "ü"}
VOCALES_LARGAS = set(VOCALES_LARGAS_MAP.values())
VOCALES_TODAS = VOCALES_CORTAS | VOCALES_LARGAS

ORDEN_SEGMENTACION = [
    "chh", "ch'", "ch", "ph", "p'", "th", "t'", "kh", "k'", "qh", "q'", "ll",
    "a", "i", "u", "ä", "ï", "ü",
    "p", "t", "k", "q", "s", "j", "x", "m", "n", "ñ", "l", "r", "w", "y",
]
GRAFEMAS_VALIDOS = set(ORDEN_SEGMENTACION)
CONSONANTES = GRAFEMAS_VALIDOS - VOCALES_TODAS

LARINGALIZADOS = {"ph", "th", "chh", "kh", "qh", "p'", "t'", "ch'", "k'", "q'"}
PUNTO_DE_ARTICULACION = {
    "p":  {"p", "ph", "p'"},
    "t":  {"t", "th", "t'"},
    "ch": {"ch", "chh", "ch'"},
    "k":  {"k", "kh", "k'"},
    "q":  {"q", "qh", "q'"},
}
ASPIRADA_DE = {"p": "ph", "t": "th", "ch": "chh", "k": "kh", "q": "qh"}
GLOTAL_DE = {"p": "p'", "t": "t'", "ch": "ch'", "k": "k'", "q": "q'"}
OCLUSIVAS_SIMPLES = {"p", "t", "ch", "k", "q"}

CODAS_PERMITIDAS = {"p", "t", "ch", "k", "q", "s", "j", "x", "m", "n", "l", "r", "w", "y"}
INICIALES_PROHIBIDAS = {"x", "r"}

REGEX_RAIZ_CLASES = re.compile(r"^C?V(C{1,2}V)+$")
REGEX_PALABRA_CLASES = re.compile(r"^C?[VL](C{1,6}[VL])*$")


def normalizar(texto: str) -> str:
    """CH-102, CH-103: NFC y apóstrofo interno U+0027."""
    t = unicodedata.normalize("NFC", texto)
    for ch in APOSTROFOS_CONVERTIR:
        t = t.replace(ch, "'")
    return t


def segmentar(palabra: str) -> List[str]:
    """CH-105: Segmentación voraz en grafemas del alfabeto aymara."""
    p = normalizar(palabra).lower()
    if not p:
        raise ValueError("Palabra vacía")
    grafemas: List[str] = []
    i, n = 0, len(p)
    while i < n:
        matched = False
        for g in ORDEN_SEGMENTACION:
            if p.startswith(g, i):
                grafemas.append(g)
                i += len(g)
                matched = True
                break
        if not matched:
            raise ValueError(f"Carácter/grafema prohibido en '{palabra}' pos {i}: '{p[i:]}'")
    return grafemas


def clases_de(grafemas: List[str]) -> str:
    return "".join("V" if g in VOCALES_CORTAS else ("L" if g in VOCALES_LARGAS else "C") for g in grafemas)


def es_coda_en_raiz(grafemas: List[str], i: int) -> bool:
    return grafemas[i] in CONSONANTES and i + 1 < len(grafemas) and grafemas[i + 1] in CONSONANTES


def es_raiz_valida(raiz: str) -> bool:
    """CH-201 a CH-209: Valida una raíz inventada."""
    try:
        g = segmentar(raiz)
    except ValueError:
        return False
    if any(x in VOCALES_LARGAS for x in g):  # CH-209
        return False
    if not REGEX_RAIZ_CLASES.match(clases_de(g)):  # CH-201
        return False
    if g[0] in INICIALES_PROHIBIDAS:  # CH-202
        return False
    if "lll" in raiz.lower():  # CH-207
        return False
    for i, gr in enumerate(g):
        if es_coda_en_raiz(g, i):
            if gr not in CODAS_PERMITIDAS:  # CH-206
                return False
            if gr == g[i + 1]:  # CH-206 sin geminadas
                return False
            if gr == "l" and g[i + 1] in {"l", "ll"}:  # CH-207
                return False
    lar = [x for x in g if x in LARINGALIZADOS]
    if len(set(lar)) > 1:  # CH-205 regla 2
        return False
    for i, gr in enumerate(g):
        if gr in LARINGALIZADOS and es_coda_en_raiz(g, i):  # CH-205 regla 1
            return False
    for serie in PUNTO_DE_ARTICULACION.values():  # CH-205 regla 3
        presentes = [x for x in g if x in serie]
        if len(presentes) >= 2 and len(set(presentes)) > 1:
            return False
    return True


def validar_palabra(palabra: str) -> bool:
    """CH-101..105 y CH-202..208 para cualquier palabra de salida (B o A)."""
    p = normalizar(palabra).lower()
    if "lll" in p:
        return False
    try:
        g = segmentar(p)
    except ValueError:
        return False
    if g[0] in INICIALES_PROHIBIDAS:
        return False
    return bool(REGEX_PALABRA_CLASES.match(clases_de(g)))


def contar_silabas(raiz: str) -> int:
    return sum(1 for x in segmentar(raiz) if x in VOCALES_TODAS)

print("Alfabeto y fonotáctica cargados correctamente.")

## 2. Inventario de Morfemas B/A y Palabras Gramaticales (`CH-501` a `CH-521`)

In [ ]:
SUFIJOS_LISTA = [
    # --- nombre ---
    {"id": "PL",        "glosa": "PL",        "b": "naka",  "a": "sulu",  "slot": "N1", "antes": "V", "despues": "V"},
    {"id": "POSS.1",    "glosa": "POSS.1",    "b": "ja",    "a": "lu",    "slot": "N2", "antes": "V", "despues": "V"},
    {"id": "POSS.2",    "glosa": "POSS.2",    "b": "ma",    "a": "qi",    "slot": "N2", "antes": "V", "despues": "V"},
    {"id": "POSS.3",    "glosa": "POSS.3",    "b": "pa",    "a": "chu",   "slot": "N2", "antes": "V", "despues": "V"},
    {"id": "POSS.4",    "glosa": "POSS.4",    "b": "sa",    "a": "khu",   "slot": "N2", "antes": "V", "despues": "V"},
    {"id": "POSR",      "glosa": "POSR",      "b": "ni",    "a": "phu",   "slot": "N3", "antes": "V", "despues": "V"},
    {"id": "GEN",       "glosa": "GEN",       "b": "na",    "a": "mu",    "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "LOC",       "glosa": "LOC",       "b": "na",    "a": "mu",    "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "ALL",       "glosa": "ALL",       "b": "ru",    "a": "llu",   "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "COMPL.RU",  "glosa": "COMPL.RU",  "b": "ru",    "a": "llu",   "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "ABL",       "glosa": "ABL",       "b": "ta",    "a": "ku",    "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "COM",       "glosa": "COM",       "b": "mpi",   "a": "nku",   "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "BEN",       "glosa": "BEN",       "b": "taki",  "a": "puli",  "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "UNTIL",     "glosa": "UNTIL",     "b": "kama",  "a": "chiwi", "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "CAUSE",     "glosa": "CAUSE",     "b": "layku", "a": "tiwqa", "slot": "N4", "antes": "V", "despues": "V"},
    {"id": "VBLZ.KA",   "glosa": "VBLZ.KA",   "b": "ka",    "a": "ñu",    "slot": "N5", "antes": "C", "despues": "-"},
    {"id": "VBLZ.LONG", "glosa": "VBLZ.LONG", "b": "¨",     "a": "¨",     "slot": "N5", "antes": "V", "despues": "-", "invisible_tras": ["POSR"]},
    # --- verbo ---
    {"id": "PL.V",   "glosa": "PL",      "b": "pxa",  "a": "mtu",  "slot": "V1", "antes": "V", "despues": "-"},
    {"id": "PROG",   "glosa": "PROG",    "b": "ska",  "a": "nchi", "slot": "V1", "antes": "V", "despues": "-"},
    {"id": "NEG",    "glosa": "INCOMPL", "b": "ka",   "a": "ñu",   "slot": "V1", "antes": "C", "despues": "-"},
    {"id": "REM.C",  "glosa": "REM.C",   "b": "yä",   "a": "jü",   "slot": "V4", "antes": "V", "despues": "-"},
    {"id": "REM.L",  "glosa": "REM.L",   "b": "tä",   "a": "wü",   "slot": "V4", "antes": "V", "despues": "-"},
    {"id": "P1",     "glosa": "1",       "b": "tha",  "a": "phi",  "slot": ["V5", "N6"], "antes": "C", "despues": "C"},
    {"id": "P2",     "glosa": "2",       "b": "ta",   "a": "ku",   "slot": ["V5", "N6"], "antes": "C", "despues": "V"},
    {"id": "P3",     "glosa": "3>3",     "b": "i",    "a": "u",    "slot": ["V5", "N6"], "antes": "C", "despues": "V"},
    {"id": "P4",     "glosa": "4",       "b": "tana", "a": "kilu", "slot": ["V5", "N6"], "antes": "C", "despues": "C"},
    {"id": "P3.REM", "glosa": "3",       "b": "na",   "a": "mu",   "slot": "V5",         "antes": "-", "despues": "V"},
    {"id": "T12",    "glosa": "1>2",     "b": "sma",  "a": "jmi",  "slot": "V5",         "antes": "C", "despues": "V"},
    {"id": "T21",    "glosa": "2>1",     "b": "ista", "a": "ulqa", "slot": "V5",         "antes": "C", "despues": "V"},
    {"id": "T31",    "glosa": "3>1",     "b": "itu",  "a": "uqu",  "slot": "V5",         "antes": "C", "despues": "V"},
    {"id": "T32",    "glosa": "3>2",     "b": "tama", "a": "kiqu", "slot": "V5",         "antes": "C", "despues": "C"},
    {"id": "T34",    "glosa": "3>4",     "b": "istu", "a": "ulqu", "slot": "V5",         "antes": "C", "despues": "V"},
    {"id": "FUT.1",  "glosa": "1>3.FUT", "b": "¨",    "a": "¨",    "slot": "V5",         "antes": "V", "despues": "V"},
    {"id": "FUT.2",  "glosa": "2>3.FUT", "b": "¨ta",  "a": "¨ku",  "slot": "V5",         "antes": "V", "despues": "V"},
    {"id": "FUT.3",  "glosa": "3>3.FUT", "b": "ni",   "a": "phu",  "slot": "V5",         "antes": "V", "despues": "V"},
    {"id": "FUT.4",  "glosa": "4>3.FUT", "b": "ñäni", "a": "thäqi","slot": "V5",         "antes": "V", "despues": "V"},
    # --- oracionales ---
    {"id": "AFF",    "glosa": "AFF",     "b": "wa",   "a": "yi",   "slot": ["N7", "V6"], "antes": "-", "despues": "-"},
    {"id": "TOP",    "glosa": "TOP",     "b": "xa",   "a": "xu",   "slot": "N7",         "antes": "-", "despues": "-"},
    {"id": "Q.YN",   "glosa": "Q.YN",    "b": "ti",   "a": "ji",   "slot": ["N7", "V6"], "antes": "-", "despues": "-"},
    {"id": "NEG.TI", "glosa": "NEG",     "b": "ti",   "a": "ji",   "slot": "V6",         "antes": "-", "despues": "-"},
    {"id": "Q.INF",  "glosa": "Q.INF",   "b": "sa",   "a": "khu",  "slot": "N7",         "antes": "-", "despues": "-"},
]
MORFEMAS: Dict[str, Dict[str, Any]] = {m["id"]: m for m in SUFIJOS_LISTA}

PALABRAS_GRAM_LISTA = [
    {"id": "PRON.1",    "b": "naya",   "a": "muqi",   "persona": 1},
    {"id": "PRON.1.PL", "b": "nänaka", "a": "müsulu", "persona": 1, "plural": True},
    {"id": "PRON.2",    "b": "juma",   "a": "tilu",   "persona": 2},
    {"id": "PRON.3",    "b": "jupa",   "a": "kaxi",   "persona": 3},
    {"id": "PRON.4",    "b": "jiwasa", "a": "wiqutu", "persona": 4},
    {"id": "NEG.PAL",   "b": "jani",   "a": "yuli"},
    {"id": "INT.QUIEN", "b": "khiti",  "a": "simu"},
    {"id": "INT.QUE",   "b": "kuna",   "a": "mulu"},
    {"id": "INT.DONDE", "b": "kawki",  "a": "yaqu"},
    {"id": "DEM.1",     "b": "aka",    "a": "ilmi"},
    {"id": "DEM.2",     "b": "uka",    "a": "ulmi"},
    {"id": "DEM.3",     "b": "khuri",  "a": "phiqa"},
    {"id": "NUM.3",     "b": "kimsa",  "a": "musqi",  "valor": 3},
    {"id": "NUM.4",     "b": "pusi",   "a": "tayli",  "valor": 4},
    {"id": "NUM.6",     "b": "suxta",  "a": "wichqa", "valor": 6},
    {"id": "NUM.10",    "b": "tunka",  "a": "qanti",  "valor": 10},
]
PALABRAS_GRAM: Dict[str, Dict[str, Any]] = {p["id"]: p for p in PALABRAS_GRAM_LISTA}
GRAM_POR_B: Dict[str, str] = {p["b"]: p["id"] for p in PALABRAS_GRAM_LISTA}


def verificar_tabla_modo_a() -> None:
    """CH-501, CH-502, CH-503: Comprueba biyección B↔A, sílabas, inicios y vocales largas."""
    b_to_a: Dict[str, str] = {}
    a_to_b: Dict[str, str] = {}
    for m in SUFIJOS_LISTA:
        fb, fa = m["b"], m["a"]
        if fb in b_to_a:
            assert b_to_a[fb] == fa, f"CH-501: homófono B '{fb}' mapea a dos A distintas"
        b_to_a[fb] = fa
        if fa in a_to_b:
            assert a_to_b[fa] == fb, f"CH-503: dos B distintas ('{a_to_b[fa]}', '{fb}') mapean a misma A '{fa}'"
        a_to_b[fa] = fb
        if fb == "¨":
            assert fa == "¨", "CH-502: alargamiento debe conservarse"
            continue
        sb, sa = fb.lstrip("¨"), fa.lstrip("¨")
        assert fb.startswith("¨") == fa.startswith("¨")
        gb, ga = segmentar(sb), segmentar(sa)
        cb, ca = clases_de(gb), clases_de(ga)
        assert cb.count("V") + cb.count("L") == ca.count("V") + ca.count("L"), f"CH-503 sílabas: {fb} vs {fa}"
        # Mismo tipo de inicio (número de C iniciales)
        c_ini_b = len(cb) - len(cb.lstrip("C"))
        c_ini_a = len(ca) - len(ca.lstrip("C"))
        assert c_ini_b == c_ini_a, f"CH-503 inicio: {fb} ({cb}) vs {fa} ({ca})"
        # Posición de vocal larga idéntica
        vocs_b = [c for c in cb if c in ("V", "L")]
        vocs_a = [c for c in ca if c in ("V", "L")]
        assert vocs_b == vocs_a, f"CH-502/503 vocal larga: {fb} vs {fa}"

verificar_tabla_modo_a()
print(f"Inventario cargado: {len(SUFIJOS_LISTA)} sufijos, {len(PALABRAS_GRAM_LISTA)} palabras gramaticales. CH-501..503 OK.")

## 3. Motor Morfofonológico y Etiquetador `[DERIVADA]` (`CH-601` a `CH-710`)

In [ ]:
REGEX_NP_NORMAL = re.compile(
    r"^RAIZ_N( PL)?( POSS\.[1-4])?( (GEN|LOC|ALL|COMPL\.RU|ABL|COM|BEN|UNTIL|CAUSE))?( (TOP|AFF|Q\.YN|Q\.INF))?$"
)
REGEX_ESTAR = re.compile(
    r"^RAIZ_N( PL)?( POSS\.[1-4])? LOC VBLZ\.KA (P1|P2|P3|P4)( (AFF|Q\.YN|Q\.INF))?$"
)
REGEX_TENER = re.compile(
    r"^RAIZ_N( PL)? POSR( VBLZ\.LONG (P1|P2|P4))?( (AFF|Q\.YN))?$"
)
REGEX_SER = re.compile(
    r"^RAIZ_N VBLZ\.LONG (P1|P2|P4)( (AFF|Q\.YN))?$"
)
REGEX_VERBO = re.compile(
    r"^RAIZ_V( (PL\.V|PROG|NEG))?( (P1|P2|P3|P4|T12|T21|T31|T32|T34)| (FUT\.[1-4])| (REM\.C|REM\.L) (P1|P2|P3\.REM|P4))( (AFF|Q\.YN|NEG\.TI))?$"
)


def validar_slots_palabra(es_verbo: bool, ids: List[str]) -> bool:
    """CH-601, CH-602: Comprueba plantilla de slots de una palabra."""
    if es_verbo:
        cad = " ".join(["RAIZ_V"] + ids)
        if not REGEX_VERBO.match(cad):
            return False
        if ("NEG" in ids) != ("NEG.TI" in ids):
            return False
        if "NEG" in ids and any(x in ids for x in ("REM.C", "REM.L")):
            return False
        if "PROG" in ids and any(x in ids for x in ("FUT.1", "FUT.2", "FUT.3", "FUT.4", "REM.C", "REM.L")):
            return False
        return True
    cad = " ".join(["RAIZ_N"] + ids)
    return bool(
        REGEX_NP_NORMAL.match(cad)
        or REGEX_ESTAR.match(cad)
        or REGEX_TENER.match(cad)
        or REGEX_SER.match(cad)
    )


def construir_palabra(base: str, ids: List[str], modo: str = "B", devolver_traza: bool = False) -> Any:
    """CH-701: Algoritmo morfofonológico unificado para Modo B y Modo A."""
    if base in PALABRAS_GRAM:
        forma0 = PALABRAS_GRAM[base]["b"] if modo == "B" else PALABRAS_GRAM[base]["a"]
    elif base in GRAM_POR_B:
        pg = PALABRAS_GRAM[GRAM_POR_B[base]]
        forma0 = pg["b"] if modo == "B" else pg["a"]
    else:
        forma0 = base

    g = segmentar(forma0)
    despues_prev = "V"
    id_prev = "RAIZ"
    protegida = False
    traza: List[str] = []

    for k, mid in enumerate(ids):
        m = MORFEMAS[mid]
        f = m["b"] if modo == "B" else m["a"]

        # 1. Alomorfo REM.C ante P3.REM (CH-511)
        if mid == "REM.C" and k + 1 < len(ids) and ids[k + 1] == "P3.REM":
            f = "¨"

        # 1b. VBLZ.LONG tras POSR (CH-705): no se escribe, protege vocal anterior
        if mid == "VBLZ.LONG" and id_prev == "POSR":
            protegida = True
            despues_prev = "V"
            id_prev = mid
            traza.append("proteger")
            continue

        # 2. Alargamiento (CH-704)
        if f.startswith("¨"):
            if g[-1] in VOCALES_CORTAS:
                g[-1] = VOCALES_LARGAS_MAP[g[-1]]
                traza.append("alargar")
            else:
                traza.append("larga-sin-cambio")
            resto = f[1:]
            if resto:
                g.extend(segmentar(resto))
            protegida = False
            despues_prev = m["despues"]
            id_prev = mid
            continue

        # 3. Borrado (CH-702, CH-703, CH-705)
        borrar = (m["antes"] == "C") or (m["antes"] == "-" and despues_prev == "C")
        if borrar:
            if protegida:
                traza.append("protegida-no-borra")
            elif g[-1] in VOCALES_CORTAS:
                g = g[:-1]
                traza.append("borrar")
            elif g[-1] in VOCALES_LARGAS:
                traza.append("larga-no-borra")
            else:
                traza.append("consonante-no-borra")
        else:
            traza.append("conservar")

        protegida = False

        # 4. Pegar sin reducir grupos (CH-706)
        g.extend(segmentar(f))
        despues_prev = m["despues"]
        id_prev = mid

    palabra = "".join(g)
    if not validar_palabra(palabra):
        raise ValueError(f"CH-709 palabra inválida: '{palabra}' ({base}, {ids}, modo={modo})")
    return (palabra, traza) if devolver_traza else palabra


# --- Detector de reglas [DERIVADA] (CH-702, Anexo A) ---
TRANSICIONES_CONSERVAN = {"T12", "T21", "T31", "T34"}
TODAS_TRANSICIONES = TRANSICIONES_CONSERVAN | {"T32"}
ORACIONALES = {"AFF", "Q.YN", "NEG.TI"}


def detectar_dv_palabra(ids: List[str]) -> List[str]:
    dvs: Set[str] = set()
    idset = set(ids)
    has_orac = bool(idset & ORACIONALES)
    has_ti = bool(idset & {"Q.YN", "NEG.TI"})
    has_rem = bool(idset & {"REM.C", "REM.L"})
    es_pnom = bool(idset & {"VBLZ.KA", "VBLZ.LONG", "POSR"})

    if "T32" in idset and has_orac:
        dvs.add("DV-01")
    if (idset & TRANSICIONES_CONSERVAN) and has_orac:
        dvs.add("DV-02")
    if has_ti and (idset & {"P2", "P3", "P4"}) and not has_rem and not es_pnom:
        dvs.add("DV-03")
    if (idset & {"FUT.1", "FUT.2"}) and has_orac:
        dvs.add("DV-04")
    if "FUT.4" in idset and has_ti:
        dvs.add("DV-04")
    if has_rem and has_orac:
        dvs.add("DV-05")
    if "PL.V" in idset and (idset & {"FUT.1", "FUT.3", "FUT.4", "REM.L"}):
        dvs.add("DV-06")
    if "PL.V" in idset and (idset & TODAS_TRANSICIONES):
        dvs.add("DV-07")
    if "NEG" in idset and not ("P1" in idset or "FUT.3" in idset):
        dvs.add("DV-08")
    if "PROG" in idset and "P3" not in idset:
        dvs.add("DV-09")
    if "VBLZ.KA" in idset and (idset & {"P2", "P3", "P4"}):
        dvs.add("DV-10")
    if "POSR" in idset and "VBLZ.LONG" in idset and (idset & {"P2", "P4"}):
        dvs.add("DV-11")
    if "VBLZ.LONG" in idset and "POSR" not in idset and (idset & {"P2", "P4"}):
        dvs.add("DV-12")
    if es_pnom and "Q.YN" in idset:
        dvs.add("DV-13")
    return sorted(dvs)

print("Motor morfofonológico y detector DV listos.")

## 4. Gate de Regresión: `CH-710` (102 raíces reales) y Paradigmas §8 (64 celdas)

Antes de generar cualquier dato sintético, la especificación exige comprobar que el motor:
1. Reproduce exactamente las **102 formas atestiguadas de L2** en Modo B (`CH-710`).
2. Toma exactamente la **misma secuencia de decisiones morfofonológicas** en Modo A (paralelismo estricto).
3. Reproduce las **64 celdas verbales** de `paradigmas.yaml` sobre `ch'ixra` en ambos modos.

In [ ]:
REGRESION_CH710 = [["sara", ["P1", "AFF"], "sarthwa"], ["manq'a", ["P2", "AFF"], "manq'tawa"], ["manq'a", ["P4", "AFF"], "manq'tanwa"], ["lura", ["P3", "AFF"], "luriwa"], ["lura", ["P1", "AFF"], "lurthwa"], ["lura", ["P2", "AFF"], "lurtawa"], ["juta", ["P1", "AFF"], "jutthwa"], ["manq'a", ["P1", "AFF"], "manq'thwa"], ["muna", ["P1"], "muntha"], ["apa", ["P2"], "apta"], ["sara", ["P1"], "sartha"], ["sara", ["P2"], "sarta"], ["sara", ["P3"], "sari"], ["sara", ["P4"], "sartana"], ["chura", ["P1"], "churtha"], ["chura", ["P3"], "churi"], ["manq'a", ["P3"], "manq'i"], ["awata", ["P3"], "awati"], ["irnaqa", ["P1"], "irnaqtha"], ["chura", ["T12"], "chursma"], ["chura", ["T21"], "churista"], ["chura", ["T31"], "churitu"], ["chura", ["T32"], "churtama"], ["chura", ["T34"], "churistu"], ["sara", ["FUT.1"], "sarä"], ["sara", ["FUT.2"], "saräta"], ["sara", ["FUT.3"], "sarani"], ["sara", ["FUT.4"], "sarañäni"], ["chura", ["FUT.1"], "churä"], ["chura", ["FUT.2"], "churäta"], ["chura", ["FUT.3"], "churani"], ["chura", ["FUT.4"], "churañäni"], ["juta", ["FUT.3"], "jutani"], ["juta", ["FUT.3", "Q.YN"], "jutaniti"], ["achu", ["FUT.3", "AFF"], "achuniwa"], ["lura", ["FUT.4", "AFF"], "lurañäniwa"], ["thuqhu", ["FUT.4"], "thuqhuñäni"], ["sara", ["REM.C", "P1"], "sarayätha"], ["sara", ["REM.C", "P2"], "sarayäta"], ["sara", ["REM.C", "P3.REM"], "saräna"], ["sara", ["REM.C", "P4"], "sarayätana"], ["muna", ["REM.C", "P1"], "munayätha"], ["muna", ["REM.C", "P3.REM"], "munäna"], ["sara", ["REM.L", "P1"], "saratätha"], ["sara", ["REM.L", "P2"], "saratäta"], ["sara", ["REM.L", "P3.REM"], "saratäna"], ["sara", ["REM.L", "P4"], "saratätana"], ["muna", ["REM.L", "P1"], "munatätha"], ["muna", ["REM.L", "P3.REM"], "munatäna"], ["sara", ["PL.V", "P1"], "sarapxtha"], ["sara", ["PL.V", "P3"], "sarapxi"], ["jikisi", ["PL.V", "P3"], "jikisipxi"], ["sara", ["PL.V", "FUT.2"], "sarapxäta"], ["sara", ["PL.V", "REM.C", "P1"], "sarapxayätha"], ["sara", ["PL.V", "REM.C", "P3.REM"], "sarapxäna"], ["sara", ["PROG", "P3"], "saraski"], ["iki", ["PROG", "P3"], "ikiski"], ["juta", ["NEG", "FUT.3", "NEG.TI"], "jutkaniti"], ["uta", ["LOC", "VBLZ.KA", "P1", "AFF"], "utankthwa"], ["uta", ["POSR", "VBLZ.LONG", "P1", "AFF"], "utanithwa"], ["masi", ["PL", "POSR", "VBLZ.LONG", "P1", "AFF"], "masinakanithwa"], ["uta", ["POSR", "AFF"], "utaniwa"], ["jach'a", ["VBLZ.LONG", "P1", "AFF"], "jach'äthwa"], ["uta", ["POSS.1"], "utaja"], ["phisi", ["POSS.2"], "phisima"], ["anu", ["POSS.3"], "anupa"], ["phuchha", ["POSS.4"], "phuchhasa"], ["anu", ["PL"], "anunaka"], ["uta", ["PL", "POSS.1", "LOC"], "utanakajana"], ["tata", ["POSS.1", "GEN"], "tatajana"], ["uta", ["POSS.3"], "utapa"], ["wawa", ["COMPL.RU"], "wawaru"], ["tata", ["POSS.2", "COMPL.RU"], "tatamaru"], ["marka", ["ALL"], "markaru"], ["uta", ["ALL", "AFF"], "utaruwa"], ["marka", ["ABL"], "markata"], ["marka", ["POSS.1", "ABL"], "markajata"], ["qala", ["ABL", "AFF"], "qalatawa"], ["pampa", ["LOC"], "pampana"], ["uka", ["LOC"], "ukana"], ["wawa", ["POSS.3", "BEN"], "wawapataki"], ["jila", ["POSS.1", "BEN", "AFF"], "jilajatakiwa"], ["tiwula", ["COM"], "tiwulampi"], ["wallata", ["COM", "AFF"], "wallatampiwa"], ["marka", ["UNTIL", "AFF"], "markakamawa"], ["uta", ["UNTIL"], "utakama"], ["qullu", ["UNTIL"], "qullukama"], ["juma", ["CAUSE", "AFF"], "jumalaykuwa"], ["kawki", ["ALL", "Q.INF"], "kawkirusa"], ["khiti", ["Q.INF"], "khitisa"], ["naya", ["TOP"], "nayaxa"], ["juma", ["TOP"], "jumaxa"], ["jiwasa", ["TOP"], "jiwasaxa"], ["jupa", ["PL", "TOP"], "jupanakaxa"], ["jani", ["AFF"], "janiwa"], ["anu", ["TOP"], "anuxa"], ["tata", ["TOP"], "tataxa"], ["awicha", ["TOP"], "awichaxa"], ["jilaqata", ["TOP"], "jilaqataxa"], ["jila", ["POSS.2", "TOP"], "jilamaxa"], ["mama", ["POSS.1", "AFF"], "mamajawa"], ["mara", ["TOP"], "maraxa"]]
PARADIGMAS_S8 = [[["P1"], "ch'ixrtha", "ch'ixrphi"], [["P2"], "ch'ixrta", "ch'ixrku"], [["P3"], "ch'ixri", "ch'ixru"], [["P4"], "ch'ixrtana", "ch'ixrkilu"], [["T12"], "ch'ixrsma", "ch'ixrjmi"], [["T21"], "ch'ixrista", "ch'ixrulqa"], [["T31"], "ch'ixritu", "ch'ixruqu"], [["T32"], "ch'ixrtama", "ch'ixrkiqu"], [["T34"], "ch'ixristu", "ch'ixrulqu"], [["FUT.1"], "ch'ixrä", "ch'ixrä"], [["FUT.2"], "ch'ixräta", "ch'ixräku"], [["FUT.3"], "ch'ixrani", "ch'ixraphu"], [["FUT.4"], "ch'ixrañäni", "ch'ixrathäqi"], [["REM.C", "P1"], "ch'ixrayätha", "ch'ixrajüphi"], [["REM.C", "P2"], "ch'ixrayäta", "ch'ixrajüku"], [["REM.C", "P3.REM"], "ch'ixräna", "ch'ixrämu"], [["REM.C", "P4"], "ch'ixrayätana", "ch'ixrajükilu"], [["REM.L", "P1"], "ch'ixratätha", "ch'ixrawüphi"], [["REM.L", "P2"], "ch'ixratäta", "ch'ixrawüku"], [["REM.L", "P3.REM"], "ch'ixratäna", "ch'ixrawümu"], [["REM.L", "P4"], "ch'ixratätana", "ch'ixrawükilu"], [["PL.V", "P1"], "ch'ixrapxtha", "ch'ixramtphi"], [["PL.V", "P2"], "ch'ixrapxta", "ch'ixramtku"], [["PL.V", "P3"], "ch'ixrapxi", "ch'ixramtu"], [["PL.V", "P4"], "ch'ixrapxtana", "ch'ixramtkilu"], [["PL.V", "T12"], "ch'ixrapxsma", "ch'ixramtjmi"], [["PL.V", "T21"], "ch'ixrapxista", "ch'ixramtulqa"], [["PL.V", "T31"], "ch'ixrapxitu", "ch'ixramtuqu"], [["PL.V", "T32"], "ch'ixrapxtama", "ch'ixramtkiqu"], [["PL.V", "T34"], "ch'ixrapxistu", "ch'ixramtulqu"], [["PL.V", "FUT.1"], "ch'ixrapxä", "ch'ixramtü"], [["PL.V", "FUT.2"], "ch'ixrapxäta", "ch'ixramtüku"], [["PL.V", "FUT.3"], "ch'ixrapxani", "ch'ixramtuphu"], [["PL.V", "FUT.4"], "ch'ixrapxañäni", "ch'ixramtuthäqi"], [["PL.V", "REM.C", "P1"], "ch'ixrapxayätha", "ch'ixramtujüphi"], [["PL.V", "REM.C", "P2"], "ch'ixrapxayäta", "ch'ixramtujüku"], [["PL.V", "REM.C", "P3.REM"], "ch'ixrapxäna", "ch'ixramtümu"], [["PL.V", "REM.C", "P4"], "ch'ixrapxayätana", "ch'ixramtujükilu"], [["PL.V", "REM.L", "P1"], "ch'ixrapxatätha", "ch'ixramtuwüphi"], [["PL.V", "REM.L", "P2"], "ch'ixrapxatäta", "ch'ixramtuwüku"], [["PL.V", "REM.L", "P3.REM"], "ch'ixrapxatäna", "ch'ixramtuwümu"], [["PL.V", "REM.L", "P4"], "ch'ixrapxatätana", "ch'ixramtuwükilu"], [["NEG", "P1", "NEG.TI"], "ch'ixrkthti", "ch'ixrñphji"], [["NEG", "P2", "NEG.TI"], "ch'ixrktati", "ch'ixrñkuji"], [["NEG", "P3", "NEG.TI"], "ch'ixrkiti", "ch'ixrñuji"], [["NEG", "P4", "NEG.TI"], "ch'ixrktanti", "ch'ixrñkilji"], [["NEG", "T12", "NEG.TI"], "ch'ixrksmati", "ch'ixrñjmiji"], [["NEG", "T21", "NEG.TI"], "ch'ixrkistati", "ch'ixrñulqaji"], [["NEG", "T31", "NEG.TI"], "ch'ixrkituti", "ch'ixrñuquji"], [["NEG", "T32", "NEG.TI"], "ch'ixrktamti", "ch'ixrñkiqji"], [["NEG", "T34", "NEG.TI"], "ch'ixrkistuti", "ch'ixrñulquji"], [["NEG", "FUT.1", "NEG.TI"], "ch'ixrkäti", "ch'ixrñüji"], [["NEG", "FUT.2", "NEG.TI"], "ch'ixrkätati", "ch'ixrñükuji"], [["NEG", "FUT.3", "NEG.TI"], "ch'ixrkaniti", "ch'ixrñuphuji"], [["NEG", "FUT.4", "NEG.TI"], "ch'ixrkañäniti", "ch'ixrñuthäqiji"], [["PROG", "P1"], "ch'ixrasktha", "ch'ixranchphi"], [["PROG", "P2"], "ch'ixraskta", "ch'ixranchku"], [["PROG", "P3"], "ch'ixraski", "ch'ixranchu"], [["PROG", "P4"], "ch'ixrasktana", "ch'ixranchkilu"], [["PROG", "T12"], "ch'ixrasksma", "ch'ixranchjmi"], [["PROG", "T21"], "ch'ixraskista", "ch'ixranchulqa"], [["PROG", "T31"], "ch'ixraskitu", "ch'ixranchuqu"], [["PROG", "T32"], "ch'ixrasktama", "ch'ixranchkiqu"], [["PROG", "T34"], "ch'ixraskistu", "ch'ixranchulqu"]]

# 1. Prueba CH-710 (102 casos)
for raiz, ids, esperado in REGRESION_CH710:
    b_out, traza_b = construir_palabra(raiz, ids, modo="B", devolver_traza=True)
    a_out, traza_a = construir_palabra(raiz, ids, modo="A", devolver_traza=True)
    assert b_out == esperado, f"Fallo CH-710 B: {raiz} + {ids} -> '{b_out}' != '{esperado}'"
    assert traza_b == traza_a, f"Fallo paralelismo A: {raiz} + {ids} -> {traza_b} != {traza_a}"

# 2. Prueba paradigmas §8 (64 celdas)
for ids, esp_b, esp_a in PARADIGMAS_S8:
    b_out = construir_palabra("ch'ixra", ids, modo="B")
    a_out = construir_palabra("ch'ixra", ids, modo="A")
    assert b_out == esp_b and a_out == esp_a, f"Fallo paradigma {ids}: ({b_out}, {a_out}) != ({esp_b}, {esp_a})"

print(f"OK! Regresión CH-710: {len(REGRESION_CH710)}/{len(REGRESION_CH710)} (igualdad B + paralelismo A).")
print(f"OK! Paradigmas §8:    {len(PARADIGMAS_S8)}/{len(PARADIGMAS_S8)} (B y A).")

## 5. Generador de Raíces Inventadas (`CH-301` a `CH-310`)

Incluye embebida la lista de **952 tokens reales de aymara** extraídos de `guia_estructural_aymara_v3.md` (`modo_filtro="guia"`, umbral de prefijo = 1).  
Cuando el equipo incorpore el lado aymara del *train* de **AmericasNLP**, basta pasar esa lista a `generar_raices(..., modo_filtro="corpus")` (umbral de prefijo = 5, `CH-306`).

In [ ]:
PALABRAS_REALES_GUIA = "acc achjitaspa achuniwa achuya achuyta aka akankakatti akankasktwa akankktti akax akaxa akch'awa akhama akhitan aksaruwa aksaxar alani alaniwaytha alaniña alantha alasaw alata alawi alañatakiw aliri aliriw aljkataña all alsuña alta am amuyt'am ana anakiristti ananukuyasinima anatarakiwa anatiri anma antukuxa anujama anuma anumaraki anunaka anupa anuxa apa apachita apanaqaña apani apaniwayaña apaniña apantaña apasiña apasuta apata apawayaña apayaniwayasipki apayaña apaña apañani apillasa apjaña apnaqaña appr apsta apsuña aptanim aptaña ar arktaña aru aruwa aski at awati awatiri away awayuru awichaxa awtut aymar aymara aymaras car cat catp caus causr ch'a ch'iyar ch'iyara ch'iyaraptaña ch'uju ch'uki ch'umphi ch'unuchiri ch'uq ch'uqi ch'uqirakikiti ch'uqisa ch'uqiw ch'uqiwa ch'usa ch'uxu ch'uxña ch'uñuwir cha challwitu chaqa chayaka chi chim china chini chiqa chiqan chisapana chispan chista chistu chitayna chitu chix chura churana churani churarapi churarapta churata churatawa churatayna churayta churañataki churañäni churi churista churistu churita churiti churitu churpachana churpachani churpachawa chursma chursta churtama churtha churthwa churtwa churxaña churä churäna chäka ciri citu citäta cplac csi cta ha ict ikikathati ikiski ikitayna ikïtatixa im imillacha imp in inas inasa inc incl int ipana iqiqu ir iri iriskasamana iriskata iriskatamana irisma irispan irista iristam irnaqtha irpa irpatam irpitani irpsu irpsutäta iskuylar ist ista istana istani istaspan istasta istpa istu istustu ita itana itani itasamana itasapana itasma itaspan itata itpa itu itutu itäta ja jach'a jach'athawa jach'awa jach'jamawa jach'äthwa jach'äña jach'üru jakawikïpanaya jali jama jamach'impi jan jani janikucha janimna janira janisa janit janiw janiwa janja janjama janq'u jaqiña jaqjamawa jath jatha jathi jawira jawirchaqa jawirkata jayp'ukïpanaya jayp'uru jaytjawayamawa ji jichha jichhakama jichhakipuniw jichhasti jichhax jichhüru jikisipxi jikisiñkama jilajatakiwa jilamasti jilamaxa jilaqataxa jiliri jilïri jinaya jinchuwisa jirupxi jiskhit'akathati jiskht'kthti jissma jista jistam jistan jiwasa jiwasanaka jiwasaxa jiwaspach jiwaspacha jum juma jumakiy jumalaykuwa jumamp juman jumanaka jumanakast jumasti jumatakiw jumaw jumaxa jupa jupamp jupanaka jupanakaw jupanakaxa jupanakpura jupankiti jupapacha jupapuniwa jupar jupasti jupatak jupaw jupax jupaxa jutakitapiy jutama jutani jutaniti jutaniwa jutapana jutaspax jutatha jutathawa jutipana jutkaniti jutkchiti jutkpanti jutta juttha jutthwa jutxasin juwana juwanti juwantimp k'ak'a k'alla ka kajun kama kankaña kankthwa karlu kata katu katuri kawkha kawkhankisa kawki kawkins kawkir kawkirus kawkirusa kha khaya khaykatin khitirus khitis khitisa khuchi khuchimp khuri khusu khä ki kimsanimpiw kimsanpachaw kimsax kimspacha kimsïri kipa kiway kullakañawa kullakpura kullakpurawa kunarus kunas kunasa kunatiy kunjamjamasa kuns kunsa kutinitama kuyurmi kürmi la lanti lapis lapisa lawampiw layku li lim liwru liwrux liyiñ liyiña ll lla llqañäni lunisa lura luraiwa lurani luratawa lurathawa lurawi lurañäniwa luriwa lurtanwa lurtawa lurthwa ma machaqa mamajawa mamajaxa mamanitwa mamanïthwa manq'a manq'athawa manq'awi manq'aña manq'i manq'iri manq'tanwa manq'tawa manq'thwa manqha manqhankiw manulaykuw mara maraxa mariy mariya mariyakiti mariyat mariyax mark markajata markakama markakamawa markaru markaruw markata masinakanithwa maya mayïri mc misa misax ml mp mph mpi mpl mplru munana munatayna munatha munatäna munatätha munayätha munta muntha muntwa munäna muytätaxa mä na naka nama naqa nay naya nayan nayanx nayar nayaw nayax nayaxa nayrawisa ni ninachim ninaxachim nka nki nma nml nq nqin nt nta nti nuku nuqa näna nänaka p'akita p'iqiw p'isi pa pacha pachana pachani pachasma pachata pachatam pachatan pachayata pachayatan pachista pachistu pachitu palljanipkamti palljasipkaraktwa pampana pan pana parlantaña parlapxaña parlapxitu parlañ parlt'aña parlxaraktasa patatuqin paxsi paya payakama payaw phapha phaxsi phaxsitpach phaxspachat phaxsüru phayaña philiksu phinat phisi phisima phuchhasa phutukraphiyt'aniwayta phäña pi pichaña pinitayna pirutimp pist'asaw pita piya pl pp pr pst pstin pt pta puni punku punkun pura puri purinisti puriñkama purp pusi px pxa pxi pä pï q' q'illu q'ipinitayna q'ipitawa q'ipitwa q'ipiwa q'ipixa q'ipiñäni q'ipta q'iptawa q'iptwa qa qalararaw qalatawa qalt qamiriptaña qamirpura qata qawqha qayqa qhathu qhatut qillqa qillqañajawa qillqañaxaw qillqañäni qin qsu quant qui qulluchiqa qullukama qurïpachawa quña qyn qüña ra raki rana raphaku rapi raqa rara ratuki riluju rimiña rixinta riyata rpaya rt rtp ru rumarisu rusintita ruwirtuw sa samana samaña sana sapana sara sarai sarajaya saram sarama saramaya sarani sarapxaspa sarapxayätha sarapxañani sarapxañaniw sarapxchi sarapxi sarapxsnaw sarapxtha sarapxäna sarapxäta saraqtawayaña sararakiniw saraskaña saraski sarasma sarasna saraspa sarata saratayna saratayta sarataytana sarataytha saratäna saratäta saratätana saratätha sarawa sarawaykxa saraxa sarayäta sarayätana sarayätha saraña sarañajawa sarañani sarañaxaw sarañäni sarchi sari sariri sariristha sarkamti sarkaniti sarkaña sarkiti sarnaqawi sarnaqäta sarpa sarpachawa sarpaya sarsta sarstana sarstha sarta sartana sartayaraqsma sartha sarthwa sarä saräna saräta sasa sasaw satachaqana sataw saña si silulara sim simil simp sina sirwis sischi sispa sispacha sissma sissna sista sistam sistan sistu situ sk ska sma sna spa spal span sr ss ssma sta stam stan stani sti su sum suma supp sutimaxa sutiyata sutiyawi sutiyayaña sutiyaña suxta suykama t'a t'ant' t'ant'a t'ant'aki t'ant't ta tac taki tam tama tan tana tantachasipxi tantachawi tantachthapiña tantathapiña taqpacha tasma tata tatajana tatajaxa tatam tatama tatamaru tatan tataw tatax tataxa tataxawa tawa taya tayna taypi taypinkiw taypipan taña tha thakhichiqa thapi thawa thuqhuwa thuqhuñäni thwa ti tinta tintana tintawa tisti tiwanaya tiwulampi tixa tiya tpan tukuyapxtanw tumpkama tunka tunkakama tuqi tä täna täta tätana tätha ua uka ukampix ukana ukanama ukat ukatsti ukawjana ukax ukch'apurawa ukha ukhama ukhamapï ukhamarakipi ukhan ukkata uknama ukxaru ukxaruxa uliwyar ull umampi umaña umiri umäña until up uraqita uru urukiipana urukiipanaya urukipana urukipanaya urukïpana urukïpanaya usu usuta usutacha usutu ut uta utaja utajawa utajx utakama utakichixay utama utana utanaka utanakajana utanakapasa utani utanithawa utanithwa utanitwa utaniwa utankaña utankthwa utapa utapan utaruw utaruwa utasa utawja utaya utjaña utji utn uyuchiqa uñjir uñjtawa wa wakawa wakiski wakwa wali walikiw walikpi waliptaña walja walla wallatampiwa wampu wara warmikamawa warmipura warmipurawa warmiña waskisatwa wasüru wasüruxa waw wawachaña wawam wawanak wawanaka wawapataki wawar wawaru wawaña way waya wi wila wisa wisk'achampi with wja wn wänaka xa xact xaru xasi xata xaya xcl xt ya yampu yapa yapupankaña yapux yasma yata yatam yatan yatichapxitatapiy yatichayaña yatichaña yatiqasamana yatiqatayna yatiqañ yatisixasmatisti yatixathawa yatktti yatthwa yatxthwa yaya yuqallat yä yäpa yäta yätana yätha ïri ña ñama ñani ñapa ñasa ñataki ñaxa ñäni".split()

SILABAS_PESOS = {
    "N_HUM":   {2: 0.70, 3: 0.30},
    "N_NOHUM": {2: 0.70, 3: 0.30},
    "V_INTR":  {2: 0.75, 3: 0.25},
    "V_MOV":   {2: 0.75, 3: 0.25},
    "V_TR":    {2: 0.75, 3: 0.25},
    "V_DITR":  {2: 0.75, 3: 0.25},
    "ADJ":     {2: 0.60, 3: 0.40},
    "ADV_T":   {2: 0.50, 3: 0.50},
}
PLANTILLAS_PESOS = {
    "primera":    {"V": 0.10, "CV": 0.65, "VC": 0.05, "CVC": 0.20},
    "intermedia": {"CV": 0.70, "CVC": 0.30},
    "ultima":     {"CV": 1.00},
}
INICIO_PESOS = {
    "p": 0.07, "t": 0.09, "ch": 0.07, "k": 0.09, "q": 0.08,
    "s": 0.07, "j": 0.06, "x": 0.04, "m": 0.07, "n": 0.06,
    "ñ": 0.02, "l": 0.06, "ll": 0.04, "r": 0.05, "w": 0.05, "y": 0.08,
}
INICIO_PRIMERA_PESOS = {k: v for k, v in INICIO_PESOS.items() if k not in INICIALES_PROHIBIDAS}
CODA_PESOS = {
    "p": 0.06, "t": 0.04, "ch": 0.04, "k": 0.08, "q": 0.08,
    "s": 0.14, "j": 0.08, "x": 0.06, "m": 0.10, "n": 0.10,
    "l": 0.10, "r": 0.08, "w": 0.02, "y": 0.02,
}
VOCALES_PESOS = {"a": 0.45, "i": 0.25, "u": 0.30}


class TriePalabrasReales:
    """CH-306: Trie que cuenta cuántas palabras distintas pasan por cada prefijo."""
    def __init__(self, palabras: List[str]):
        self.root: Dict[str, Any] = {"_n": 0, "_fin": False}
        for tok in {normalizar(p).lower().strip() for p in palabras if p.strip()}:
            node = self.root
            node["_n"] += 1
            for ch in tok:
                if ch not in node:
                    node[ch] = {"_n": 0, "_fin": False}
                node = node[ch]
                node["_n"] += 1
            node["_fin"] = True

    def consultar(self, prefijo: str) -> Tuple[bool, int]:
        node = self.root
        for ch in normalizar(prefijo).lower():
            if ch not in node:
                return False, 0
            node = node[ch]
        return bool(node["_fin"]), int(node["_n"])


def _elegir_ponderado(rng: random.Random, tabla: Dict[Any, float]) -> Any:
    return rng.choices(list(tabla.keys()), weights=list(tabla.values()), k=1)[0]


def _formas_sufijos_y_gram() -> Tuple[Set[str], Set[str], Set[str]]:
    formas_suf_todas: Set[str] = set()
    terminaciones_proh: Set[str] = set()
    prohibidas_iguales: Set[str] = set()
    for m in SUFIJOS_LISTA:
        for f in (m["b"], m["a"]):
            if f == "¨":
                continue
            f_clean = f.lstrip("¨")
            if not f_clean:
                continue
            formas_suf_todas.add(f_clean)
            prohibidas_iguales.add(f_clean)
            if len(segmentar(f_clean)) >= 3:
                terminaciones_proh.add(f_clean)
    for pg in PALABRAS_GRAM_LISTA:
        prohibidas_iguales.add(pg["b"])
        prohibidas_iguales.add(pg["a"])
    return prohibidas_iguales, terminaciones_proh, formas_suf_todas


def generar_raices(
    lexico_sin_raices: List[Dict[str, Any]],
    palabras_reales: List[str] = PALABRAS_REALES_GUIA,
    modo_filtro: str = "guia",
    semilla: int = 20261119,
    preservar_existentes: bool = False,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """§3.1: Genera una raíz inventada única y validada para cada entrada del léxico."""
    rng = random.Random(semilla)
    trie = TriePalabrasReales(palabras_reales)
    umbral = 1 if modo_filtro == "guia" else 5
    prohibidas_iguales, terminaciones_proh, formas_suf_todas = _formas_sufijos_y_gram()

    lexico_ordenado = sorted(deepcopy(lexico_sin_raices), key=lambda e: e["id"])
    aceptadas: Dict[str, str] = {}
    rechazos_cat: Counter = Counter()
    intentos_cat: Counter = Counter()

    if preservar_existentes:
        for e in lexico_ordenado:
            if e.get("raiz"):
                aceptadas[e["raiz"]] = e["id"]

    def choca_raiz_mas_sufijo(r: str) -> bool:
        for s in formas_suf_todas:
            if r.endswith(s) and r[: -len(s)] in aceptadas:
                return True
            if (r + s) in aceptadas:
                return True
        return False

    for e in lexico_ordenado:
        if preservar_existentes and e.get("raiz"):
            e["silabas"] = contar_silabas(e["raiz"])
            continue
        cat = e["cat"]
        exito = False
        for _ in range(1000):
            intentos_cat[cat] += 1
            n_sil = _elegir_ponderado(rng, SILABAS_PESOS[cat])
            grafemas: List[str] = []
            for idx in range(n_sil):
                pos = "primera" if idx == 0 else ("ultima" if idx == n_sil - 1 else "intermedia")
                plantilla = _elegir_ponderado(rng, PLANTILLAS_PESOS[pos])
                for slot_c in plantilla:
                    if slot_c == "C":
                        if not grafemas:
                            grafemas.append(_elegir_ponderado(rng, INICIO_PRIMERA_PESOS))
                        elif grafemas[-1] in VOCALES_CORTAS and plantilla.endswith("C") and slot_c == plantilla[-1]:
                            grafemas.append(_elegir_ponderado(rng, CODA_PESOS))
                        else:
                            grafemas.append(_elegir_ponderado(rng, INICIO_PESOS))
                    else:
                        if idx == n_sil - 1 and cat.startswith("V_"):
                            grafemas.append("a")  # CH-304
                        else:
                            grafemas.append(_elegir_ponderado(rng, VOCALES_PESOS))

            # CH-305: laringalización con p = 0.30
            if rng.random() < 0.30:
                inicios_ocl = [
                    i for i, gr in enumerate(grafemas)
                    if gr in OCLUSIVAS_SIMPLES and not es_coda_en_raiz(grafemas, i)
                ]
                if inicios_ocl:
                    ocl = grafemas[rng.choice(inicios_ocl)]
                    nueva = ASPIRADA_DE[ocl] if rng.random() < 0.5 else GLOTAL_DE[ocl]
                    for j in range(len(grafemas)):
                        if grafemas[j] == ocl and not es_coda_en_raiz(grafemas, j):
                            grafemas[j] = nueva

            r = "".join(grafemas)
            if not es_raiz_valida(r):
                rechazos_cat[cat] += 1
                continue
            es_pal, n_pref = trie.consultar(r)
            if es_pal or n_pref >= umbral or r in prohibidas_iguales:
                rechazos_cat[cat] += 1
                continue
            if any(r.endswith(s) for s in terminaciones_proh) or r in aceptadas or choca_raiz_mas_sufijo(r):
                rechazos_cat[cat] += 1
                continue

            aceptadas[r] = e["id"]
            e["raiz"] = r
            e["silabas"] = n_sil
            exito = True
            break

        if not exito:
            raise RuntimeError(f"Error CH-306: sin raíz válida para {e['id']} ({cat}) tras 1000 intentos.")

    stats = {}
    avisos = []
    for cat in sorted(intentos_cat):
        tasa = rechazos_cat[cat] / max(1, intentos_cat[cat])
        stats[cat] = {"intentos": intentos_cat[cat], "rechazos": rechazos_cat[cat], "tasa_rechazo": round(tasa, 3)}
        if tasa > 0.60:
            avisos.append(f"AVISO CH-306: {cat} tasa de rechazo {tasa:.1%} (>60%)")
    return lexico_ordenado, {"stats_por_cat": stats, "avisos": avisos, "total_raices": len(aceptadas)}

print(f"Generador de raíces listo. Palabras reales de la guía cargadas: {len(PALABRAS_REALES_GUIA)}.")

## 6. Léxico Inicial (`CH-401` a `CH-407`)

Incluye:
- `LEXICO_VALIDACION_CSV` (`LV01`–`LV18`): raíces reales usadas **exclusivamente** para verificar los 17 casos `[ATESTIGUADO]` de §12 (nunca en el dataset sintético).
- `LEXICO_EJEMPLO_SPEC_CSV` (`L0001`–`L0025`): las 25 entradas exactas de §4.1 para verificar `C18`–`C40`.
- `LEXICO_INICIAL_EXTRA_CSV` (`L0026`–`L0093`): 68 entradas adicionales para que el generador inicial de **1.000 pares** cubra las **13 clases semánticas** (`CH-406`), los 4 tipos de verbos, adjetivos y adverbios de tiempo.

> **Escalado futuro:** cuando el equipo entregue `lexico_sin_raices.csv` con ~3.130 entradas (`CH-407`), solo cárgalo con `cargar_lexico_csv("lexico_sin_raices.csv")` y pásalo a `generar_raices(...)`.

In [ ]:
# -*- coding: utf-8 -*-
import csv, io

LEXICO_VALIDACION_CSV = """id,cat,subcat,clase_sem,es_m,es_f,es_pl_m,es_pl_f,genero,contable,raiz,silabas,rango,rige_ru,obj_persona,suj_clases,obj_clases,instr_clases,loc_clases,mod_clases,notas
LV01,N_HUM,-,persona,niño,niña,niños,niñas,var,s,wawa,2,1,-,-,-,-,-,-,-,validacion
LV02,N_NOHUM,-,comida,pan,,panes,,m,n,t'ant'a,2,2,-,-,-,-,-,-,-,validacion
LV03,V_DITR,-,-,dar,,,,-,-,chura,2,1,s,s,persona,comida|bebida|ropa|herramienta|recipiente|animal|planta|objeto,-,-,-,validacion
LV04,N_HUM,-,persona,padre,,padres,,m,s,tata,2,2,-,-,-,-,-,-,-,validacion
LV05,N_NOHUM,-,lugar,pueblo,,pueblos,,m,s,marka,2,3,-,-,-,-,-,-,-,validacion
LV06,V_MOV,-,-,venir,,,,-,-,juta,2,2,n,-,persona|animal,-,-,NINGUNA,-,validacion
LV07,V_MOV,-,-,ir,,,,-,-,sara,2,1,n,-,persona|animal,-,-,NINGUNA,-,validacion
LV08,V_INTR,-,-,comer,,,,-,-,manq'a,2,1,n,-,persona|animal,-,-,-,-,validacion
LV09,V_TR,-,-,comer,,,,-,-,manq'a,2,1,n,n,persona|animal,comida,herramienta,-,-,validacion
LV10,N_NOHUM,-,animal,perro,,perros,,m,s,anu,2,4,-,-,-,-,-,-,-,validacion
LV11,N_NOHUM,-,vivienda,casa,,casas,,f,s,uta,2,1,-,-,-,-,-,-,-,validacion
LV12,N_HUM,-,persona,hermano,hermana,hermanos,hermanas,var,s,jila,2,3,-,-,-,-,-,-,-,validacion
LV13,N_HUM,-,persona,autoridad,,autoridades,,f,s,jilaqata,4,4,-,-,-,-,-,-,-,validacion
LV14,N_HUM,-,persona,mamá,,mamás,,f,s,mama,2,5,-,-,-,-,-,-,-,validacion
LV15,N_NOHUM,-,animal,zorro,,zorros,,m,s,tiwula,3,5,-,-,-,-,-,-,-,validacion
LV16,N_NOHUM,-,animal,ganso,,gansos,,m,s,wallata,3,6,-,-,-,-,-,-,-,validacion
LV17,V_INTR,-,-,encontrarse,,,,-,-,jikisi,3,2,n,-,persona|animal,-,-,-,-,validacion
LV18,V_TR,-,-,querer,,,,-,-,muna,2,2,n,s,persona,comida|bebida|ropa|herramienta|recipiente|animal|planta|vivienda|objeto,-,-,-,validacion
"""

LEXICO_EJEMPLO_SPEC_CSV = """id,cat,subcat,clase_sem,es_m,es_f,es_pl_m,es_pl_f,genero,contable,raiz,silabas,rango,rige_ru,obj_persona,suj_clases,obj_clases,instr_clases,loc_clases,mod_clases,notas
L0001,N_HUM,-,persona,niño,niña,niños,niñas,var,s,lisqu,2,1,-,-,-,-,-,-,-,
L0002,N_HUM,-,persona,hermano,hermana,hermanos,hermanas,var,s,pultiya,3,2,-,-,-,-,-,-,-,
L0003,N_HUM,-,persona,maestro,maestra,maestros,maestras,var,s,yaruqhi,3,3,-,-,-,-,-,-,-,
L0004,N_HUM,-,persona,mujer,,mujeres,,f,s,simlla,2,4,-,-,-,-,-,-,-,
L0005,N_NOHUM,-,vivienda,casa,,casas,,f,s,chimula,3,1,-,-,-,-,-,-,-,
L0006,N_NOHUM,-,comida,pan,,panes,,m,n,q'aspi,2,2,-,-,-,-,-,-,-,
L0007,N_NOHUM,-,lugar,pueblo,,pueblos,,m,s,wirkha,2,3,-,-,-,-,-,-,-,
L0008,N_NOHUM,-,animal,perro,,perros,,m,s,kirpu,2,4,-,-,-,-,-,-,-,
L0009,N_NOHUM,-,lugar,mercado,,mercados,,m,s,pinqara,3,5,-,-,-,-,-,-,-,
L0010,N_NOHUM,-,herramienta,palo,,palos,,m,s,salmi,2,6,-,-,-,-,-,-,-,
L0011,N_NOHUM,-,herramienta,cuchillo,,cuchillos,,m,s,jitqu,2,7,-,-,-,-,-,-,-,
L0012,N_NOHUM,-,abstracto,deuda,,deudas,,f,s,ch'alqu,2,8,-,-,-,-,-,-,-,
L0013,V_MOV,-,-,ir,,,,-,-,tisqa,2,1,n,-,persona|animal,-,-,NINGUNA,-,
L0014,V_MOV,-,-,venir,,,,-,-,pajlla,2,2,n,-,persona|animal,-,-,NINGUNA,-,
L0015,V_INTR,-,-,dormir,,,,-,-,nuyk'a,2,1,n,-,persona|animal,-,-,-,-,
L0016,V_TR,-,-,comer,,,,-,-,lamch'a,2,1,n,n,persona|animal,comida,herramienta,-,-,
L0017,V_TR,-,-,ver,,,,-,-,ch'ixra,2,2,n,s,persona|animal,animal|comida|bebida|ropa|herramienta|recipiente|planta|vivienda|objeto,-,-,-,
L0018,V_DITR,-,-,dar,,,,-,-,tulpha,2,1,s,s,persona,comida|bebida|ropa|herramienta|recipiente|animal|planta|objeto,-,-,-,
L0019,V_TR,-,-,comprar,,,,-,-,nirqa,2,3,n,n,persona,comida|bebida|ropa|herramienta|recipiente|animal,-,-,-,
L0020,V_TR,-,-,cortar,,,,-,-,k'iyma,2,4,n,n,persona,comida|planta|ropa,herramienta,-,-,
L0021,ADJ,tamano,-,grande,grande,grandes,grandes,-,-,jumpu,2,1,-,-,-,-,-,-,persona|animal|comida|ropa|herramienta|recipiente|planta|lugar|vivienda|objeto,
L0022,ADJ,color,-,rojo,roja,rojos,rojas,-,-,q'ulli,2,2,-,-,-,-,-,-,animal|comida|bebida|ropa|herramienta|recipiente|planta|vivienda|objeto,
L0023,ADV_T,pasado,-,ayer,,,,-,-,jaskhi,2,1,-,-,-,-,-,-,-,
L0024,ADV_T,futuro,-,mañana,,,,-,-,sik'uyu,3,2,-,-,-,-,-,-,-,
L0025,ADV_T,presente,-,ahora,,,,-,-,nuspi,2,3,-,-,-,-,-,-,-,
"""

LEXICO_INICIAL_EXTRA_CSV = """id,cat,subcat,clase_sem,es_m,es_f,es_pl_m,es_pl_f,genero,contable,raiz,silabas,rango,rige_ru,obj_persona,suj_clases,obj_clases,instr_clases,loc_clases,mod_clases,notas
L0026,N_HUM,-,persona,padre,,padres,,m,s,,0,5,-,-,-,-,-,-,-,
L0027,N_HUM,-,persona,madre,,madres,,f,s,,0,6,-,-,-,-,-,-,-,
L0028,N_HUM,-,persona,amigo,amiga,amigos,amigas,var,s,,0,7,-,-,-,-,-,-,-,
L0029,N_HUM,-,persona,abuelo,abuela,abuelos,abuelas,var,s,,0,8,-,-,-,-,-,-,-,
L0030,N_HUM,-,persona,vecino,vecina,vecinos,vecinas,var,s,,0,9,-,-,-,-,-,-,-,
L0031,N_HUM,-,persona,médico,médica,médicos,médicas,var,s,,0,10,-,-,-,-,-,-,-,
L0032,N_NOHUM,-,animal,gato,,gatos,,m,s,,0,9,-,-,-,-,-,-,-,
L0033,N_NOHUM,-,animal,llama,,llamas,,f,s,,0,10,-,-,-,-,-,-,-,
L0034,N_NOHUM,-,animal,pájaro,,pájaros,,m,s,,0,11,-,-,-,-,-,-,-,
L0035,N_NOHUM,-,comida,papa,,papas,,f,s,,0,12,-,-,-,-,-,-,-,
L0036,N_NOHUM,-,comida,carne,,carnes,,f,n,,0,13,-,-,-,-,-,-,-,
L0037,N_NOHUM,-,comida,queso,,quesos,,m,n,,0,14,-,-,-,-,-,-,-,
L0038,N_NOHUM,-,comida,maíz,,maíces,,m,n,,0,15,-,-,-,-,-,-,-,
L0039,N_NOHUM,-,bebida,agua,,aguas,,f,n,,0,16,-,-,-,-,-,-,-,
L0040,N_NOHUM,-,bebida,leche,,leches,,f,n,,0,17,-,-,-,-,-,-,-,
L0041,N_NOHUM,-,bebida,chicha,,chichas,,f,n,,0,18,-,-,-,-,-,-,-,
L0042,N_NOHUM,-,ropa,poncho,,ponchos,,m,s,,0,19,-,-,-,-,-,-,-,
L0043,N_NOHUM,-,ropa,sombrero,,sombreros,,m,s,,0,20,-,-,-,-,-,-,-,
L0044,N_NOHUM,-,ropa,camisa,,camisas,,f,s,,0,21,-,-,-,-,-,-,-,
L0045,N_NOHUM,-,ropa,aguayo,,aguayos,,m,s,,0,22,-,-,-,-,-,-,-,
L0046,N_NOHUM,-,herramienta,pala,,palas,,f,s,,0,23,-,-,-,-,-,-,-,
L0047,N_NOHUM,-,herramienta,martillo,,martillos,,m,s,,0,24,-,-,-,-,-,-,-,
L0048,N_NOHUM,-,recipiente,olla,,ollas,,f,s,,0,25,-,-,-,-,-,-,-,
L0049,N_NOHUM,-,recipiente,vaso,,vasos,,m,s,,0,26,-,-,-,-,-,-,-,
L0050,N_NOHUM,-,recipiente,bolsa,,bolsas,,f,s,,0,27,-,-,-,-,-,-,-,
L0051,N_NOHUM,-,planta,árbol,,árboles,,m,s,,0,28,-,-,-,-,-,-,-,
L0052,N_NOHUM,-,planta,flor,,flores,,f,s,,0,29,-,-,-,-,-,-,-,
L0053,N_NOHUM,-,planta,hierba,,hierbas,,f,n,,0,30,-,-,-,-,-,-,-,
L0054,N_NOHUM,-,lugar,río,,ríos,,m,s,,0,31,-,-,-,-,-,-,-,
L0055,N_NOHUM,-,lugar,cerro,,cerros,,m,s,,0,32,-,-,-,-,-,-,-,
L0056,N_NOHUM,-,lugar,plaza,,plazas,,f,s,,0,33,-,-,-,-,-,-,-,
L0057,N_NOHUM,-,vivienda,cuarto,,cuartos,,m,s,,0,34,-,-,-,-,-,-,-,
L0058,N_NOHUM,-,vivienda,corral,,corrales,,m,s,,0,35,-,-,-,-,-,-,-,
L0059,N_NOHUM,-,vivienda,escuela,,escuelas,,f,s,,0,36,-,-,-,-,-,-,-,
L0060,N_NOHUM,-,tiempo,día,,días,,m,s,,0,37,-,-,-,-,-,-,-,
L0061,N_NOHUM,-,tiempo,noche,,noches,,f,s,,0,38,-,-,-,-,-,-,-,
L0062,N_NOHUM,-,tiempo,año,,años,,m,s,,0,39,-,-,-,-,-,-,-,
L0063,N_NOHUM,-,abstracto,trabajo,,trabajos,,m,s,,0,40,-,-,-,-,-,-,-,
L0064,N_NOHUM,-,abstracto,fiesta,,fiestas,,f,s,,0,41,-,-,-,-,-,-,-,
L0065,N_NOHUM,-,objeto,piedra,,piedras,,f,s,,0,42,-,-,-,-,-,-,-,
L0066,N_NOHUM,-,objeto,libro,,libros,,m,s,,0,43,-,-,-,-,-,-,-,
L0067,N_NOHUM,-,objeto,mesa,,mesas,,f,s,,0,44,-,-,-,-,-,-,-,
L0068,V_MOV,-,-,caminar,,,,-,-,,0,3,n,-,persona|animal,-,-,NINGUNA,-,
L0069,V_MOV,-,-,correr,,,,-,-,,0,4,n,-,persona|animal,-,-,NINGUNA,-,
L0070,V_MOV,-,-,llegar,,,,-,-,,0,5,n,-,persona|animal,-,-,NINGUNA,-,
L0071,V_INTR,-,-,llorar,,,,-,-,,0,2,n,-,persona,-,-,-,-,
L0072,V_INTR,-,-,descansar,,,,-,-,,0,3,n,-,persona|animal,-,-,-,-,
L0073,V_INTR,-,-,trabajar,,,,-,-,,0,4,n,-,persona,-,herramienta,-,-,
L0074,V_INTR,-,-,cantar,,,,-,-,,0,5,n,-,persona,-,-,-,-,
L0075,V_TR,-,-,beber,,,,-,-,,0,5,n,n,persona|animal,bebida,recipiente,-,-,
L0076,V_TR,-,-,llevar,,,,-,-,,0,6,n,s,persona,comida|bebida|ropa|herramienta|recipiente|planta|animal|objeto,recipiente,-,-,
L0077,V_TR,-,-,buscar,,,,-,-,,0,7,n,s,persona|animal,animal|comida|bebida|ropa|herramienta|recipiente|planta|objeto,-,-,-,
L0078,V_TR,-,-,lavar,,,,-,-,,0,8,n,n,persona,ropa|recipiente|comida|herramienta,-,-,-,
L0079,V_TR,-,-,leer,,,,-,-,,0,9,n,n,persona,objeto,-,-,-,
L0080,V_TR,-,-,cocinar,,,,-,-,,0,10,n,n,persona,comida,recipiente,-,-,
L0081,V_TR,-,-,construir,,,,-,-,,0,11,n,n,persona,vivienda|objeto,herramienta,-,-,
L0082,V_DITR,-,-,entregar,,,,-,-,,0,2,s,s,persona,comida|bebida|ropa|herramienta|recipiente|objeto,-,-,-,
L0083,V_DITR,-,-,mostrar,,,,-,-,,0,3,s,s,persona,animal|comida|ropa|herramienta|recipiente|planta|vivienda|objeto,-,-,-,
L0084,V_DITR,-,-,prestar,,,,-,-,,0,4,s,s,persona,ropa|herramienta|recipiente|objeto,-,-,-,
L0085,ADJ,tamano,-,pequeño,pequeña,pequeños,pequeñas,-,-,,0,3,-,-,-,-,-,-,persona|animal|comida|ropa|herramienta|recipiente|planta|lugar|vivienda|objeto,
L0086,ADJ,tamano,-,largo,larga,largos,largas,-,-,,0,4,-,-,-,-,-,-,ropa|herramienta|planta|lugar|objeto,
L0087,ADJ,color,-,blanco,blanca,blancos,blancas,-,-,,0,5,-,-,-,-,-,-,animal|comida|bebida|ropa|herramienta|recipiente|planta|vivienda|objeto,
L0088,ADJ,color,-,negro,negra,negros,negras,-,-,,0,6,-,-,-,-,-,-,animal|ropa|herramienta|recipiente|vivienda|objeto,
L0089,ADJ,otro,-,nuevo,nueva,nuevos,nuevas,-,-,,0,7,-,-,-,-,-,-,ropa|herramienta|recipiente|vivienda|objeto,
L0090,ADJ,otro,-,bueno,buena,buenos,buenas,-,-,,0,8,-,-,-,-,-,-,persona|animal|comida|bebida|ropa|herramienta|objeto,
L0091,ADV_T,pasado,-,anoche,,,,-,-,,0,4,-,-,-,-,-,-,-,
L0092,ADV_T,presente,-,hoy,,,,-,-,,0,5,-,-,-,-,-,-,-,
L0093,ADV_T,futuro,-,pronto,,,,-,-,,0,6,-,-,-,-,-,-,-,
"""


def cargar_csv_str(csv_text: str):
    return list(csv.DictReader(io.StringIO(csv_text.strip())))


def cargar_lexico_csv(ruta_csv: str) -> List[Dict[str, Any]]:
    with open(ruta_csv, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))

# Construir léxico inicial de 93 entradas y generar sus raíces inventadas (CH-309)
lexico_base_93 = cargar_csv_str(LEXICO_EJEMPLO_SPEC_CSV) + cargar_csv_str(LEXICO_INICIAL_EXTRA_CSV)
for _e in lexico_base_93:
    _e["raiz"] = ""
    _e["silabas"] = 0

LEXICO_CHULUPI, STATS_RAICES = generar_raices(
    lexico_base_93,
    palabras_reales=PALABRAS_REALES_GUIA,
    modo_filtro="guia",
    semilla=20261119,
)

print(f"Léxico Chulupi generado: {STATS_RAICES['total_raices']} entradas únicas. Avisos: {STATS_RAICES['avisos']}")
print("Primeras 8 entradas generadas:")
for _e in LEXICO_CHULUPI[:8]:
    print(f"  {_e['id']} | {_e['cat']:7s} | {_e['clase_sem']:11s} | {_e['es_m']:10s} -> {_e['raiz']} ({_e['silabas']} sil)")

## 7. Conjugador Español y Constructor de Oraciones B, A y ES (§9 y §10)

In [ ]:
# --- Conjugador español (CH-1001 a CH-1006) ---
# Índices persona ES: 0=yo, 1=tú, 2=él/ella, 3=nosotros, 4=ustedes/ellos
AUX_HABER_PRES = ["he", "has", "ha", "hemos", "han"]
AUX_HABER_IMP = ["había", "habías", "había", "habíamos", "habían"]
AUX_ESTAR_PRES = ["estoy", "estás", "está", "estamos", "están"]
VERBO_SER_PRES = ["soy", "eres", "es", "somos", "son"]
VERBO_TENER_PRES = ["tengo", "tienes", "tiene", "tenemos", "tienen"]

CONJUGACIONES_IRREGULARES: Dict[str, Dict[str, Any]] = {
    "ir":          {"pres": ["voy", "vas", "va", "vamos", "van"], "pret": ["fui", "fuiste", "fue", "fuimos", "fueron"], "fut": ["iré", "irás", "irá", "iremos", "irán"], "part": "ido", "ger": "yendo"},
    "venir":       {"pres": ["vengo", "vienes", "viene", "venimos", "vienen"], "pret": ["vine", "viniste", "vino", "vinimos", "vinieron"], "fut": ["vendré", "vendrás", "vendrá", "vendremos", "vendrán"], "part": "venido", "ger": "viniendo"},
    "dar":         {"pres": ["doy", "das", "da", "damos", "dan"], "pret": ["di", "diste", "dio", "dimos", "dieron"], "fut": ["daré", "darás", "dará", "daremos", "darán"], "part": "dado", "ger": "dando"},
    "ver":         {"pres": ["veo", "ves", "ve", "vemos", "ven"], "pret": ["vi", "viste", "vio", "vimos", "vieron"], "fut": ["veré", "verás", "verá", "veremos", "verán"], "part": "visto", "ger": "viendo"},
    "querer":      {"pres": ["quiero", "quieres", "quiere", "queremos", "quieren"], "pret": ["quise", "quisiste", "quiso", "quisimos", "quisieron"], "fut": ["querré", "querrás", "querrá", "querremos", "querrán"], "part": "querido", "ger": "queriendo"},
    "dormir":      {"pres": ["duermo", "duermes", "duerme", "dormimos", "duermen"], "pret": ["dormí", "dormiste", "durmió", "dormimos", "durmieron"], "fut": ["dormiré", "dormirás", "dormirá", "dormiremos", "dormirán"], "part": "dormido", "ger": "durmiendo"},
    "reír":        {"pres": ["río", "ríes", "ríe", "reímos", "ríen"], "pret": ["reí", "reíste", "rio", "reímos", "rieron"], "fut": ["reiré", "reirás", "reirá", "reiremos", "reirán"], "part": "reído", "ger": "riendo"},
    "leer":        {"pres": ["leo", "lees", "lee", "leemos", "leen"], "pret": ["leí", "leíste", "leyó", "leímos", "leyeron"], "fut": ["leeré", "leerás", "leerá", "leeremos", "leerán"], "part": "leído", "ger": "leyendo"},
    "construir":   {"pres": ["construyo", "construyes", "construye", "construimos", "construyen"], "pret": ["construí", "construiste", "construyó", "construimos", "construyeron"], "fut": ["construiré", "construirás", "construirá", "construiremos", "construirán"], "part": "construido", "ger": "construyendo"},
    "mostrar":     {"pres": ["muestro", "muestras", "muestra", "mostramos", "muestran"], "pret": ["mostré", "mostraste", "mostró", "mostramos", "mostraron"], "fut": ["mostraré", "mostrarás", "mostrará", "mostraremos", "mostrarán"], "part": "mostrado", "ger": "mostrando"},
    "romper":      {"pres": ["rompo", "rompes", "rompe", "rompemos", "rompen"], "pret": ["rompí", "rompiste", "rompió", "rompimos", "rompieron"], "fut": ["romperé", "romperás", "romperá", "romperemos", "romperán"], "part": "roto", "ger": "rompiendo"},
    "escribir":    {"pres": ["escribo", "escribes", "escribe", "escribimos", "escriben"], "pret": ["escribí", "escribiste", "escribió", "escribimos", "escribieron"], "fut": ["escribiré", "escribirás", "escribirá", "escribiremos", "escribirán"], "part": "escrito", "ger": "escribiendo"},
    "encontrarse": {"refl": True, "pres": ["encuentro", "encuentras", "encuentra", "encontramos", "encuentran"], "pret": ["encontré", "encontraste", "encontró", "encontramos", "encontraron"], "fut": ["encontraré", "encontrarás", "encontrará", "encontraremos", "encontrarán"], "part": "encontrado", "ger": "encontrando"},
    "hacer":       {"pres": ["hago", "haces", "hace", "hacemos", "hacen"], "pret": ["hice", "hiciste", "hizo", "hicimos", "hicieron"], "fut": ["haré", "harás", "hará", "haremos", "harán"], "part": "hecho", "ger": "haciendo"},
    "decir":       {"pres": ["digo", "dices", "dice", "decimos", "dicen"], "pret": ["dije", "dijiste", "dijo", "dijimos", "dijeron"], "fut": ["diré", "dirás", "dirá", "diremos", "dirán"], "part": "dicho", "ger": "diciendo"},
    "traer":       {"pres": ["traigo", "traes", "trae", "traemos", "traen"], "pret": ["traje", "trajiste", "trajo", "trajimos", "trajeron"], "fut": ["traeré", "traerás", "traerá", "traeremos", "traerán"], "part": "traído", "ger": "trayendo"},
    "poner":       {"pres": ["pongo", "pones", "pone", "ponemos", "ponen"], "pret": ["puse", "pusiste", "puso", "pusimos", "pusieron"], "fut": ["pondré", "pondrás", "pondrá", "pondremos", "pondrán"], "part": "puesto", "ger": "poniendo"},
    "salir":       {"pres": ["salgo", "sales", "sale", "salimos", "salen"], "pret": ["salí", "saliste", "salió", "salimos", "salieron"], "fut": ["saldré", "saldrás", "saldrá", "saldremos", "saldrán"], "part": "salido", "ger": "saliendo"},
    "volver":      {"pres": ["vuelvo", "vuelves", "vuelve", "volvemos", "vuelven"], "pret": ["volví", "volviste", "volvió", "volvimos", "volvieron"], "fut": ["volveré", "volverás", "volverá", "volveremos", "volverán"], "part": "vuelto", "ger": "volviendo"},
    "abrir":       {"pres": ["abro", "abres", "abre", "abrimos", "abren"], "pret": ["abrí", "abriste", "abrió", "abrimos", "abrieron"], "fut": ["abriré", "abrirás", "abrirá", "abriremos", "abrirán"], "part": "abierto", "ger": "abriendo"},
    "pedir":       {"pres": ["pido", "pides", "pide", "pedimos", "piden"], "pret": ["pedí", "pediste", "pidió", "pedimos", "pidieron"], "fut": ["pediré", "pedirás", "pedirá", "pediremos", "pedirán"], "part": "pedido", "ger": "pidiendo"},
    "jugar":       {"pres": ["juego", "juegas", "juega", "jugamos", "juegan"], "pret": ["jugué", "jugaste", "jugó", "jugamos", "jugaron"], "fut": ["jugaré", "jugarás", "jugará", "jugaremos", "jugarán"], "part": "jugado", "ger": "jugando"},
}

CLITICOS_REFL = ["me", "te", "se", "nos", "se"]
FEM_A_TONICA = {"agua", "águila", "alma", "arma", "hacha", "hambre", "área", "ala", "alba"}


def _idx_persona_es(persona: int, plural: bool) -> int:
    if persona == 1:
        return 3 if plural else 0
    if persona == 2:
        return 4 if plural else 1
    if persona == 3:
        return 4 if plural else 2
    return 3


def conjugar_verbo_es(infinitivo: str, tiempo: str, aspecto: str, adv_subcat: Optional[str], idx_p: int) -> Tuple[str, Optional[str]]:
    refl = False
    if infinitivo in CONJUGACIONES_IRREGULARES:
        info = CONJUGACIONES_IRREGULARES[infinitivo]
        refl = bool(info.get("refl", False))
        pres, pret, fut, part, ger = info["pres"][idx_p], info["pret"][idx_p], info["fut"][idx_p], info["part"], info["ger"]
    else:
        base_inf = infinitivo
        if base_inf.endswith("se"):
            refl = True
            base_inf = base_inf[:-2]
        raiz, term = base_inf[:-2], base_inf[-2:]
        if term == "ar":
            pres = [raiz + s for s in ["o", "as", "a", "amos", "an"]][idx_p]
            r1 = raiz[:-1] + "qu" if raiz.endswith("c") else (raiz + "u" if raiz.endswith("g") else (raiz[:-1] + "c" if raiz.endswith("z") else raiz))
            pret = [r1 + "é", raiz + "aste", raiz + "ó", raiz + "amos", raiz + "aron"][idx_p]
            fut = [base_inf + s for s in ["é", "ás", "á", "emos", "án"]][idx_p]
            part, ger = raiz + "ado", raiz + "ando"
        elif term == "er":
            pres = [raiz + s for s in ["o", "es", "e", "emos", "en"]][idx_p]
            pret = [raiz + s for s in ["í", "iste", "ió", "imos", "ieron"]][idx_p]
            fut = [base_inf + s for s in ["é", "ás", "á", "emos", "án"]][idx_p]
            part, ger = raiz + "ido", raiz + "iendo"
        elif term == "ir":
            pres = [raiz + s for s in ["o", "es", "e", "imos", "en"]][idx_p]
            pret = [raiz + s for s in ["í", "iste", "ió", "imos", "ieron"]][idx_p]
            fut = [base_inf + s for s in ["é", "ás", "á", "emos", "án"]][idx_p]
            part, ger = raiz + "ido", raiz + "iendo"
        else:
            raise ValueError(f"Infinitivo no reconocido: {infinitivo}")

    clit_refl = CLITICOS_REFL[idx_p] if refl else None
    if aspecto == "progresivo":
        return f"{AUX_ESTAR_PRES[idx_p]} {ger}", clit_refl
    if tiempo == "SIMP":
        return (pret if adv_subcat == "pasado" else pres), clit_refl
    if tiempo == "FUT":
        return fut, clit_refl
    if tiempo == "REM.C":
        return f"{AUX_HABER_PRES[idx_p]} {part}", clit_refl
    if tiempo == "REM.L":
        return f"{AUX_HABER_IMP[idx_p]} {part}", clit_refl
    raise ValueError(f"Tiempo inválido: {tiempo}")


DEM_ES = {
    "DEM.1": {("m", False): "este",  ("f", False): "esta",  ("m", True): "estos",   ("f", True): "estas"},
    "DEM.2": {("m", False): "ese",   ("f", False): "esa",   ("m", True): "esos",    ("f", True): "esas"},
    "DEM.3": {("m", False): "aquel", ("f", False): "aquella", ("m", True): "aquellos", ("f", True): "aquellas"},
}
DEICTICO_ES = {"DEM.1": "aquí", "DEM.2": "ahí", "DEM.3": "más allá"}
NUM_ES = {"NUM.3": "tres", "NUM.4": "cuatro", "NUM.6": "seis", "NUM.10": "diez"}


def _genero_np(np: Dict[str, Any], lex_by_id: Dict[str, Dict[str, Any]]) -> str:
    ent = lex_by_id[np["n"]]
    if ent["genero"] == "var":
        return np.get("genero") if np.get("genero") in ("m", "f") else "m"
    return ent["genero"] if ent["genero"] in ("m", "f") else "m"


def _es_plural_np(np: Dict[str, Any]) -> bool:
    return bool(np.get("plural")) or bool(np.get("num"))


def _forma_nombre_es(ent: Dict[str, Any], genero: str, plural: bool) -> str:
    if plural:
        if ent["genero"] == "var":
            return ent["es_pl_f"] if genero == "f" else ent["es_pl_m"]
        return ent["es_pl_m"] or (ent["es_m"] + "s")
    if ent["genero"] == "var":
        return ent["es_f"] if genero == "f" else ent["es_m"]
    return ent["es_m"]


def _forma_adj_es(adj_ent: Dict[str, Any], genero: str, plural: bool) -> str:
    if plural:
        return adj_ent["es_pl_f"] if (genero == "f" and adj_ent.get("es_pl_f")) else adj_ent["es_pl_m"]
    return adj_ent["es_f"] if (genero == "f" and adj_ent.get("es_f")) else adj_ent["es_m"]


def _posesivo_es(poss: int, genero: str, plural: bool) -> str:
    if poss == 1:
        return "mis" if plural else "mi"
    if poss == 2:
        return "tus" if plural else "tu"
    if poss == 3:
        return "sus" if plural else "su"
    return ("nuestras" if genero == "f" else "nuestros") if plural else ("nuestra" if genero == "f" else "nuestro")


def _art_def_es(genero: str, plural: bool, sust: str = "") -> str:
    if plural:
        return "las" if genero == "f" else "los"
    if genero == "f" and sust.lower() in FEM_A_TONICA:
        return "el"
    return "la" if genero == "f" else "el"


def _art_indef_es(genero: str, sust: str = "") -> str:
    if genero == "f" and sust.lower() in FEM_A_TONICA:
        return "un"
    return "una" if genero == "f" else "un"


def _contraer_prep_es(prep: str, frase: str) -> str:
    if prep == "a" and frase.startswith("el "):
        return "al " + frase[3:]
    if prep == "de" and frase.startswith("el "):
        return "del " + frase[3:]
    return f"{prep} {frase}"


def render_np_es(np: Dict[str, Any], lex_by_id: Dict[str, Dict[str, Any]], rol: str = "general", forzar_plural: Optional[bool] = None) -> Tuple[str, str, bool]:
    """CH-1008: Reglas de prioridad de artículos y concordancia en español."""
    ent = lex_by_id[np["n"]]
    genero = _genero_np(np, lex_by_id)
    plural = _es_plural_np(np) if forzar_plural is None else forzar_plural
    contable = ent.get("contable", "s") == "s"
    es_hum = ent["cat"] == "N_HUM"

    sust = _forma_nombre_es(ent, genero, plural)
    adj_txt = _forma_adj_es(lex_by_id[np["adj"]], genero, plural) if np.get("adj") else None

    partes: List[str] = []
    if np.get("dem"):
        partes.append(DEM_ES[np["dem"]][(genero, plural)])
    elif np.get("num"):
        partes.append(NUM_ES[np["num"]])
    elif np.get("poss") and not np.get("poseedor"):
        partes.append(_posesivo_es(int(np["poss"]), genero, plural))
    elif np.get("poseedor"):
        partes.append(_art_def_es(genero, plural, sust))
    elif rol == "ser_pred":
        if not es_hum and not plural:
            partes.append(_art_indef_es(genero, sust))
    elif rol == "tener_pred":
        if contable and not plural:
            partes.append(_art_indef_es(genero, sust))
    elif rol == "od" and not contable:
        pass
    else:
        partes.append(_art_def_es(genero, plural, sust))

    partes.append(sust)
    if adj_txt:
        partes.append(adj_txt)
    if np.get("poseedor"):
        pos_txt, _, _ = render_np_es(np["poseedor"], lex_by_id, rol="poseedor")
        partes.append(_contraer_prep_es("de", pos_txt))
    return " ".join(partes), genero, plural


def _capitalizar_oracion(texto: str, es_pregunta: bool) -> str:
    if not texto:
        return texto
    c0 = texto[0].upper() + texto[1:]
    return f"¿{c0}?" if es_pregunta else f"{c0}."


def _pronombre_sujeto_es(persona: int, plural: bool, genero: Optional[str]) -> str:
    if persona == 1:
        return "nosotros" if plural else "yo"
    if persona == 2:
        return "ustedes" if plural else "tú"
    if persona == 4:
        return "nosotros"
    g = genero if genero in ("m", "f") else "m"
    return ("ellas" if g == "f" else "ellos") if plural else ("ella" if g == "f" else "él")


def construir_espanol(idea: Dict[str, Any], lex_by_id: Dict[str, Dict[str, Any]]) -> str:
    """§10 (CH-1001 a CH-1012): Genera la oración española determinista desde la idea."""
    pred, modalidad = idea["predicado"], idea["modalidad"]
    tiempo, aspecto = idea.get("tiempo", "SIMP"), idea.get("aspecto", "simple")
    interr = idea.get("interrogativo")
    suj = idea["sujeto"]
    p_suj, pl_suj = int(suj["persona"]), bool(suj.get("plural", False))
    expreso = suj.get("expreso", "ninguno")
    p_obj = int(idea.get("objeto_persona", 3))
    es_pregunta = modalidad in ("pregunta_si_no", "pregunta_info")
    idx_p = _idx_persona_es(p_suj, pl_suj)

    suj_txt: Optional[str] = None
    suj_gen = suj.get("genero") or "m"
    es_quien_suj = (modalidad == "pregunta_info" and interr == "quien_suj")

    if not es_quien_suj and expreso != "ninguno":
        if expreso == "pronombre":
            suj_txt = _pronombre_sujeto_es(p_suj, pl_suj, suj.get("genero"))
            suj_gen = "m" if (p_suj == 4 or (p_suj == 1 and pl_suj)) else (suj.get("genero") or "m")
        elif expreso == "nombre":
            suj_np = deepcopy(suj["np"])
            if pl_suj and not suj_np.get("num"):
                suj_np["plural"] = True
            suj_txt, suj_gen, _ = render_np_es(suj_np, lex_by_id, rol="general")
        elif expreso == "coordinado":
            t1, g1, _ = render_np_es(suj["coord"][0], lex_by_id, rol="general")
            t2, g2, _ = render_np_es(suj["coord"][1], lex_by_id, rol="general")
            suj_txt = f"{t1} y {t2}"
            suj_gen = "f" if (g1 == "f" and g2 == "f") else "m"
        elif expreso == "demostrativo":
            g_pred = _genero_np(idea["pnom"]["np"], lex_by_id)
            suj_txt = DEM_ES[suj["dem"]][(g_pred, pl_suj)]
            suj_gen = g_pred

    adv_id = idea.get("adv_t")
    adv_txt = lex_by_id[adv_id]["es_m"] if adv_id else None
    adv_subcat = lex_by_id[adv_id]["subcat"] if adv_id else None

    if pred != "verbo":
        pnom = idea.get("pnom") or {}
        if pred == "ser":
            verbo_txt = VERBO_SER_PRES[idx_p]
            es_pl_pred = pl_suj or (p_suj == 4)
            if pnom.get("adj"):
                pred_txt = _forma_adj_es(lex_by_id[pnom["adj"]], suj_gen if suj_gen in ("m", "f") else "m", es_pl_pred)
            else:
                np_p = deepcopy(pnom["np"])
                if not np_p.get("genero") and suj_gen in ("m", "f"):
                    np_p["genero"] = suj_gen
                pred_txt, _, _ = render_np_es(np_p, lex_by_id, rol="ser_pred", forzar_plural=es_pl_pred)
            return _capitalizar_oracion(" ".join(p for p in [suj_txt, verbo_txt, pred_txt, adv_txt] if p), es_pregunta)
        elif pred == "estar":
            verbo_txt = AUX_ESTAR_PRES[idx_p]
            if modalidad == "pregunta_info" and interr == "donde_estar":
                return _capitalizar_oracion(" ".join(p for p in ["dónde", verbo_txt, suj_txt, adv_txt] if p), True)
            pred_txt = DEICTICO_ES[pnom["deictico"]] if pnom.get("deictico") else f"en {render_np_es(pnom['np'], lex_by_id, rol='general')[0]}"
            return _capitalizar_oracion(" ".join(p for p in [suj_txt, verbo_txt, pred_txt, adv_txt] if p), es_pregunta)
        else:  # tener
            verbo_txt = VERBO_TENER_PRES[idx_p]
            pred_txt, _, _ = render_np_es(pnom["np"], lex_by_id, rol="tener_pred")
            return _capitalizar_oracion(" ".join(p for p in [suj_txt, verbo_txt, pred_txt, adv_txt] if p), es_pregunta)

    inf = lex_by_id[idea["verbo"]]["es_m"]
    verbo_txt, clit_refl = conjugar_verbo_es(inf, tiempo, aspecto, adv_subcat, idx_p)

    cliticos: List[str] = []
    if clit_refl:
        cliticos.append(clit_refl)
    if p_obj != 3:
        cliticos.append({1: "me", 2: "te", 4: "nos"}[p_obj])
    elif modalidad == "pregunta_info" and interr == "quien_oi":
        cliticos.append("le")
    elif idea.get("oi") and idea["oi"]["funcion"] == "COMPL.RU" and idea["oi"].get("np"):
        cliticos.append("les" if _es_plural_np(idea["oi"]["np"]) else "le")
    bloque_verbo = " ".join(cliticos + [verbo_txt])

    od_txt: Optional[str] = None
    if idea.get("od_coord"):
        t1, _, _ = render_np_es(idea["od_coord"][0], lex_by_id, rol="od")
        t2, _, _ = render_np_es(idea["od_coord"][1], lex_by_id, rol="od")
        od_txt = f"{t1} y {t2}"
    elif idea.get("od") and not (modalidad == "pregunta_info" and interr == "que_od"):
        od_txt, _, _ = render_np_es(idea["od"], lex_by_id, rol="od")

    oi_txt: Optional[str] = None
    if idea.get("oi") and idea["oi"].get("np") and not (modalidad == "pregunta_info" and interr == "quien_oi"):
        np_txt, _, _ = render_np_es(idea["oi"]["np"], lex_by_id, rol="general")
        oi_txt = _contraer_prep_es("a", np_txt) if idea["oi"]["funcion"] == "COMPL.RU" else f"para {np_txt}"

    lug_txt: Optional[str] = None
    if idea.get("lugar") and not (modalidad == "pregunta_info" and interr == "donde_all"):
        lug = idea["lugar"]
        caso = lug["caso"]
        if lug.get("deictico"):
            lug_txt = DEICTICO_ES[lug["deictico"]]
        elif lug.get("np"):
            np_txt, _, _ = render_np_es(lug["np"], lex_by_id, rol="general")
            lug_txt = {"ALL": _contraer_prep_es("a", np_txt), "ABL": _contraer_prep_es("de", np_txt), "LOC": f"en {np_txt}", "UNTIL": f"hasta {np_txt}"}[caso]

    cc_txt: Optional[str] = None
    if idea.get("comcausa"):
        np_txt, _, _ = render_np_es(idea["comcausa"]["np"], lex_by_id, rol="general")
        cc_txt = f"{'con' if idea['comcausa']['caso'] == 'COM' else 'por'} {np_txt}"

    if modalidad == "pregunta_info":
        int_pal = {"quien_suj": "quién", "que_od": "qué", "donde_all": "a dónde", "quien_oi": "a quién"}[interr]
        partes = [int_pal, bloque_verbo]
        if suj_txt and interr != "quien_suj":
            partes.append(suj_txt)
        partes.extend(e for e in [od_txt, oi_txt, lug_txt, cc_txt, adv_txt] if e)
        return _capitalizar_oracion(" ".join(partes), True)

    partes = []
    if suj_txt:
        partes.append(suj_txt)
    if modalidad == "negativa":
        partes.append("no")
    partes.append(bloque_verbo)
    partes.extend(e for e in [od_txt, oi_txt, lug_txt, cc_txt, adv_txt] if e)
    return _capitalizar_oracion(" ".join(partes), es_pregunta)


# --- Constructor de oraciones aymara (B y A, §9) ---
def _construir_np_aymara(np: Dict[str, Any], lex_by_id: Dict[str, Dict[str, Any]], sufijos_nucleo: List[str]) -> List[Tuple[str, List[str]]]:
    palabras: List[Tuple[str, List[str]]] = []
    tiene_poseedor = bool(np.get("poseedor"))
    if tiene_poseedor:
        pos_np = np["poseedor"]
        palabras.append((lex_by_id[pos_np["n"]]["raiz"], (["PL"] if pos_np.get("plural") else []) + ["GEN"]))
    if np.get("dem"):
        palabras.append((PALABRAS_GRAM[np["dem"]]["b"], []))
    elif np.get("num"):
        palabras.append((PALABRAS_GRAM[np["num"]]["b"], []))
    if np.get("adj"):
        palabras.append((lex_by_id[np["adj"]]["raiz"], []))

    head_ids: List[str] = []
    if np.get("plural") and not np.get("num"):
        head_ids.append("PL")
    if tiene_poseedor:
        head_ids.append("POSS.3")
    elif np.get("poss"):
        head_ids.append(f"POSS.{int(np['poss'])}")
    head_ids.extend(sufijos_nucleo)
    palabras.append((lex_by_id[np["n"]]["raiz"], head_ids))
    return palabras


def _pronombre_base_y_ids(persona: int, plural: bool) -> Tuple[str, List[str]]:
    if persona == 1:
        return ("nänaka", []) if plural else ("naya", [])
    if persona == 2:
        return ("juma", ["PL"] if plural else [])
    if persona == 3:
        return ("jupa", ["PL"] if plural else [])
    return ("jiwasa", ["PL"] if plural else [])


def _ids_flexion_verbo(idea: Dict[str, Any]) -> List[str]:
    tiempo, aspecto, modalidad = idea.get("tiempo", "SIMP"), idea.get("aspecto", "simple"), idea["modalidad"]
    p_suj, pl_suj = int(idea["sujeto"]["persona"]), bool(idea["sujeto"].get("plural", False))
    p_obj = int(idea.get("objeto_persona", 3))
    ids: List[str] = []
    if modalidad == "negativa":
        ids.append("NEG")
    elif aspecto == "progresivo":
        ids.append("PROG")
    elif pl_suj:
        ids.append("PL.V")

    if tiempo == "SIMP":
        ids.append(f"T{p_suj}{p_obj}" if p_obj != 3 else f"P{p_suj}")
    elif tiempo == "FUT":
        ids.append(f"FUT.{p_suj}")
    elif tiempo in ("REM.C", "REM.L"):
        ids.extend([tiempo, "P3.REM" if p_suj == 3 else f"P{p_suj}"])
    return ids


def construir_oracion(idea: Dict[str, Any], lex_by_id: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    """Construye {b, a, es, palabras_b, dv} a partir de una idea JSON."""
    pred, modalidad = idea["predicado"], idea["modalidad"]
    interr = idea.get("interrogativo")
    suj = idea["sujeto"]
    p_suj, pl_suj = int(suj["persona"]), bool(suj.get("plural", False))
    expreso = suj.get("expreso", "ninguno")
    p_obj = int(idea.get("objeto_persona", 3))
    pos_wa = idea.get("posicion_wa", "verbo")

    palabras_b: List[Tuple[str, List[str]]] = []
    def add_words(wlist: List[Tuple[str, List[str]]]):
        for b_form, w_ids in wlist:
            palabras_b.append((b_form, list(w_ids)))

    es_quien_suj = (modalidad == "pregunta_info" and interr == "quien_suj")
    tiene_sujeto_expreso = (not es_quien_suj) and (expreso != "ninguno")

    wa_target = None
    if pred == "verbo" and modalidad == "afirmativa":
        if pos_wa == "ninguna":
            wa_target = None
        elif pos_wa == "sujeto":
            wa_target = "sujeto" if tiene_sujeto_expreso else "verbo"
        elif pos_wa == "complemento":
            tiene_comp = bool(idea.get("oi") or idea.get("oi_pronombre_expreso") or idea.get("comcausa") or (idea.get("lugar") and idea["lugar"]["caso"] in ("ALL", "UNTIL")))
            wa_target = "complemento" if tiene_comp else "verbo"
        else:
            wa_target = "verbo"
    elif pred != "verbo" and modalidad == "afirmativa":
        wa_target = "predicado"

    # Posición 1: Sujeto
    if es_quien_suj:
        add_words([("khiti", ["Q.INF"])])
    elif tiene_sujeto_expreso:
        if expreso == "pronombre":
            p_base, p_ids = _pronombre_base_y_ids(p_suj, pl_suj)
            p_ids.append("AFF" if wa_target == "sujeto" else "TOP")
            add_words([(p_base, p_ids)])
        elif expreso == "nombre":
            suj_np = deepcopy(suj["np"])
            if pl_suj and not suj_np.get("num"):
                suj_np["plural"] = True
            add_words(_construir_np_aymara(suj_np, lex_by_id, ["AFF" if wa_target == "sujeto" else "TOP"]))
        elif expreso == "demostrativo":
            d_ids = (["PL"] if pl_suj else []) + ["AFF" if wa_target == "sujeto" else "TOP"]
            add_words([(PALABRAS_GRAM[suj["dem"]]["b"], d_ids)])
        elif expreso == "coordinado":
            c1, c2 = suj["coord"]
            add_words(_construir_np_aymara(c1, lex_by_id, ["COM"]))
            add_words(_construir_np_aymara(c2, lex_by_id, ["COM", "AFF"] if wa_target == "sujeto" else ["COM"]))

    # Posición 2: jani-wa
    if modalidad == "negativa":
        add_words([("jani", ["AFF"])])

    # Posición 3: Tiempo
    if idea.get("adv_t"):
        adv_ids = ["TOP"] if (not tiene_sujeto_expreso and not es_quien_suj) else []
        add_words([(lex_by_id[idea["adv_t"]]["raiz"], adv_ids)])

    if pred == "verbo":
        indices_comp_wa: List[int] = []
        # Posición 4: Lugar
        if modalidad == "pregunta_info" and interr == "donde_all":
            add_words([("kawki", ["ALL", "Q.INF"])])
        elif idea.get("lugar"):
            lug = idea["lugar"]
            caso = lug["caso"]
            if lug.get("deictico"):
                add_words([(PALABRAS_GRAM[lug["deictico"]]["b"], [caso])])
            elif lug.get("np"):
                add_words(_construir_np_aymara(lug["np"], lex_by_id, [caso]))
                if caso in ("ALL", "UNTIL"):
                    indices_comp_wa.append(len(palabras_b) - 1)
        # Posición 5: COM / CAUSE
        if idea.get("comcausa"):
            add_words(_construir_np_aymara(idea["comcausa"]["np"], lex_by_id, [idea["comcausa"]["caso"]]))
            indices_comp_wa.append(len(palabras_b) - 1)
        # Posición 6: OI
        if modalidad == "pregunta_info" and interr == "quien_oi":
            add_words([("khiti", ["COMPL.RU", "Q.INF"])])
        elif idea.get("oi_pronombre_expreso") and p_obj in (1, 2, 4):
            p_base, p_ids = _pronombre_base_y_ids(p_obj, False)
            p_ids.append("COMPL.RU")
            add_words([(p_base, p_ids)])
            indices_comp_wa.append(len(palabras_b) - 1)
        elif idea.get("oi") and idea["oi"].get("np"):
            add_words(_construir_np_aymara(idea["oi"]["np"], lex_by_id, [idea["oi"]["funcion"]]))
            indices_comp_wa.append(len(palabras_b) - 1)

        if wa_target == "complemento" and indices_comp_wa:
            palabras_b[indices_comp_wa[-1]][1].append("AFF")

        # Posición 7: OD
        if modalidad == "pregunta_info" and interr == "que_od":
            add_words([("kuna", ["Q.INF"])])
        elif idea.get("od_coord"):
            add_words(_construir_np_aymara(idea["od_coord"][0], lex_by_id, ["COM"]))
            add_words(_construir_np_aymara(idea["od_coord"][1], lex_by_id, ["COM"]))
        elif idea.get("od"):
            add_words(_construir_np_aymara(idea["od"], lex_by_id, []))

        # Posición 8: Verbo
        v_ids = _ids_flexion_verbo(idea)
        if modalidad == "negativa":
            v_ids.append("NEG.TI")
        elif modalidad == "pregunta_si_no":
            v_ids.append("Q.YN")
        elif wa_target == "verbo":
            v_ids.append("AFF")
        add_words([(lex_by_id[idea["verbo"]]["raiz"], v_ids)])
    else:
        pnom = idea.get("pnom") or {}
        orac_pred = ["AFF"] if modalidad == "afirmativa" else (["Q.YN"] if modalidad == "pregunta_si_no" else [])
        if pred == "ser":
            if pnom.get("adj"):
                p_ids = list(orac_pred) if p_suj == 3 else (["VBLZ.LONG", f"P{p_suj}"] + orac_pred)
                add_words([(lex_by_id[pnom["adj"]]["raiz"], p_ids)])
            else:
                np_p = deepcopy(pnom["np"])
                if p_suj == 3:
                    if pl_suj and not np_p.get("num"):
                        np_p["plural"] = True
                    add_words(_construir_np_aymara(np_p, lex_by_id, orac_pred))
                else:
                    add_words([(lex_by_id[np_p["n"]]["raiz"], ["VBLZ.LONG", f"P{p_suj}"] + orac_pred)])
        elif pred == "estar":
            if modalidad == "pregunta_info" and interr == "donde_estar":
                add_words([("kawki", ["LOC", "VBLZ.KA", f"P{p_suj}", "Q.INF"])])
            elif pnom.get("deictico"):
                add_words([(PALABRAS_GRAM[pnom["deictico"]]["b"], ["LOC", "VBLZ.KA", f"P{p_suj}"] + orac_pred)])
            else:
                add_words(_construir_np_aymara(pnom["np"], lex_by_id, ["LOC", "VBLZ.KA", f"P{p_suj}"] + orac_pred))
        else:  # tener
            suf = (["POSR"] + orac_pred) if p_suj == 3 else (["POSR", "VBLZ.LONG", f"P{p_suj}"] + orac_pred)
            add_words(_construir_np_aymara(pnom["np"], lex_by_id, suf))

    es_pregunta = modalidad in ("pregunta_si_no", "pregunta_info")
    toks_b, toks_a, dvs = [], [], set()
    for base_b, w_ids in palabras_b:
        toks_b.append(construir_palabra(base_b, w_ids, modo="B"))
        toks_a.append(construir_palabra(base_b, w_ids, modo="A"))
        dvs.update(detectar_dv_palabra(w_ids))

    return {
        "b": _capitalizar_oracion(" ".join(toks_b), es_pregunta),
        "a": _capitalizar_oracion(" ".join(toks_a), es_pregunta),
        "es": construir_espanol(idea, lex_by_id),
        "palabras_b": [[b, ids] for b, ids in palabras_b],
        "dv": sorted(dvs),
    }

print("Constructor de oraciones B, A y Español cargado.")

## 8. Gate de Casos de Prueba Oficiales (`CH-1201`: `C01`–`C40`)

Verifica los **40 casos oficiales** de §12 (17 `[ATESTIGUADO]` de L2 + 23 casos con raíces de ejemplo), comprobando coincidencia carácter por carácter en **Modo B**, **Modo A**, **Español** y etiquetas **`[DERIVADA]`**.

In [ ]:
CASOS_PRUEBA_C40 = [{'id': 'C01', 'desc': 'SIMP 1>3 · V_DITR · COMPL.RU · OD · pronombre 1 · posicion_wa=ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV03', 'sujeto': {'persona': 1, 'expreso': 'pronombre'}, 'oi': {'funcion': 'COMPL.RU', 'np': {'n': 'LV01', 'genero': 'm'}}, 'od': {'n': 'LV02'}, 'posicion_wa': 'ninguna'}, 'b': "Nayaxa wawaru t'ant'a churtha.", 'a': "Muqixu wawallu t'ant'a churphi.", 'es': 'Yo le doy pan al niño.', 'dv': []}, {'id': 'C02', 'desc': 'SIMP 3>3 · sujeto omitido · COMPL.RU + POSS.2 · ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV03', 'sujeto': {'persona': 3, 'expreso': 'ninguno', 'genero': 'm'}, 'oi': {'funcion': 'COMPL.RU', 'np': {'n': 'LV04', 'poss': 2}}, 'od': {'n': 'LV02'}, 'posicion_wa': 'ninguna'}, 'b': "Tatamaru t'ant'a churi.", 'a': "Tataqillu t'ant'a churu.", 'es': 'Le da pan a tu padre.', 'dv': []}, {'id': 'C03', 'desc': 'FUT 3 · V_MOV · ALL · ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'FUT', 'verbo': 'LV06', 'sujeto': {'persona': 3, 'expreso': 'ninguno', 'genero': 'm'}, 'lugar': {'caso': 'ALL', 'np': {'n': 'LV05'}}, 'posicion_wa': 'ninguna'}, 'b': 'Markaru jutani.', 'a': 'Markallu jutaphu.', 'es': 'Vendrá al pueblo.', 'dv': []}, {'id': 'C04', 'desc': 'SIMP 1 · AFF en verbo', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV07', 'sujeto': {'persona': 1, 'expreso': 'pronombre'}, 'posicion_wa': 'verbo'}, 'b': 'Nayaxa sarthwa.', 'a': 'Muqixu sarphyi.', 'es': 'Yo voy.', 'dv': []}, {'id': 'C05', 'desc': 'SIMP 2 · AFF en verbo', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV08', 'sujeto': {'persona': 2, 'expreso': 'pronombre'}, 'posicion_wa': 'verbo'}, 'b': "Jumaxa manq'tawa.", 'a': "Tiluxu manq'kuyi.", 'es': 'Tú comes.', 'dv': []}, {'id': 'C06', 'desc': 'SIMP 4 · AFF en verbo', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV08', 'sujeto': {'persona': 4, 'expreso': 'pronombre'}, 'posicion_wa': 'verbo'}, 'b': "Jiwasaxa manq'tanwa.", 'a': "Wiqutuxu manq'kilyi.", 'es': 'Nosotros comemos.', 'dv': []}, {'id': 'C07', 'desc': 'SIMP 3 · OD · sujeto nombre · ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV09', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'LV10'}}, 'od': {'n': 'LV02'}, 'posicion_wa': 'ninguna'}, 'b': "Anuxa t'ant'a manq'i.", 'a': "Anuxu t'ant'a manq'u.", 'es': 'El perro come pan.', 'dv': []}, {'id': 'C08', 'desc': 'SIMP 3 · ALL · posicion_wa=complemento', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV07', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'LV04'}}, 'lugar': {'caso': 'ALL', 'np': {'n': 'LV11'}}, 'posicion_wa': 'complemento'}, 'b': 'Tataxa utaruwa sari.', 'a': 'Tataxu utalluyi saru.', 'es': 'El padre va a la casa.', 'dv': []}, {'id': 'C09', 'desc': 'Negativa FUT 3', 'idea': {'predicado': 'verbo', 'modalidad': 'negativa', 'tiempo': 'FUT', 'verbo': 'LV06', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'LV13'}}}, 'b': 'Jilaqataxa janiwa jutkaniti.', 'a': 'Jilaqataxu yuliyi jutñuphuji.', 'es': 'La autoridad no vendrá.', 'dv': []}, {'id': 'C10', 'desc': 'Pregunta sí/no FUT 3 · POSS.2', 'idea': {'predicado': 'verbo', 'modalidad': 'pregunta_si_no', 'tiempo': 'FUT', 'verbo': 'LV06', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'LV12', 'poss': 2, 'genero': 'm'}}}, 'b': '¿Jilamaxa jutaniti?', 'a': '¿Jilaqixu jutaphuji?', 'es': '¿Tu hermano vendrá?', 'dv': []}, {'id': 'C11', 'desc': 'Coordinación de sujeto · PL.V · posicion_wa=sujeto', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV17', 'sujeto': {'persona': 3, 'plural': True, 'expreso': 'coordinado', 'coord': [{'n': 'LV15'}, {'n': 'LV16'}]}, 'posicion_wa': 'sujeto'}, 'b': 'Tiwulampi wallatampiwa jikisipxi.', 'a': 'Tiwulanku wallatankuyi jikisimtu.', 'es': 'El zorro y el ganso se encuentran.', 'dv': []}, {'id': 'C12', 'desc': 'Plural 3 (pronombre + PL.V) · ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV07', 'sujeto': {'persona': 3, 'plural': True, 'expreso': 'pronombre', 'genero': 'm'}, 'posicion_wa': 'ninguna'}, 'b': 'Jupanakaxa sarapxi.', 'a': 'Kaxisuluxu saramtu.', 'es': 'Ellos van.', 'dv': []}, {'id': 'C13', 'desc': 'SIMP 1 · sujeto omitido · OD · ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV18', 'sujeto': {'persona': 1, 'expreso': 'ninguno'}, 'od': {'n': 'LV02'}, 'posicion_wa': 'ninguna'}, 'b': "T'ant'a muntha.", 'a': "T'ant'a munphi.", 'es': 'Quiero pan.', 'dv': []}, {'id': 'C14', 'desc': 'FUT 3 · POSS.1 · posicion_wa=sujeto', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'FUT', 'verbo': 'LV06', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'LV14', 'poss': 1}}, 'posicion_wa': 'sujeto'}, 'b': 'Mamajawa jutani.', 'a': 'Mamaluyi jutaphu.', 'es': 'Mi mamá vendrá.', 'dv': []}, {'id': 'C15', 'desc': 'Pregunta de información (donde_all) · FUT 4', 'idea': {'predicado': 'verbo', 'modalidad': 'pregunta_info', 'tiempo': 'FUT', 'verbo': 'LV07', 'sujeto': {'persona': 4, 'expreso': 'ninguno'}, 'interrogativo': 'donde_all', 'lugar': {'caso': 'ALL'}}, 'b': '¿Kawkirusa sarañäni?', 'a': '¿Yaqullukhu sarathäqi?', 'es': '¿A dónde iremos?', 'dv': []}, {'id': 'C16', 'desc': 'UNTIL · posicion_wa=complemento', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'LV07', 'sujeto': {'persona': 3, 'expreso': 'ninguno', 'genero': 'm'}, 'lugar': {'caso': 'UNTIL', 'np': {'n': 'LV05'}}, 'posicion_wa': 'complemento'}, 'b': 'Markakamawa sari.', 'a': 'Markachiwiyi saru.', 'es': 'Va hasta el pueblo.', 'dv': []}, {'id': 'C17', 'desc': 'FUT 1 · ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'FUT', 'verbo': 'LV07', 'sujeto': {'persona': 1, 'expreso': 'pronombre'}, 'posicion_wa': 'ninguna'}, 'b': 'Nayaxa sarä.', 'a': 'Muqixu sarä.', 'es': 'Yo iré.', 'dv': []}, {'id': 'C18', 'desc': 'Transición 1>2', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'L0017', 'sujeto': {'persona': 1, 'expreso': 'pronombre'}, 'objeto_persona': 2, 'posicion_wa': 'verbo'}, 'b': "Nayaxa ch'ixrsmawa.", 'a': "Muqixu ch'ixrjmiyi.", 'es': 'Yo te veo.', 'dv': ['DV-02']}, {'id': 'C19', 'desc': 'Transición 2>1 · pregunta sí/no', 'idea': {'predicado': 'verbo', 'modalidad': 'pregunta_si_no', 'tiempo': 'SIMP', 'verbo': 'L0017', 'sujeto': {'persona': 2, 'expreso': 'pronombre'}, 'objeto_persona': 1}, 'b': "¿Jumaxa ch'ixristati?", 'a': "¿Tiluxu ch'ixrulqaji?", 'es': '¿Tú me ves?', 'dv': ['DV-02']}, {'id': 'C20', 'desc': 'Transición 3>1 · V_DITR · pronombre con COMPL.RU · demostrativo', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'L0018', 'sujeto': {'persona': 3, 'expreso': 'pronombre', 'genero': 'm'}, 'objeto_persona': 1, 'oi_pronombre_expreso': True, 'oi': {'funcion': 'COMPL.RU'}, 'od': {'n': 'L0006', 'dem': 'DEM.1'}, 'posicion_wa': 'verbo'}, 'b': "Jupaxa nayaru aka q'aspi tulphituwa.", 'a': "Kaxixu muqillu ilmi q'aspi tulphuquyi.", 'es': 'Él me da este pan.', 'dv': ['DV-02']}, {'id': 'C21', 'desc': 'Transición 3>2 · adverbio pasado con TOP (sin sujeto) · SIMP → pretérito', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'verbo': 'L0017', 'sujeto': {'persona': 3, 'expreso': 'ninguno', 'genero': 'f'}, 'objeto_persona': 2, 'adv_t': 'L0023', 'posicion_wa': 'verbo'}, 'b': "Jaskhixa ch'ixrtamwa.", 'a': "Jaskhixu ch'ixrkiqyi.", 'es': 'Te vio ayer.', 'dv': ['DV-01']}, {'id': 'C22', 'desc': 'Transición 3>4 · negativa SIMP', 'idea': {'predicado': 'verbo', 'modalidad': 'negativa', 'tiempo': 'SIMP', 'verbo': 'L0017', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'L0008'}}, 'objeto_persona': 4}, 'b': "Kirpuxa janiwa ch'ixrkistuti.", 'a': "Kirpuxu yuliyi ch'ixrñulquji.", 'es': 'El perro no nos ve.', 'dv': ['DV-02', 'DV-08']}, {'id': 'C23', 'desc': 'REM.C 1 plural (nänaka + PL.V) · ABL', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'REM.C', 'verbo': 'L0014', 'sujeto': {'persona': 1, 'plural': True, 'expreso': 'pronombre'}, 'lugar': {'caso': 'ABL', 'np': {'n': 'L0009'}}, 'posicion_wa': 'verbo'}, 'b': 'Nänakaxa pinqarata pajllapxayäthwa.', 'a': 'Müsuluxu pinqaraku pajllamtujüphyi.', 'es': 'Nosotros hemos venido del mercado.', 'dv': ['DV-05']}, {'id': 'C24', 'desc': 'REM.L 2 · coordinación de OD · ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'REM.L', 'verbo': 'L0017', 'sujeto': {'persona': 2, 'expreso': 'pronombre'}, 'od_coord': [{'n': 'L0008'}, {'n': 'L0011'}], 'posicion_wa': 'ninguna'}, 'b': "Jumaxa kirpumpi jitqumpi ch'ixratäta.", 'a': "Tiluxu kirpunku jitqunku ch'ixrawüku.", 'es': 'Tú habías visto el perro y el cuchillo.', 'dv': []}, {'id': 'C25', 'desc': 'REM.L 4 plural · ALL + CAUSE · posicion_wa=complemento', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'REM.L', 'verbo': 'L0013', 'sujeto': {'persona': 4, 'plural': True, 'expreso': 'pronombre'}, 'lugar': {'caso': 'ALL', 'np': {'n': 'L0007'}}, 'comcausa': {'caso': 'CAUSE', 'np': {'n': 'L0012'}}, 'posicion_wa': 'complemento'}, 'b': "Jiwasanakaxa wirkharu ch'alqulaykuwa tisqapxatätana.", 'a': "Wiqutusuluxu wirkhallu ch'alqutiwqayi tisqamtuwükilu.", 'es': 'Nosotros habíamos ido al pueblo por la deuda.', 'dv': ['DV-06']}, {'id': 'C26', 'desc': 'REM.C 3 plural (nombre + PL.V) · BEN · OD', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'REM.C', 'verbo': 'L0019', 'sujeto': {'persona': 3, 'plural': True, 'expreso': 'nombre', 'np': {'n': 'L0003', 'plural': True, 'genero': 'm'}}, 'oi': {'funcion': 'BEN', 'np': {'n': 'L0001', 'genero': 'f'}}, 'od': {'n': 'L0006'}, 'posicion_wa': 'verbo'}, 'b': "Yaruqhinakaxa lisqutaki q'aspi nirqapxänawa.", 'a': "Yaruqhisuluxu lisqupuli q'aspi nirqamtümuyi.", 'es': 'Los maestros han comprado pan para la niña.', 'dv': ['DV-05']}, {'id': 'C27', 'desc': 'FUT 2 · GEN (poseedor) + POSS.3 · adverbio futuro · ninguna', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'FUT', 'verbo': 'L0017', 'sujeto': {'persona': 2, 'expreso': 'pronombre'}, 'adv_t': 'L0024', 'od': {'n': 'L0005', 'poseedor': {'n': 'L0003', 'genero': 'm'}}, 'posicion_wa': 'ninguna'}, 'b': "Jumaxa sik'uyu yaruqhina chimulapa ch'ixräta.", 'a': "Tiluxu sik'uyu yaruqhimu chimulachu ch'ixräku.", 'es': 'Tú verás la casa del maestro mañana.', 'dv': []}, {'id': 'C28', 'desc': 'Negativa FUT 4 · COM instrumento · OD', 'idea': {'predicado': 'verbo', 'modalidad': 'negativa', 'tiempo': 'FUT', 'verbo': 'L0020', 'sujeto': {'persona': 4, 'expreso': 'pronombre'}, 'comcausa': {'caso': 'COM', 'np': {'n': 'L0011'}}, 'od': {'n': 'L0006'}}, 'b': "Jiwasaxa janiwa jitqumpi q'aspi k'iymkañäniti.", 'a': "Wiqutuxu yuliyi jitqunku q'aspi k'iymñuthäqiji.", 'es': 'Nosotros no cortaremos pan con el cuchillo.', 'dv': ['DV-04', 'DV-08']}, {'id': 'C29', 'desc': 'Negativa SIMP 2 plural · LOC', 'idea': {'predicado': 'verbo', 'modalidad': 'negativa', 'tiempo': 'SIMP', 'verbo': 'L0015', 'sujeto': {'persona': 2, 'plural': True, 'expreso': 'pronombre'}, 'lugar': {'caso': 'LOC', 'np': {'n': 'L0005'}}}, 'b': "Jumanakaxa janiwa chimulana nuyk'ktati.", 'a': "Tilusuluxu yuliyi chimulamu nuyk'ñkuji.", 'es': 'Ustedes no duermen en la casa.', 'dv': ['DV-03', 'DV-08']}, {'id': 'C30', 'desc': 'Progresivo 1 · adverbio presente · LOC deíctico · OD', 'idea': {'predicado': 'verbo', 'modalidad': 'afirmativa', 'tiempo': 'SIMP', 'aspecto': 'progresivo', 'verbo': 'L0020', 'sujeto': {'persona': 1, 'expreso': 'pronombre'}, 'adv_t': 'L0025', 'lugar': {'caso': 'LOC', 'deictico': 'DEM.1'}, 'od': {'n': 'L0006'}, 'posicion_wa': 'verbo'}, 'b': "Nayaxa nuspi akana q'aspi k'iymaskthwa.", 'a': "Muqixu nuspi ilmimu q'aspi k'iymanchphyi.", 'es': 'Yo estoy cortando pan aquí ahora.', 'dv': ['DV-09']}, {'id': 'C31', 'desc': 'Pregunta de información (quien_suj) · FUT · ALL', 'idea': {'predicado': 'verbo', 'modalidad': 'pregunta_info', 'tiempo': 'FUT', 'verbo': 'L0013', 'sujeto': {'persona': 3}, 'interrogativo': 'quien_suj', 'lugar': {'caso': 'ALL', 'np': {'n': 'L0009'}}}, 'b': '¿Khitisa pinqararu tisqani?', 'a': '¿Simukhu pinqarallu tisqaphu?', 'es': '¿Quién irá al mercado?', 'dv': []}, {'id': 'C32', 'desc': 'Pregunta de información (que_od) · REM.C 2', 'idea': {'predicado': 'verbo', 'modalidad': 'pregunta_info', 'tiempo': 'REM.C', 'verbo': 'L0019', 'sujeto': {'persona': 2, 'expreso': 'pronombre'}, 'interrogativo': 'que_od'}, 'b': '¿Jumaxa kunasa nirqayäta?', 'a': '¿Tiluxu mulukhu nirqajüku?', 'es': '¿Qué has comprado tú?', 'dv': []}, {'id': 'C33', 'desc': 'Pregunta de información (quien_oi) · SIMP 2', 'idea': {'predicado': 'verbo', 'modalidad': 'pregunta_info', 'tiempo': 'SIMP', 'verbo': 'L0018', 'sujeto': {'persona': 2, 'expreso': 'ninguno'}, 'interrogativo': 'quien_oi', 'oi': {'funcion': 'COMPL.RU'}, 'od': {'n': 'L0006'}}, 'b': "¿Khitirusa q'aspi tulphta?", 'a': "¿Simullukhu q'aspi tulphku?", 'es': '¿A quién le das pan?', 'dv': []}, {'id': 'C34', 'desc': 'Estar · pregunta de información (donde_estar) · 3', 'idea': {'predicado': 'estar', 'modalidad': 'pregunta_info', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'L0008'}}, 'interrogativo': 'donde_estar', 'pnom': {}}, 'b': '¿Kirpuxa kawkinkisa?', 'a': '¿Kirpuxu yaqumñukhu?', 'es': '¿Dónde está el perro?', 'dv': ['DV-10']}, {'id': 'C35', 'desc': 'Ser + adjetivo · 4 plural', 'idea': {'predicado': 'ser', 'modalidad': 'afirmativa', 'sujeto': {'persona': 4, 'plural': True, 'expreso': 'pronombre'}, 'pnom': {'adj': 'L0021'}}, 'b': 'Jiwasanakaxa jumpütanwa.', 'a': 'Wiqutusuluxu jumpükilyi.', 'es': 'Nosotros somos grandes.', 'dv': ['DV-12']}, {'id': 'C36', 'desc': 'Ser + nombre · demostrativo plural (3)', 'idea': {'predicado': 'ser', 'modalidad': 'afirmativa', 'sujeto': {'persona': 3, 'plural': True, 'expreso': 'demostrativo', 'dem': 'DEM.2'}, 'pnom': {'np': {'n': 'L0005'}}}, 'b': 'Ukanakaxa chimulanakawa.', 'a': 'Ulmisuluxu chimulasuluyi.', 'es': 'Esas son casas.', 'dv': []}, {'id': 'C37', 'desc': 'Ser + nombre humano (3) · sin artículo', 'idea': {'predicado': 'ser', 'modalidad': 'afirmativa', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'L0004'}}, 'pnom': {'np': {'n': 'L0003', 'genero': 'f'}}}, 'b': 'Simllaxa yaruqhiwa.', 'a': 'Simllaxu yaruqhiyi.', 'es': 'La mujer es maestra.', 'dv': []}, {'id': 'C38', 'desc': 'Estar + POSS.4 · pregunta sí/no · 3', 'idea': {'predicado': 'estar', 'modalidad': 'pregunta_si_no', 'sujeto': {'persona': 3, 'expreso': 'nombre', 'np': {'n': 'L0008'}}, 'pnom': {'np': {'n': 'L0005', 'poss': 4}}}, 'b': '¿Kirpuxa chimulasankiti?', 'a': '¿Kirpuxu chimulakhumñuji?', 'es': '¿El perro está en nuestra casa?', 'dv': ['DV-10', 'DV-13']}, {'id': 'C39', 'desc': 'Tener + adjetivo · pregunta sí/no · 2', 'idea': {'predicado': 'tener', 'modalidad': 'pregunta_si_no', 'sujeto': {'persona': 2, 'expreso': 'pronombre'}, 'pnom': {'np': {'n': 'L0005', 'adj': 'L0021'}}}, 'b': '¿Jumaxa jumpu chimulanitati?', 'a': '¿Tiluxu jumpu chimulaphukuji?', 'es': '¿Tú tienes una casa grande?', 'dv': ['DV-11', 'DV-13']}, {'id': 'C40', 'desc': 'Tener + numeral · 3 plural', 'idea': {'predicado': 'tener', 'modalidad': 'afirmativa', 'sujeto': {'persona': 3, 'plural': True, 'expreso': 'nombre', 'np': {'n': 'L0002', 'plural': True, 'genero': 'm'}}, 'pnom': {'np': {'n': 'L0008', 'num': 'NUM.3'}}}, 'b': 'Pultiyanakaxa kimsa kirpuniwa.', 'a': 'Pultiyasuluxu musqi kirpuphuyi.', 'es': 'Los hermanos tienen tres perros.', 'dv': []}]

lex_val = cargar_csv_str(LEXICO_VALIDACION_CSV) + cargar_csv_str(LEXICO_EJEMPLO_SPEC_CSV)
lex_val_by_id = {e["id"]: e for e in lex_val}

for c in CASOS_PRUEBA_C40:
    res = construir_oracion(c["idea"], lex_val_by_id)
    assert res["b"] == c["b"], f"{c['id']} B: '{res['b']}' != '{c['b']}'"
    assert res["a"] == c["a"], f"{c['id']} A: '{res['a']}' != '{c['a']}'"
    assert res["es"] == c["es"], f"{c['id']} ES: '{res['es']}' != '{c['es']}'"
    assert res["dv"] == c["dv"], f"{c['id']} DV: {res['dv']} != {c['dv']}"

print(f"OK! Los {len(CASOS_PRUEBA_C40)}/{len(CASOS_PRUEBA_C40)} casos de prueba (C01-C40) pasan al 100% en B, A, ES y DV.")

## 9. Validador de Ideas (`R1`–`R20` + `CH-406`), Generador Estratificado (§11) y Verificador `V01`–`V31` (§13)

In [ ]:
# ==============================================================================
# 8. VALIDADOR DE IDEAS R1-R20 Y CH-406
# ==============================================================================

CLASES_TENER_POSEIDO = {"animal", "comida", "bebida", "ropa", "herramienta", "recipiente", "planta", "vivienda", "objeto"}
CLASES_ESTAR_LUGAR = {"lugar", "vivienda", "recipiente"}
CLASES_ESTAR_SUJ_PROH = {"lugar", "tiempo", "abstracto"}
CLASES_ALL_ABL = {"lugar", "vivienda"}
CLASES_UNTIL = {"lugar", "vivienda", "tiempo"}
CLASES_CAUSE = {"abstracto", "persona"}
CLASES_COM_COMP = {"persona", "animal"}


def _parse_clases(val: str) -> Set[str]:
    if not val or val in ("-", "NINGUNA"):
        return set()
    return set(val.split("|"))


def _validar_np(np: Dict[str, Any], lex_by_id: Dict[str, Dict[str, Any]], errores: List[str], ctx: str):
    if np["n"] not in lex_by_id:
        errores.append(f"{ctx}: id léxico desconocido {np['n']}")
        return
    ent = lex_by_id[np["n"]]
    # R12: num excluye plural
    if np.get("num") and np.get("plural"):
        errores.append(f"{ctx}: R12 numeral y plural juntos")
    # R13: solo NUM.3/4/6/10 y DEM.1/2/3
    if np.get("num") and np["num"] not in {"NUM.3", "NUM.4", "NUM.6", "NUM.10"}:
        errores.append(f"{ctx}: R13 numeral prohibido {np['num']}")
    if np.get("dem") and np["dem"] not in {"DEM.1", "DEM.2", "DEM.3"}:
        errores.append(f"{ctx}: R13 demostrativo prohibido {np['dem']}")
    # CH-912: dem y num excluyentes; dem no va con poss ni poseedor
    if np.get("dem") and np.get("num"):
        errores.append(f"{ctx}: CH-912 dem y num juntos")
    if np.get("dem") and (np.get("poss") or np.get("poseedor")):
        errores.append(f"{ctx}: CH-912 dem con poss/poseedor")
    # CH-406: adjetivo compatible con mod_clases
    if np.get("adj"):
        adj_ent = lex_by_id.get(np["adj"])
        if not adj_ent or adj_ent["cat"] != "ADJ":
            errores.append(f"{ctx}: adj inválido {np['adj']}")
        else:
            mod_c = _parse_clases(adj_ent["mod_clases"])
            if ent["clase_sem"] not in mod_c:
                errores.append(f"{ctx}: R20 adj '{adj_ent['es_m']}' incompatible con clase '{ent['clase_sem']}'")
    # Poseedor (CH-911, CH-406)
    if np.get("poseedor"):
        if ent["clase_sem"] == "tiempo":
            errores.append(f"{ctx}: R20 nombre de tiempo no puede ser poseído")
        pos_ent = lex_by_id.get(np["poseedor"]["n"])
        if not pos_ent or pos_ent["cat"] != "N_HUM":
            errores.append(f"{ctx}: CH-911 poseedor debe ser N_HUM")
        if pos_ent and pos_ent["genero"] == "var" and np["poseedor"].get("genero") not in ("m", "f"):
            errores.append(f"{ctx}: R19 falta género en poseedor var")
    if np.get("poss") and ent["clase_sem"] == "tiempo":
        errores.append(f"{ctx}: R20 nombre de tiempo no puede llevar posesivo")
    # R19: género obligatorio en N_HUM var
    if ent["cat"] == "N_HUM" and ent["genero"] == "var" and np.get("genero") not in ("m", "f"):
        errores.append(f"{ctx}: R19 falta género en N_HUM var '{ent['es_m']}'")


def validar_idea(idea: Dict[str, Any], lex_by_id: Dict[str, Dict[str, Any]]) -> List[str]:
    """CH-1103: Comprueba R1 a R20 y CH-406. Devuelve lista de errores (vacía = válida)."""
    errores: List[str] = []
    pred = idea.get("predicado")
    mod = idea.get("modalidad")
    tiempo = idea.get("tiempo", "SIMP")
    asp = idea.get("aspecto", "simple")
    suj = idea.get("sujeto", {})
    p_suj = int(suj.get("persona", 3))
    pl_suj = bool(suj.get("plural", False))
    expreso = suj.get("expreso", "ninguno")
    p_obj = int(idea.get("objeto_persona", 3))
    interr = idea.get("interrogativo")

    # R15: predicado nominal
    if pred != "verbo":
        if tiempo != "SIMP":
            errores.append("R15: predicado nominal solo en SIMP")
        if pred == "estar":
            if mod not in ("afirmativa", "pregunta_si_no", "pregunta_info"):
                errores.append("R15: estar modalidad inválida")
            if mod == "pregunta_info" and interr != "donde_estar":
                errores.append("R15/R17: estar pregunta_info exige donde_estar")
        else:
            if mod not in ("afirmativa", "pregunta_si_no"):
                errores.append("R15: ser/tener solo afirmativa o pregunta_si_no")
        if asp != "simple":
            errores.append("R15: predicado nominal no admite progresivo")

    # R3: negativa
    if mod == "negativa":
        if tiempo not in ("SIMP", "FUT"):
            errores.append("R3: negativa solo en SIMP o FUT")
        if asp != "simple":
            errores.append("R3: negativa no admite progresivo")
        if pl_suj:
            if expreso not in ("pronombre", "nombre"):
                errores.append("R11: negativa plural exige sujeto expreso pronombre o nombre")
            if expreso == "nombre" and suj.get("np", {}).get("num"):
                errores.append("R11: negativa plural prohíbe numeral en sujeto")

    # R4: progresivo
    if asp == "progresivo" and tiempo != "SIMP":
        errores.append("R4: progresivo solo en SIMP")
    if asp == "progresivo" and pl_suj:
        errores.append("CH-603: PROG y PL.V se excluyen")

    # R11: plural en predicados nominales
    if pred != "verbo" and pl_suj:
        if expreso not in ("pronombre", "nombre", "demostrativo"):
            errores.append("R11: predicado nominal plural exige sujeto expreso")
        if expreso == "nombre" and suj.get("np", {}).get("num"):
            errores.append("R11: predicado nominal plural exige -naka (sin numeral)")

    # R14: adv_t compatible con tiempo
    if idea.get("adv_t"):
        adv_ent = lex_by_id.get(idea["adv_t"])
        if not adv_ent or adv_ent["cat"] != "ADV_T":
            errores.append("R14: adv_t inválido")
        else:
            sc = adv_ent["subcat"]
            if sc == "pasado" and (tiempo == "FUT" or asp == "progresivo"):
                errores.append("R14: adv pasado incompatible con FUT o progresivo")
            elif sc == "presente" and tiempo != "SIMP":
                errores.append("R14: adv presente solo con SIMP")
            elif sc == "futuro" and tiempo != "FUT":
                errores.append("R14: adv futuro solo con FUT")

    # Validar sujeto NP / coord / genero
    suj_clases_reales: Set[str] = set()
    if expreso == "pronombre":
        suj_clases_reales = {"persona"}
        if p_suj == 3 and suj.get("genero") not in ("m", "f"):
            errores.append("R19: pronombre 3.ª requiere genero m/f")
    elif expreso == "ninguno":
        suj_clases_reales = {"persona"}
    elif expreso == "nombre":
        _validar_np(suj["np"], lex_by_id, errores, "sujeto")
        if suj["np"]["n"] in lex_by_id:
            suj_clases_reales = {lex_by_id[suj["np"]["n"]]["clase_sem"]}
        if suj["np"].get("num") and not pl_suj:
            errores.append("R11: sujeto con numeral debe tener sujeto.plural=True")
    elif expreso == "coordinado":
        if mod in ("negativa", "pregunta_info"):
            errores.append("R18: coordinación prohibida en negativa o pregunta_info")
        if not pl_suj or p_suj != 3:
            errores.append("CH-907: sujeto coordinado debe ser 3.ª plural")
        if idea.get("comcausa") and idea["comcausa"]["caso"] == "COM":
            errores.append("R18: coordinación excluye complemento COM")
        for idx_c, cnp in enumerate(suj.get("coord", [])):
            _validar_np(cnp, lex_by_id, errores, f"suj_coord[{idx_c}]")
            if cnp["n"] in lex_by_id:
                suj_clases_reales.add(lex_by_id[cnp["n"]]["clase_sem"])
    elif expreso == "demostrativo":
        if pred != "ser" or not (idea.get("pnom") or {}).get("np"):
            errores.append("R16: demostrativo sujeto solo en ser + nombre")

    # R17: interrogativo
    if interr is not None:
        if mod != "pregunta_info":
            errores.append("R17: interrogativo exige modalidad=pregunta_info")
    elif mod == "pregunta_info":
        errores.append("R17: pregunta_info exige interrogativo")

    if pred == "verbo":
        v_ent = lex_by_id.get(idea.get("verbo", ""))
        if not v_ent or not v_ent["cat"].startswith("V_"):
            errores.append("Verbo inválido")
            return errores
        v_cat = v_ent["cat"]

        # R1 y R2: transiciones
        if p_obj != 3:
            if tiempo != "SIMP" or v_ent.get("obj_persona") != "s":
                errores.append("R1: transición solo en SIMP con obj_persona=s")
            if (p_suj, p_obj) not in {(1, 2), (2, 1), (3, 1), (3, 2), (3, 4)}:
                errores.append(f"R2: transición inválida {p_suj}>{p_obj}")

        # R20: sujeto compatible con suj_clases del verbo
        v_suj_c = _parse_clases(v_ent["suj_clases"])
        if interr == "quien_suj":
            if p_suj != 3 or pl_suj or "persona" not in v_suj_c:
                errores.append("R17: quien_suj exige 3.ª singular y verbo con suj_clases ∋ persona")
        elif suj_clases_reales and not suj_clases_reales.issubset(v_suj_c):
            errores.append(f"R20: sujeto {suj_clases_reales} incompatible con suj_clases {v_suj_c}")

        # R7 y R8: OD
        v_obj_c = _parse_clases(v_ent["obj_clases"])
        tiene_od = bool(idea.get("od")) or bool(idea.get("od_coord")) or (interr == "que_od")
        if v_cat in ("V_TR", "V_DITR"):
            if p_obj == 3 and not tiene_od:
                errores.append("R8: V_TR/V_DITR con objeto 3 exige OD o que_od")
            if v_cat == "V_TR" and p_obj != 3 and tiene_od:
                errores.append("R8: V_TR con objeto 1/2/4 prohíbe OD")
            if v_cat == "V_DITR" and p_obj != 3 and not tiene_od:
                errores.append("R8: V_DITR con objeto 1/2/4 exige OD")
        else:
            if tiene_od:
                errores.append("R8: V_INTR/V_MOV no admiten OD")

        if idea.get("od"):
            _validar_np(idea["od"], lex_by_id, errores, "od")
            od_ent = lex_by_id.get(idea["od"]["n"])
            if od_ent:
                if od_ent["cat"] == "N_HUM" or od_ent["clase_sem"] not in v_obj_c:
                    errores.append(f"R7/R20: OD '{od_ent['es_m']}' incompatible con obj_clases {v_obj_c}")
        if idea.get("od_coord"):
            if mod in ("negativa", "pregunta_info"):
                errores.append("R18: od_coord prohibido en negativa/pregunta_info")
            if idea.get("comcausa") and idea["comcausa"]["caso"] == "COM":
                errores.append("R18: od_coord excluye complemento COM")
            for idx_c, cnp in enumerate(idea["od_coord"]):
                _validar_np(cnp, lex_by_id, errores, f"od_coord[{idx_c}]")
                od_ent = lex_by_id.get(cnp["n"])
                if od_ent and (od_ent["cat"] == "N_HUM" or od_ent["clase_sem"] not in v_obj_c):
                    errores.append(f"R7/R20: od_coord incompatible")

        # R5 y R6: OI
        if interr == "quien_oi":
            if v_cat != "V_DITR" or p_obj != 3:
                errores.append("R17: quien_oi exige V_DITR en X>3")
        if idea.get("oi") and idea["oi"].get("np"):
            oi = idea["oi"]
            _validar_np(oi["np"], lex_by_id, errores, "oi")
            oi_ent = lex_by_id.get(oi["np"]["n"])
            if oi["funcion"] == "COMPL.RU":
                if v_ent.get("rige_ru") != "s":
                    errores.append("R5: COMPL.RU exige rige_ru=s")
                if p_obj != 3:
                    errores.append("R5: COMPL.RU con NP solo en X>3")
            if oi_ent and oi_ent["cat"] != "N_HUM":
                errores.append("R5/R20: OI (COMPL.RU/BEN) debe ser persona")

        # R9 y R10: Lugar
        if interr == "donde_all" and v_cat != "V_MOV":
            errores.append("R17: donde_all exige V_MOV")
        if idea.get("lugar") and interr != "donde_all":
            lug = idea["lugar"]
            caso = lug["caso"]
            if caso in ("ALL", "ABL", "UNTIL"):
                if v_cat != "V_MOV":
                    errores.append(f"R9: {caso} solo con V_MOV")
                if not lug.get("np"):
                    errores.append(f"R9: {caso} requiere NP")
                else:
                    _validar_np(lug["np"], lex_by_id, errores, "lugar")
                    l_ent = lex_by_id.get(lug["np"]["n"])
                    if l_ent:
                        if caso in ("ALL", "ABL") and l_ent["clase_sem"] not in CLASES_ALL_ABL:
                            errores.append(f"R20: {caso} incompatible con {l_ent['clase_sem']}")
                        if caso == "UNTIL":
                            if l_ent["clase_sem"] not in CLASES_UNTIL:
                                errores.append(f"R20: UNTIL incompatible con {l_ent['clase_sem']}")
                            if int(l_ent["silabas"]) != 2 or lug["np"].get("plural") or lug["np"].get("poss") or lug["np"].get("poseedor"):
                                errores.append("R10: UNTIL exige raíz de 2 sílabas sin plural ni posesivo")
            elif caso == "LOC":
                if v_ent.get("loc_clases") == "NINGUNA" or v_cat == "V_MOV":
                    errores.append("R9: verbo no admite LOC")
                if lug.get("np"):
                    _validar_np(lug["np"], lex_by_id, errores, "lugar")
                    l_ent = lex_by_id.get(lug["np"]["n"])
                    loc_c = _parse_clases(v_ent["loc_clases"]) or CLASES_ALL_ABL
                    if l_ent and l_ent["clase_sem"] not in loc_c:
                        errores.append(f"R20: LOC incompatible con {l_ent['clase_sem']}")

        # COM / CAUSE (CH-406)
        if idea.get("comcausa"):
            cc = idea["comcausa"]
            _validar_np(cc["np"], lex_by_id, errores, "comcausa")
            cc_ent = lex_by_id.get(cc["np"]["n"])
            if cc_ent:
                csem = cc_ent["clase_sem"]
                if cc["caso"] == "CAUSE":
                    if csem not in CLASES_CAUSE:
                        errores.append(f"R20: CAUSE incompatible con {csem}")
                elif cc["caso"] == "COM":
                    instr_c = _parse_clases(v_ent["instr_clases"])
                    es_instr = csem in instr_c
                    es_comp = (csem in CLASES_COM_COMP) and (v_cat in ("V_MOV", "V_INTR")) and ("persona" in suj_clases_reales)
                    if not (es_instr or es_comp):
                        errores.append(f"R20: COM '{cc_ent['es_m']}' ({csem}) incompatible con verbo '{v_ent['es_m']}'")

    else:
        # Predicados nominales: R16, R20
        pnom = idea.get("pnom") or {}
        if pred == "ser":
            if pnom.get("adj"):
                adj_ent = lex_by_id.get(pnom["adj"])
                if not adj_ent or adj_ent["cat"] != "ADJ":
                    errores.append("R20: pnom.adj inválido")
                else:
                    mod_c = _parse_clases(adj_ent["mod_clases"])
                    if suj_clases_reales and not suj_clases_reales.issubset(mod_c):
                        errores.append(f"R20: ser+adj incompatible con sujeto {suj_clases_reales}")
            elif pnom.get("np"):
                _validar_np(pnom["np"], lex_by_id, errores, "pnom_ser")
                np_ent = lex_by_id.get(pnom["np"]["n"])
                if expreso != "demostrativo":
                    if suj_clases_reales != {"persona"} or (np_ent and np_ent["cat"] != "N_HUM"):
                        errores.append("R16: ser+nombre sin demostrativo exige sujeto y nombre persona")
            else:
                errores.append("R16: ser requiere adj o np")

        elif pred == "estar":
            if suj_clases_reales & CLASES_ESTAR_SUJ_PROH:
                errores.append(f"R20: estar sujeto prohibido {suj_clases_reales}")
            if interr != "donde_estar":
                if pnom.get("np"):
                    _validar_np(pnom["np"], lex_by_id, errores, "pnom_estar")
                    l_ent = lex_by_id.get(pnom["np"]["n"])
                    if l_ent and l_ent["clase_sem"] not in CLASES_ESTAR_LUGAR:
                        errores.append(f"R20: estar lugar incompatible con {l_ent['clase_sem']}")
                elif not pnom.get("deictico"):
                    errores.append("R20: estar requiere np o deictico")

        elif pred == "tener":
            if suj_clases_reales and suj_clases_reales != {"persona"}:
                errores.append("R20: tener exige poseedor persona")
            if not pnom.get("np"):
                errores.append("R20: tener requiere np")
            else:
                _validar_np(pnom["np"], lex_by_id, errores, "pnom_tener")
                t_ent = lex_by_id.get(pnom["np"]["n"])
                if t_ent and t_ent["clase_sem"] not in CLASES_TENER_POSEIDO:
                    errores.append(f"R20: tener poseído incompatible con {t_ent['clase_sem']}")

    return errores


# ==============================================================================
# 9. GENERADOR DE IDEAS CON PESOS Y MUESTREO ESTRATIFICADO (CH-1104 a CH-1113)
# ==============================================================================

PESOS = {
    "predicado_modalidad": {
        ("verbo", "afirmativa"):     0.42,
        ("verbo", "negativa"):       0.14,
        ("verbo", "pregunta_si_no"): 0.10,
        ("verbo", "pregunta_info"):  0.09,
        ("ser", "afirmativa"):       0.06,
        ("ser", "pregunta_si_no"):   0.02,
        ("estar", "afirmativa"):     0.05,
        ("estar", "pregunta_si_no"): 0.015,
        ("estar", "pregunta_info"):  0.015,
        ("tener", "afirmativa"):     0.07,
        ("tener", "pregunta_si_no"): 0.02,
    },
    "tiempo": {
        "general":  {"SIMP": 0.40, "FUT": 0.25, "REM.C": 0.15, "REM.L": 0.20},
        "negativa": {"SIMP": 0.60, "FUT": 0.40},
    },
    "progresivo_prob": 0.15,
    "persona_sujeto": {
        "general": {1: 0.22, 2: 0.22, 3: 0.40, 4: 0.16},
        "REM.C":   {1: 0.15, 2: 0.10, 3: 0.60, 4: 0.15},
        "REM.L":   {1: 0.10, 2: 0.10, 3: 0.70, 4: 0.10},
    },
    "transicion": {"objeto_3": 0.65, "T12": 0.07, "T21": 0.07, "T31": 0.07, "T32": 0.07, "T34": 0.07},
    "sujeto_plural": 0.30,
    "expreso_1_2_4": {"pronombre": 0.50, "ninguno": 0.50},
    "expreso_3": {"nombre": 0.60, "pronombre": 0.20, "ninguno": 0.15, "coordinado": 0.05},
    "genero": {"m": 0.50, "f": 0.50},
    "cuantos_opcionales": {0: 0.30, 1: 0.45, 2: 0.20, 3: 0.05},
    "tipo_comp": {"tiempo": 0.30, "lugar": 0.35, "comcausa": 0.20, "oi": 0.15},
    "lugar_V_MOV": {"ALL": 0.55, "ABL": 0.30, "UNTIL": 0.15},
    "lugar_LOC": {"np": 0.75, "deictico": 0.25},
    "comcausa": {"COM": 0.70, "CAUSE": 0.30},
    "oi_V_DITR": {"COMPL.RU": 0.70, "BEN": 0.30},
    "oi_pronombre_expreso": 0.40,
    "od_coord": 0.05,
    "np_modificador": {"ninguno": 0.55, "adj": 0.20, "dem": 0.08, "num": 0.07, "poss": 0.07, "poseedor": 0.03},
    "plural_no_sujeto": 0.25,
    "num": {"NUM.3": 0.35, "NUM.4": 0.30, "NUM.6": 0.20, "NUM.10": 0.15},
    "dem": {"DEM.1": 0.40, "DEM.2": 0.40, "DEM.3": 0.20},
    "poss": {1: 0.30, 2: 0.25, 3: 0.30, 4: 0.15},
    "posicion_wa": {"verbo": 0.50, "ninguna": 0.20, "sujeto": 0.15, "complemento": 0.15},
    "interrogativo_verbo": {"quien_suj": 0.35, "que_od": 0.30, "donde_all": 0.20, "quien_oi": 0.15},
    "ser_pnom": {"adj": 0.60, "np": 0.40},
    "ser_suj_con_np": {"demostrativo": 0.60, "humano": 0.40},
}


class GeneradorIdeasChulupi:
    """CH-403, CH-404, CH-406, CH-1104 a CH-1113."""

    def __init__(self, lexico: List[Dict[str, Any]], semilla: int = 42):
        self.lexico = lexico
        self.lex_by_id = {e["id"]: e for e in lexico}
        self.rng = random.Random(semilla)

        self.por_cat: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
        self.nombres_por_clase: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
        self.verbos: List[Dict[str, Any]] = []
        self.adjs: List[Dict[str, Any]] = []
        self.advs_por_subcat: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

        for e in lexico:
            self.por_cat[e["cat"]].append(e)
            if e["cat"] in ("N_HUM", "N_NOHUM"):
                self.nombres_por_clase[e["clase_sem"]].append(e)
            elif e["cat"].startswith("V_"):
                self.verbos.append(e)
            elif e["cat"] == "ADJ":
                self.adjs.append(e)
            elif e["cat"] == "ADV_T":
                self.advs_por_subcat[e["subcat"]].append(e)

    def _zipf_elegir(self, candidatas: List[Dict[str, Any]]) -> Dict[str, Any]:
        """CH-403: Zipf con exponente 0.9 preservando orden de rango."""
        ordenadas = sorted(candidatas, key=lambda x: (int(x.get("rango", 9999)), x["id"]))
        pesos = [1.0 / ((k + 1) ** 0.9) for k in range(len(ordenadas))]
        return self.rng.choices(ordenadas, weights=pesos, k=1)[0]

    def _filtrar_nombres(self, clases: Set[str], solo_2_silabas: bool = False, forzar_id: Optional[str] = None) -> List[Dict[str, Any]]:
        cands: List[Dict[str, Any]] = []
        for c in sorted(clases):
            for e in self.nombres_por_clase.get(c, []):
                if solo_2_silabas and int(e["silabas"]) != 2:
                    continue
                cands.append(e)
        if forzar_id:
            m = [e for e in cands if e["id"] == forzar_id]
            if m:
                return m
        return cands

    def _crear_np(
        self,
        ent: Dict[str, Any],
        permitir_plural: bool = True,
        permitir_mod: bool = True,
        permitir_num: bool = True,
        permitir_dem: bool = True,
        permitir_poss: bool = True,
        permitir_poseedor: bool = True,
        forzar_plural: Optional[bool] = None,
        forzar_adj_id: Optional[str] = None,
    ) -> Dict[str, Any]:
        np: Dict[str, Any] = {
            "n": ent["id"],
            "genero": _elegir_ponderado(self.rng, PESOS["genero"]) if (ent["cat"] == "N_HUM" and ent["genero"] == "var") else None,
            "plural": False,
            "dem": None,
            "num": None,
            "adj": None,
            "poss": None,
            "poseedor": None,
        }

        if forzar_adj_id:
            adj_ent = self.lex_by_id[forzar_adj_id]
            if ent["clase_sem"] in _parse_clases(adj_ent["mod_clases"]):
                np["adj"] = forzar_adj_id
                permitir_mod = False

        if permitir_mod:
            mods_validos = dict(PESOS["np_modificador"])
            if not permitir_dem:
                mods_validos.pop("dem", None)
            if not permitir_num or ent.get("contable") != "s":
                mods_validos.pop("num", None)
            if not permitir_poss or ent["clase_sem"] == "tiempo":
                mods_validos.pop("poss", None)
            if not permitir_poseedor or ent["clase_sem"] == "tiempo":
                mods_validos.pop("poseedor", None)
            adjs_compat = [a for a in self.adjs if ent["clase_sem"] in _parse_clases(a["mod_clases"])]
            if not adjs_compat:
                mods_validos.pop("adj", None)

            mod = _elegir_ponderado(self.rng, mods_validos)
            if mod == "adj":
                np["adj"] = self._zipf_elegir(adjs_compat)["id"]
            elif mod == "dem":
                np["dem"] = _elegir_ponderado(self.rng, PESOS["dem"])
            elif mod == "num":
                np["num"] = _elegir_ponderado(self.rng, PESOS["num"])
            elif mod == "poss":
                np["poss"] = _elegir_ponderado(self.rng, PESOS["poss"])
            elif mod == "poseedor":
                pos_ent = self._zipf_elegir(self.por_cat["N_HUM"])
                np["poseedor"] = {
                    "n": pos_ent["id"],
                    "genero": _elegir_ponderado(self.rng, PESOS["genero"]) if pos_ent["genero"] == "var" else None,
                    "plural": False,
                }

        if forzar_plural is not None:
            if forzar_plural and not np["num"]:
                np["plural"] = True
        elif permitir_plural and ent.get("contable") == "s" and not np["num"]:
            if self.rng.random() < PESOS["plural_no_sujeto"]:
                np["plural"] = True

        return np

    def calcular_cuotas_estratos(self, n_total: int) -> List[Tuple[Tuple[str, str, str, int], int]]:
        """CH-1104: Calcula cuotas por estrato (predicado, modalidad, tiempo, persona)."""
        pesos_estratos: Dict[Tuple[str, str, str, int], float] = {}
        for (pred, mod), p_pm in PESOS["predicado_modalidad"].items():
            if pred == "verbo":
                t_dist = PESOS["tiempo"]["negativa"] if mod == "negativa" else PESOS["tiempo"]["general"]
            else:
                t_dist = {"SIMP": 1.0}
            for t, p_t in t_dist.items():
                if pred == "verbo" and t in ("REM.C", "REM.L"):
                    p_dist = PESOS["persona_sujeto"][t]
                else:
                    p_dist = PESOS["persona_sujeto"]["general"]
                for pers, p_pers in p_dist.items():
                    pesos_estratos[(pred, mod, t, pers)] = p_pm * p_t * p_pers

        suma = sum(pesos_estratos.values())
        cuotas_raw = {k: (v / suma) * n_total for k, v in pesos_estratos.items()}
        cuotas = {k: int(math.floor(v)) for k, v in cuotas_raw.items()}
        faltan = n_total - sum(cuotas.values())
        # Repartir restos mayores
        restos = sorted(cuotas_raw.keys(), key=lambda k: cuotas_raw[k] - cuotas[k], reverse=True)
        for k in restos[:faltan]:
            cuotas[k] += 1
        return [(k, c) for k, c in cuotas.items() if c > 0]

    def generar_idea_en_estrato(
        self,
        pred: str,
        mod: str,
        tiempo: str,
        persona: int,
        forzar_lema_id: Optional[str] = None,
    ) -> Optional[Dict[str, Any]]:
        """Genera una única idea válida dentro del estrato dado (o None si falla el intento)."""
        lema_forzado = self.lex_by_id[forzar_lema_id] if forzar_lema_id else None

        # ------------------------------------------------------------------
        # Caso 1: Predicados nominales (ser, estar, tener)
        # ------------------------------------------------------------------
        if pred != "verbo":
            pl_suj = self.rng.random() < PESOS["sujeto_plural"]
            interr = "donde_estar" if (pred == "estar" and mod == "pregunta_info") else None

            if pred == "ser":
                if lema_forzado and lema_forzado["cat"] == "ADJ":
                    tipo_pnom = "adj"
                elif lema_forzado and lema_forzado["cat"] in ("N_HUM", "N_NOHUM"):
                    tipo_pnom = "np"
                else:
                    tipo_pnom = _elegir_ponderado(self.rng, PESOS["ser_pnom"])

                if tipo_pnom == "adj":
                    adj_ent = lema_forzado if (lema_forzado and lema_forzado["cat"] == "ADJ") else self._zipf_elegir(self.adjs)
                    mod_c = _parse_clases(adj_ent["mod_clases"])
                    if persona != 3 and "persona" not in mod_c:
                        return None
                    suj = self._construir_sujeto(
                        persona, pl_suj, mod, pred, mod_c,
                        exigir_expreso=pl_suj, permitir_coord=False, permitir_num=False,
                    )
                    if suj is None:
                        return None
                    pnom = {"adj": adj_ent["id"]}
                else:
                    # ser + np
                    if persona == 3:
                        if lema_forzado and lema_forzado["cat"] == "N_NOHUM":
                            sub_suj = "demostrativo"
                        else:
                            sub_suj = _elegir_ponderado(self.rng, PESOS["ser_suj_con_np"])
                        if sub_suj == "demostrativo":
                            if lema_forzado and lema_forzado["cat"] in ("N_HUM", "N_NOHUM"):
                                pred_ent = lema_forzado
                            else:
                                todos_n = self.por_cat["N_HUM"] + self.por_cat["N_NOHUM"]
                                pred_ent = self._zipf_elegir(todos_n)
                            suj = {
                                "persona": 3, "plural": pl_suj, "expreso": "demostrativo",
                                "dem": _elegir_ponderado(self.rng, PESOS["dem"]),
                                "np": None, "coord": None, "genero": None,
                            }
                            pnom = {"np": self._crear_np(pred_ent, permitir_plural=False, permitir_mod=False)}
                        else:
                            pred_ent = lema_forzado if (lema_forzado and lema_forzado["cat"] == "N_HUM") else self._zipf_elegir(self.por_cat["N_HUM"])
                            suj = self._construir_sujeto(
                                3, pl_suj, mod, pred, {"persona"},
                                exigir_expreso=True, permitir_coord=False, permitir_num=False,
                            )
                            if suj is None:
                                return None
                            p_np = self._crear_np(pred_ent, permitir_plural=False, permitir_mod=False)
                            if pred_ent["genero"] == "var":
                                g_suj = suj.get("genero") or (suj["np"].get("genero") if suj.get("np") else None) or "m"
                                p_np["genero"] = g_suj
                            pnom = {"np": p_np}
                    else:
                        pred_ent = lema_forzado if (lema_forzado and lema_forzado["cat"] == "N_HUM") else self._zipf_elegir(self.por_cat["N_HUM"])
                        suj = self._construir_sujeto(
                            persona, pl_suj, mod, pred, {"persona"},
                            exigir_expreso=pl_suj, permitir_coord=False, permitir_num=False,
                        )
                        if suj is None:
                            return None
                        pnom = {"np": self._crear_np(pred_ent, permitir_plural=False, permitir_mod=False)}

            elif pred == "estar":
                clases_suj = set(self.nombres_por_clase.keys()) - CLASES_ESTAR_SUJ_PROH
                suj = self._construir_sujeto(
                    persona, pl_suj, mod, pred, clases_suj,
                    exigir_expreso=pl_suj, permitir_coord=False, permitir_num=False,
                    forzar_n_id=forzar_lema_id if (lema_forzado and lema_forzado["clase_sem"] in clases_suj and lema_forzado["clase_sem"] not in CLASES_ESTAR_LUGAR) else None,
                )
                if suj is None:
                    return None
                if interr == "donde_estar":
                    pnom = {}
                else:
                    if lema_forzado and lema_forzado["clase_sem"] in CLASES_ESTAR_LUGAR:
                        l_ent = lema_forzado
                        pnom = {"np": self._crear_np(l_ent, permitir_dem=False, permitir_num=False, permitir_poseedor=False)}
                    elif self.rng.random() < PESOS["lugar_LOC"]["deictico"]:
                        pnom = {"deictico": _elegir_ponderado(self.rng, PESOS["dem"])}
                    else:
                        cands = self._filtrar_nombres(CLASES_ESTAR_LUGAR)
                        l_ent = self._zipf_elegir(cands)
                        pnom = {"np": self._crear_np(l_ent, permitir_dem=False, permitir_num=False, permitir_poseedor=False)}

            else:  # tener
                suj = self._construir_sujeto(
                    persona, pl_suj, mod, pred, {"persona"},
                    exigir_expreso=pl_suj, permitir_coord=False, permitir_num=False,
                    forzar_n_id=forzar_lema_id if (lema_forzado and lema_forzado["cat"] == "N_HUM") else None,
                )
                if suj is None:
                    return None
                cands_t = self._filtrar_nombres(CLASES_TENER_POSEIDO, forzar_id=forzar_lema_id)
                t_ent = self._zipf_elegir(cands_t)
                # CH-908/CH-601: tener sin poss ni poseedor ni dem
                t_np = self._crear_np(
                    t_ent, permitir_dem=False, permitir_poss=False, permitir_poseedor=False,
                    forzar_adj_id=forzar_lema_id if (lema_forzado and lema_forzado["cat"] == "ADJ") else None,
                )
                pnom = {"np": t_np}

            idea = {
                "predicado": pred, "modalidad": mod, "tiempo": "SIMP", "aspecto": "simple",
                "verbo": None, "sujeto": suj, "objeto_persona": 3, "oi_pronombre_expreso": False,
                "od": None, "od_coord": None, "oi": None, "lugar": None, "comcausa": None,
                "adv_t": None, "interrogativo": interr, "pnom": pnom,
                "posicion_wa": "verbo",
            }
            return idea if not validar_idea(idea, self.lex_by_id) else None

        # ------------------------------------------------------------------
        # Caso 2: Oraciones verbales
        # ------------------------------------------------------------------
        interr = None
        if mod == "pregunta_info":
            interr_dist = dict(PESOS["interrogativo_verbo"])
            if persona != 3:
                interr_dist.pop("quien_suj", None)
            interr = _elegir_ponderado(self.rng, interr_dist)

        # Seleccionar verbo compatible con interrogativo y persona
        verbos_cands = list(self.verbos)
        if lema_forzado and lema_forzado["cat"].startswith("V_"):
            verbos_cands = [lema_forzado]
        if interr == "quien_suj":
            verbos_cands = [v for v in verbos_cands if "persona" in _parse_clases(v["suj_clases"])]
        elif interr == "que_od":
            verbos_cands = [v for v in verbos_cands if v["cat"] in ("V_TR", "V_DITR")]
        elif interr == "donde_all":
            verbos_cands = [v for v in verbos_cands if v["cat"] == "V_MOV"]
        elif interr == "quien_oi":
            verbos_cands = [v for v in verbos_cands if v["cat"] == "V_DITR"]
        if persona in (1, 2, 4):
            verbos_cands = [v for v in verbos_cands if "persona" in _parse_clases(v["suj_clases"])]

        # Si hay lema forzado no verbal, intentar elegir un verbo compatible con él
        if lema_forzado and lema_forzado["cat"] == "N_NOHUM":
            csem = lema_forzado["clase_sem"]
            compat_v = [
                v for v in verbos_cands
                if csem in _parse_clases(v["obj_clases"])
                or csem in _parse_clases(v["suj_clases"])
                or csem in _parse_clases(v["instr_clases"])
                or (v["cat"] == "V_MOV" and csem in CLASES_UNTIL)
                or (v["cat"] != "V_MOV" and v.get("loc_clases") != "NINGUNA" and csem in (_parse_clases(v["loc_clases"]) or CLASES_ALL_ABL))
                or (csem in CLASES_CAUSE)
            ]
            if compat_v:
                verbos_cands = compat_v

        if not verbos_cands:
            return None
        v_ent = self._zipf_elegir(verbos_cands)
        v_cat = v_ent["cat"]

        # Transición (CH-1109)
        p_obj = 3
        if tiempo == "SIMP" and v_ent.get("obj_persona") == "s" and interr not in ("quien_oi", "que_od"):
            trans_posibles = {"objeto_3": PESOS["transicion"]["objeto_3"]}
            for tname, (ps, po) in [("T12", (1, 2)), ("T21", (2, 1)), ("T31", (3, 1)), ("T32", (3, 2)), ("T34", (3, 4))]:
                if ps == persona:
                    trans_posibles[tname] = PESOS["transicion"][tname]
            t_sel = _elegir_ponderado(self.rng, trans_posibles)
            if t_sel != "objeto_3":
                p_obj = int(t_sel[2])

        # Aspecto y plural sujeto (CH-603, CH-1107)
        aspecto = "simple"
        pl_suj = False
        if interr == "quien_suj":
            pl_suj = False
        else:
            if tiempo == "SIMP" and mod != "negativa" and self.rng.random() < PESOS["progresivo_prob"]:
                aspecto = "progresivo"
                pl_suj = False  # CH-603
            else:
                pl_suj = self.rng.random() < PESOS["sujeto_plural"]

        # Sujeto
        v_suj_c = _parse_clases(v_ent["suj_clases"])
        if interr == "quien_suj":
            suj = {"persona": 3, "plural": False, "expreso": "ninguno", "np": None, "coord": None, "dem": None, "genero": None}
            suj_es_persona = True
        else:
            exigir_exp = (mod == "negativa" and pl_suj)
            permitir_coord = (mod not in ("negativa", "pregunta_info")) and pl_suj
            permitir_num = not (mod == "negativa" and pl_suj)
            suj = self._construir_sujeto(
                persona, pl_suj, mod, pred, v_suj_c,
                exigir_expreso=exigir_exp, permitir_coord=permitir_coord, permitir_num=permitir_num,
                forzar_n_id=forzar_lema_id if (lema_forzado and lema_forzado["clase_sem"] in v_suj_c) else None,
            )
            if suj is None:
                return None
            if suj["expreso"] == "nombre":
                suj_es_persona = self.lex_by_id[suj["np"]["n"]]["clase_sem"] == "persona"
            elif suj["expreso"] == "coordinado":
                suj_es_persona = all(self.lex_by_id[c["n"]]["clase_sem"] == "persona" for c in suj["coord"])
            else:
                suj_es_persona = True

        # OD (obligatorio en V_TR/V_DITR salvo V_TR con obj 1/2/4 o que_od)
        od = None
        od_coord = None
        v_obj_c = _parse_clases(v_ent["obj_clases"])
        necesita_od = (v_cat == "V_DITR" or (v_cat == "V_TR" and p_obj == 3)) and interr != "que_od"
        if necesita_od:
            cands_od = self._filtrar_nombres(v_obj_c, forzar_id=forzar_lema_id)
            if not cands_od:
                return None
            if mod not in ("negativa", "pregunta_info") and len(cands_od) >= 2 and self.rng.random() < PESOS["od_coord"]:
                o1 = self._zipf_elegir(cands_od)
                o2_cands = [x for x in self._filtrar_nombres(v_obj_c) if x["id"] != o1["id"]]
                if o2_cands:
                    o2 = self._zipf_elegir(o2_cands)
                    od_coord = [
                        self._crear_np(o1, permitir_plural=False, permitir_dem=False, permitir_num=False, permitir_poss=False, permitir_poseedor=False),
                        self._crear_np(o2, permitir_plural=False, permitir_dem=False, permitir_num=False, permitir_poss=False, permitir_poseedor=False),
                    ]
            if od_coord is None:
                od_ent = self._zipf_elegir(cands_od)
                od = self._crear_np(
                    od_ent,
                    forzar_adj_id=forzar_lema_id if (lema_forzado and lema_forzado["cat"] == "ADJ") else None,
                )

        # Complementos opcionales (CH-1111)
        cuantos = _elegir_ponderado(self.rng, PESOS["cuantos_opcionales"])
        tipos_elegidos: List[str] = []
        pool_comp = dict(PESOS["tipo_comp"])
        if interr == "donde_all" or (v_cat != "V_MOV" and v_ent.get("loc_clases") == "NINGUNA"):
            pool_comp.pop("lugar", None)
        if interr == "quien_oi":
            pool_comp.pop("oi", None)

        # Si el lema forzado es un ADV_T, asegurarse de incluir tiempo si es compatible
        if lema_forzado and lema_forzado["cat"] == "ADV_T":
            sc = lema_forzado["subcat"]
            compat_t = (sc == "pasado" and tiempo != "FUT" and aspecto != "progresivo") or (sc == "presente" and tiempo == "SIMP") or (sc == "futuro" and tiempo == "FUT")
            if compat_t and "tiempo" in pool_comp:
                tipos_elegidos.append("tiempo")
                pool_comp.pop("tiempo")
                cuantos = max(0, cuantos - 1)

        # Si el lema forzado solo puede entrar como lugar/UNTIL/CAUSE/COM
        if lema_forzado and lema_forzado["cat"] == "N_NOHUM":
            csem = lema_forzado["clase_sem"]
            if csem == "tiempo" and v_cat == "V_MOV" and "lugar" in pool_comp:
                tipos_elegidos.append("lugar")
                pool_comp.pop("lugar")
                cuantos = max(0, cuantos - 1)
            elif csem == "abstracto" and "comcausa" in pool_comp:
                tipos_elegidos.append("comcausa")
                pool_comp.pop("comcausa")
                cuantos = max(0, cuantos - 1)

        for _ in range(cuantos):
            if not pool_comp:
                break
            t_c = _elegir_ponderado(self.rng, pool_comp)
            tipos_elegidos.append(t_c)
            pool_comp.pop(t_c)

        adv_t = None
        if "tiempo" in tipos_elegidos:
            subcats = []
            if tiempo == "SIMP":
                subcats.append("presente")
                if aspecto != "progresivo":
                    subcats.append("pasado")
            elif tiempo in ("REM.C", "REM.L"):
                subcats.append("pasado")
            elif tiempo == "FUT":
                subcats.append("futuro")
            adv_cands = [a for sc in subcats for a in self.advs_por_subcat.get(sc, [])]
            if lema_forzado and lema_forzado in adv_cands:
                adv_t = lema_forzado["id"]
            elif adv_cands:
                adv_t = self._zipf_elegir(adv_cands)["id"]

        lugar = {"caso": "ALL"} if interr == "donde_all" else None
        if "lugar" in tipos_elegidos and lugar is None:
            if v_cat == "V_MOV":
                caso_l = _elegir_ponderado(self.rng, PESOS["lugar_V_MOV"])
                if lema_forzado and lema_forzado["clase_sem"] == "tiempo" and int(lema_forzado["silabas"]) == 2:
                    caso_l = "UNTIL"
                if caso_l == "UNTIL":
                    cands_l = self._filtrar_nombres(CLASES_UNTIL, solo_2_silabas=True, forzar_id=forzar_lema_id)
                    if cands_l:
                        l_ent = self._zipf_elegir(cands_l)
                        lugar = {"caso": "UNTIL", "np": self._crear_np(l_ent, permitir_plural=False, permitir_poss=False, permitir_poseedor=False, permitir_num=False), "deictico": None}
                else:
                    cands_l = self._filtrar_nombres(CLASES_ALL_ABL, forzar_id=forzar_lema_id)
                    if cands_l:
                        l_ent = self._zipf_elegir(cands_l)
                        lugar = {"caso": caso_l, "np": self._crear_np(l_ent), "deictico": None}
            else:
                if self.rng.random() < PESOS["lugar_LOC"]["deictico"]:
                    lugar = {"caso": "LOC", "np": None, "deictico": _elegir_ponderado(self.rng, PESOS["dem"])}
                else:
                    loc_c = _parse_clases(v_ent["loc_clases"]) or CLASES_ALL_ABL
                    cands_l = self._filtrar_nombres(loc_c, forzar_id=forzar_lema_id)
                    if cands_l:
                        l_ent = self._zipf_elegir(cands_l)
                        lugar = {"caso": "LOC", "np": self._crear_np(l_ent), "deictico": None}

        comcausa = None
        if "comcausa" in tipos_elegidos:
            tiene_coord = (suj["expreso"] == "coordinado") or (od_coord is not None)
            cc_opts = dict(PESOS["comcausa"])
            if tiene_coord:
                cc_opts.pop("COM", None)
            caso_cc = _elegir_ponderado(self.rng, cc_opts)
            if lema_forzado and lema_forzado["clase_sem"] == "abstracto":
                caso_cc = "CAUSE"
            if caso_cc == "COM":
                clases_com: Set[str] = set(_parse_clases(v_ent["instr_clases"]))
                if v_cat in ("V_MOV", "V_INTR") and suj_es_persona:
                    clases_com |= CLASES_COM_COMP
                cands_cc = self._filtrar_nombres(clases_com, forzar_id=forzar_lema_id)
                if cands_cc:
                    cc_ent = self._zipf_elegir(cands_cc)
                    comcausa = {"caso": "COM", "np": self._crear_np(cc_ent)}
            else:
                cands_cc = self._filtrar_nombres(CLASES_CAUSE, forzar_id=forzar_lema_id)
                if cands_cc:
                    cc_ent = self._zipf_elegir(cands_cc)
                    comcausa = {"caso": "CAUSE", "np": self._crear_np(cc_ent)}

        oi = {"funcion": "COMPL.RU"} if interr == "quien_oi" else None
        oi_pron_exp = False
        if v_cat == "V_DITR" and p_obj in (1, 2, 4):
            if self.rng.random() < PESOS["oi_pronombre_expreso"]:
                oi_pron_exp = True
                oi = {"funcion": "COMPL.RU"}
        elif "oi" in tipos_elegidos and oi is None:
            if v_cat == "V_DITR" and p_obj == 3:
                func_oi = _elegir_ponderado(self.rng, PESOS["oi_V_DITR"])
            else:
                func_oi = "BEN"
            cands_oi = self._filtrar_nombres({"persona"}, forzar_id=forzar_lema_id)
            if cands_oi:
                oi_ent = self._zipf_elegir(cands_oi)
                oi = {"funcion": func_oi, "np": self._crear_np(oi_ent)}

        pos_wa = _elegir_ponderado(self.rng, PESOS["posicion_wa"]) if mod == "afirmativa" else "verbo"

        idea = {
            "predicado": "verbo", "modalidad": mod, "tiempo": tiempo, "aspecto": aspecto,
            "verbo": v_ent["id"], "sujeto": suj, "objeto_persona": p_obj,
            "oi_pronombre_expreso": oi_pron_exp, "od": od, "od_coord": od_coord,
            "oi": oi, "lugar": lugar, "comcausa": comcausa, "adv_t": adv_t,
            "interrogativo": interr, "pnom": None, "posicion_wa": pos_wa,
        }
        return idea if not validar_idea(idea, self.lex_by_id) else None

    def _construir_sujeto(
        self,
        persona: int,
        pl_suj: bool,
        mod: str,
        pred: str,
        clases_aceptadas: Set[str],
        exigir_expreso: bool = False,
        permitir_coord: bool = True,
        permitir_num: bool = True,
        forzar_n_id: Optional[str] = None,
    ) -> Optional[Dict[str, Any]]:
        if persona in (1, 2, 4):
            if "persona" not in clases_aceptadas:
                return None
            dist = dict(PESOS["expreso_1_2_4"])
            if exigir_expreso:
                dist.pop("ninguno", None)
            exp = _elegir_ponderado(self.rng, dist)
            return {"persona": persona, "plural": pl_suj, "expreso": exp, "np": None, "coord": None, "dem": None, "genero": None}

        # persona == 3
        dist = dict(PESOS["expreso_3"])
        if exigir_expreso:
            dist.pop("ninguno", None)
        if not permitir_coord or not pl_suj:
            dist.pop("coordinado", None)
        if "persona" not in clases_aceptadas:
            dist.pop("pronombre", None)
            dist.pop("ninguno", None)
        cands_n = self._filtrar_nombres(clases_aceptadas, forzar_id=forzar_n_id)
        if not cands_n:
            dist.pop("nombre", None)
            dist.pop("coordinado", None)
        if len(self._filtrar_nombres(clases_aceptadas)) < 2:
            dist.pop("coordinado", None)
        if not dist:
            return None
        if forzar_n_id and any(e["id"] == forzar_n_id for e in cands_n) and "nombre" in dist:
            exp = "nombre"
        else:
            exp = _elegir_ponderado(self.rng, dist)

        if exp == "ninguno":
            return {"persona": 3, "plural": pl_suj, "expreso": "ninguno", "np": None, "coord": None, "dem": None, "genero": _elegir_ponderado(self.rng, PESOS["genero"])}
        if exp == "pronombre":
            return {"persona": 3, "plural": pl_suj, "expreso": "pronombre", "np": None, "coord": None, "dem": None, "genero": _elegir_ponderado(self.rng, PESOS["genero"])}
        if exp == "nombre":
            ent = self._zipf_elegir(cands_n)
            np = self._crear_np(
                ent,
                permitir_plural=False,
                permitir_num=(permitir_num and pl_suj),
                forzar_plural=pl_suj,
            )
            g = np["genero"] if ent["genero"] == "var" else (ent["genero"] if ent["genero"] in ("m", "f") else "m")
            return {"persona": 3, "plural": pl_suj, "expreso": "nombre", "np": np, "coord": None, "dem": None, "genero": g}
        # coordinado
        todos = self._filtrar_nombres(clases_aceptadas)
        e1 = self._zipf_elegir(cands_n)
        otros = [x for x in todos if x["id"] != e1["id"]]
        e2 = self._zipf_elegir(otros)
        c1 = self._crear_np(e1, permitir_plural=False, permitir_dem=False, permitir_num=False, permitir_poss=False, permitir_poseedor=False)
        c2 = self._crear_np(e2, permitir_plural=False, permitir_dem=False, permitir_num=False, permitir_poss=False, permitir_poseedor=False)
        return {"persona": 3, "plural": True, "expreso": "coordinado", "np": None, "coord": [c1, c2], "dem": None, "genero": "m"}


def _hash_idea(idea: Dict[str, Any]) -> str:
    s = json.dumps(idea, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def _lemas_usados_en_idea(idea: Dict[str, Any]) -> List[str]:
    ids: List[str] = []
    def add_np(np):
        if not np:
            return
        if np.get("n"):
            ids.append(np["n"])
        if np.get("adj"):
            ids.append(np["adj"])
        if np.get("poseedor") and np["poseedor"].get("n"):
            ids.append(np["poseedor"]["n"])

    if idea.get("verbo"):
        ids.append(idea["verbo"])
    if idea.get("adv_t"):
        ids.append(idea["adv_t"])
    suj = idea.get("sujeto") or {}
    add_np(suj.get("np"))
    for c in (suj.get("coord") or []):
        add_np(c)
    add_np(idea.get("od"))
    for c in (idea.get("od_coord") or []):
        add_np(c)
    if idea.get("oi"):
        add_np(idea["oi"].get("np"))
    if idea.get("lugar"):
        add_np(idea["lugar"].get("np"))
    if idea.get("comcausa"):
        add_np(idea["comcausa"].get("np"))
    if idea.get("pnom"):
        if idea["pnom"].get("adj"):
            ids.append(idea["pnom"]["adj"])
        add_np(idea["pnom"].get("np"))
    return ids


# ==============================================================================
# 10. VERIFICADOR AUTOMÁTICO V01-V31 (CH-1301, CH-1302)
# ==============================================================================

PALABRAS_PROHIBIDAS_B = {"maya", "paya", "khaya", "mä", "pä", "khä"}
PALABRAS_GENERO_ES = {
    "el", "la", "los", "las", "un", "una", "al", "del",
    "él", "ella", "ellos", "ellas",
    "este", "esta", "estos", "estas",
    "ese", "esa", "esos", "esas",
    "aquel", "aquella", "aquellos", "aquellas",
    "nuestro", "nuestra", "nuestros", "nuestras",
}


def _normalizar_genero_es(oracion_es: str, lex_by_id: Dict[str, Dict[str, Any]]) -> str:
    """Reemplaza variantes f→m conocidas para comprobar V26 (solo difieren en género)."""
    mapa_f_m: Dict[str, str] = {
        "la": "el", "las": "los", "una": "un",
        "ella": "él", "ellas": "ellos",
        "esta": "este", "estas": "estos",
        "esa": "ese", "esas": "esos",
        "aquella": "aquel", "aquellas": "aquellos",
        "nuestra": "nuestro", "nuestras": "nuestros",
    }
    for e in lex_by_id.values():
        if e.get("es_f") and e.get("es_m"):
            mapa_f_m[e["es_f"].lower()] = e["es_m"].lower()
        if e.get("es_pl_f") and e.get("es_pl_m"):
            mapa_f_m[e["es_pl_f"].lower()] = e["es_pl_m"].lower()

    sin_punt = re.sub(r"[¿?.]", "", oracion_es).lower()
    toks = [mapa_f_m.get(t, t) for t in sin_punt.split()]
    return " ".join(toks)


def verificar_dataset(
    pares: List[Dict[str, Any]],
    lexico: List[Dict[str, Any]],
    palabras_reales: List[str],
    modo_filtro: str = "guia",
    cobertura_minima: int = 20,
) -> Dict[str, Any]:
    """CH-1302: Ejecuta las verificaciones V01-V31 sobre el dataset generado."""
    lex_by_id = {e["id"]: e for e in lexico}
    raices_lex = {e["raiz"] for e in lexico}
    gram_b_set = {p["b"] for p in PALABRAS_GRAM_LISTA}

    errores: List[str] = []
    avisos: List[str] = []
    campos_req = {"id", "version_spec", "version_lexico", "idea", "b", "a", "es", "palabras_b", "dv"}

    hashes_vistos: Set[str] = set()
    grupos_b: Dict[str, Set[str]] = defaultdict(set)
    grupos_a: Dict[str, Set[str]] = defaultdict(set)
    conteo_lemas: Counter = Counter()
    conteo_dv: Counter = Counter()
    pares_con_dv = 0
    conteo_pm: Counter = Counter()

    for idx, par in enumerate(pares):
        pid = par.get("id", f"idx_{idx}")
        # V01: campos requeridos
        if not campos_req.issubset(par.keys()):
            errores.append(f"V01 {pid}: faltan campos {campos_req - set(par.keys())}")
            continue
        # V02: NFC, apóstrofo U+0027, sin dobles espacios
        for k in ("b", "a", "es"):
            txt = par[k]
            if txt != normalizar(txt) or "  " in txt:
                errores.append(f"V02 {pid} [{k}]: normalización/espacios")
        # V03: mayúscula inicial y puntuación
        es_preg = par["idea"]["modalidad"] in ("pregunta_si_no", "pregunta_info")
        for k in ("b", "a", "es"):
            txt = par[k]
            if es_preg:
                if not (txt.startswith("¿") and txt.endswith("?") and txt[1].isupper()):
                    errores.append(f"V03 {pid} [{k}]: puntuación pregunta '{txt}'")
            else:
                if not (txt[0].isupper() and txt.endswith(".") and not txt.startswith("¿")):
                    errores.append(f"V03 {pid} [{k}]: puntuación declarativa '{txt}'")

        # V04, V05, V07: ortografía, fonotáctica y lista negra en B y A
        toks_b = re.sub(r"[¿?.]", "", par["b"]).lower().split()
        toks_a = re.sub(r"[¿?.]", "", par["a"]).lower().split()
        for w in toks_b + toks_a:
            if not validar_palabra(w):
                errores.append(f"V04/V05 {pid}: palabra inválida '{w}'")
        for w in toks_b:
            if "ŋ" in w or w in PALABRAS_PROHIBIDAS_B or w.endswith("tayna"):
                errores.append(f"V07 {pid}: forma prohibida '{w}'")

        # V06, V08, V09, V11, V12: reconstrucción y slots
        pb = par["palabras_b"]
        if len(pb) != len(toks_b) or len(pb) != len(toks_a):
            errores.append(f"V11 {pid}: desajuste número de palabras")
        else:
            for k_w, (base_b, ids) in enumerate(pb):
                if base_b not in raices_lex and base_b not in gram_b_set:
                    errores.append(f"V06 {pid}: base desconocida '{base_b}'")
                es_v = (k_w == len(pb) - 1 and par["idea"]["predicado"] == "verbo")
                if not validar_slots_palabra(es_v, ids):
                    errores.append(f"V08 {pid}: slots inválidos {base_b} {ids}")
                rb = construir_palabra(base_b, ids, "B")
                ra = construir_palabra(base_b, ids, "A")
                if rb != toks_b[k_w]:
                    errores.append(f"V09 {pid}: B '{rb}' != '{toks_b[k_w]}'")
                if ra != toks_a[k_w]:
                    errores.append(f"V12 {pid}: A '{ra}' != '{toks_a[k_w]}'")

        # V15, V16, V17: oracionales AFF, TOP, NEG, Q
        aff_count = sum(1 for _, ids in pb if "AFF" in ids)
        if aff_count > 1:
            errores.append(f"V15 {pid}: más de un AFF ({aff_count})")
        if es_preg and aff_count > 0:
            errores.append(f"V15 {pid}: pregunta con AFF")
        if par["idea"]["modalidad"] == "negativa":
            has_jani = any(b == "jani" and "AFF" in ids for b, ids in pb)
            has_neg = any("NEG" in ids and "NEG.TI" in ids for _, ids in pb)
            if not (has_jani and has_neg):
                errores.append(f"V17 {pid}: negativa mal formada")

        # V19, V20: idea válida y coherente
        err_i = validar_idea(par["idea"], lex_by_id)
        if err_i:
            errores.append(f"V19/V20 {pid}: {err_i[0]}")

        # V22: español reconstruido
        es_rec = construir_espanol(par["idea"], lex_by_id)
        if es_rec != par["es"]:
            errores.append(f"V22 {pid}: ES '{es_rec}' != '{par['es']}'")

        # Acumular para conjunto completo (V26-V30)
        h = _hash_idea(par["idea"])
        if h in hashes_vistos:
            errores.append(f"V27 {pid}: idea duplicada")
        hashes_vistos.add(h)

        es_norm_g = _normalizar_genero_es(par["es"], lex_by_id)
        grupos_b[par["b"]].add(es_norm_g)
        grupos_a[par["a"]].add(es_norm_g)

        for lid in _lemas_usados_en_idea(par["idea"]):
            conteo_lemas[lid] += 1

        if par["dv"]:
            pares_con_dv += 1
            for dv in par["dv"]:
                conteo_dv[dv] += 1

        conteo_pm[(par["idea"]["predicado"], par["idea"]["modalidad"])] += 1

    # V26: una traducción por oración ficticia (salvo género)
    for modo_g, grupos in [("B", grupos_b), ("A", grupos_a)]:
        for orac, trads in grupos.items():
            if len(trads) > 1:
                errores.append(f"V26 ({modo_g}): '{orac}' tiene traducciones distintas más allá del género: {trads}")

    # V28: distribución marginal dentro de ±10 % relativo de pesos.yaml (predicado.modalidad)
    n_tot = max(1, len(pares))
    for (pred_k, mod_k), peso_esp in PESOS["predicado_modalidad"].items():
        obs = conteo_pm[(pred_k, mod_k)] / n_tot
        if peso_esp > 0 and abs(obs - peso_esp) / peso_esp > 0.15:
            avisos.append(f"V28: {pred_k}.{mod_k} observado {obs:.3f} vs esperado {peso_esp:.3f}")

    # V29: cobertura mínima del léxico
    bajo_min = [e["id"] for e in lexico if conteo_lemas[e["id"]] < cobertura_minima]
    if bajo_min:
        avisos.append(f"V29: {len(bajo_min)} entradas por debajo de {cobertura_minima} apariciones (mín={min(conteo_lemas[e['id']] for e in lexico)})")

    # V30: tope de [DERIVADA]
    prop_dv = pares_con_dv / n_tot
    if prop_dv > 0.50:
        avisos.append(f"V30: proporción con alguna [DERIVADA] = {prop_dv:.1%} (> 50%)")
    for dv, c in sorted(conteo_dv.items()):
        if c / n_tot > 0.15:
            avisos.append(f"V30: {dv} aparece en {c/n_tot:.1%} (> 15%)")

    # V31: ninguna raíz inventada choca con palabras_reales
    trie = TriePalabrasReales(palabras_reales)
    umbral = 1 if modo_filtro == "guia" else 5
    for e in lexico:
        es_p, n_p = trie.consultar(e["raiz"])
        if es_p or n_p >= umbral:
            errores.append(f"V31: raíz '{e['raiz']}' ({e['id']}) choca con palabras_reales (es_p={es_p}, n_p={n_p})")

    return {
        "total_pares": len(pares),
        "errores_count": len(errores),
        "errores_muestra": errores[:20],
        "avisos": avisos,
        "prop_con_dv": round(prop_dv, 4),
        "dv_detalle": {dv: round(c / n_tot, 4) for dv, c in sorted(conteo_dv.items())},
        "cobertura_min_observada": min((conteo_lemas[e["id"]] for e in lexico), default=0),
        "entradas_lexico_usadas": sum(1 for e in lexico if conteo_lemas[e["id"]] > 0),
        "total_entradas_lexico": len(lexico),
        "distribucion_predicado_modalidad": {f"{k[0]}.{k[1]}": round(v / n_tot, 4) for k, v in sorted(conteo_pm.items())},
    }


# ==============================================================================
# 11. FUNCIÓN PRINCIPAL: generar_dataset_chulupi(n=1000, ...)
# ==============================================================================

def generar_dataset_chulupi(
    n: int = 1000,
    lexico: Optional[List[Dict[str, Any]]] = None,
    palabras_reales: Optional[List[str]] = None,
    modo_filtro: str = "guia",
    semilla: int = 42,
    cobertura_minima: int = 5,
    exacto_n: bool = True,
    verificar: bool = True,
    ruta_jsonl: Optional[str] = None,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """
    Genera el dataset sintético Chulupi Lang v1.1 (Modos B, A y Español).
    - n: número objetivo de pares (por defecto 1.000).
    - cobertura_minima: mínimo de apariciones por entrada del léxico (CH-404).
      Para N=1.000 con léxico de ~90 entradas, 5-10 es ideal; para N>=20.000 con 3.130 entradas usar 20.
    - exacto_n: si True, garantiza devolver exactamente `n` pares (reemplazando excedentes de cobertura
      sin perder cobertura mínima); si False, añade los pares extra de CH-404 al final.
    """
    assert lexico is not None, "Debe proporcionar un léxico con raíces generadas."
    lex_by_id = {e["id"]: e for e in lexico}
    gen = GeneradorIdeasChulupi(lexico, semilla=semilla)
    cuotas = gen.calcular_cuotas_estratos(n)

    ideas: List[Dict[str, Any]] = []
    hashes: Set[str] = set()
    estratos_ideas: List[Tuple[str, str, str, int]] = []

    # 1. Muestreo estratificado (CH-1104)
    for (pred, mod, tiempo, pers), cuota in cuotas:
        generadas = 0
        intentos = 0
        while generadas < cuota and intentos < cuota * 40 + 200:
            intentos += 1
            idea = gen.generar_idea_en_estrato(pred, mod, tiempo, pers)
            if idea is None:
                continue
            h = _hash_idea(idea)
            if h in hashes:
                continue
            hashes.add(h)
            ideas.append(idea)
            estratos_ideas.append((pred, mod, tiempo, pers))
            generadas += 1

    # Rellenar si algún estrato difícil dejó huecos
    rng_fill = random.Random(semilla + 999)
    estratos_lista = [k for k, _ in cuotas]
    while len(ideas) < n:
        pred, mod, tiempo, pers = rng_fill.choice(estratos_lista)
        idea = gen.generar_idea_en_estrato(pred, mod, tiempo, pers)
        if idea is None:
            continue
        h = _hash_idea(idea)
        if h not in hashes:
            hashes.add(h)
            ideas.append(idea)

    # 2. Cobertura mínima del léxico (CH-404)
    conteo = Counter()
    for idea in ideas:
        for lid in _lemas_usados_en_idea(idea):
            conteo[lid] += 1

    extras: List[Dict[str, Any]] = []
    if cobertura_minima > 0:
        for e in lexico:
            lid = e["id"]
            faltan = cobertura_minima - conteo[lid]
            if faltan <= 0:
                continue
            intentos = 0
            while faltan > 0 and intentos < 500:
                intentos += 1
                # Elegir estrato compatible con el tipo de lema
                if e["cat"].startswith("V_"):
                    pred, mod, tiempo, pers = "verbo", rng_fill.choice(["afirmativa", "negativa", "pregunta_si_no"]), rng_fill.choice(["SIMP", "FUT"]), rng_fill.choice([1, 2, 3, 4])
                    if mod == "negativa" and tiempo not in ("SIMP", "FUT"):
                        tiempo = "SIMP"
                elif e["cat"] == "ADV_T":
                    t_map = {"pasado": "SIMP", "presente": "SIMP", "futuro": "FUT"}
                    pred, mod, tiempo, pers = "verbo", "afirmativa", t_map[e["subcat"]], rng_fill.choice([1, 2, 3])
                elif e["cat"] == "ADJ":
                    pred, mod, tiempo, pers = rng_fill.choice([("ser", "afirmativa", "SIMP", 3), ("verbo", "afirmativa", "SIMP", 3)])
                elif e["clase_sem"] == "tiempo":
                    if int(e["silabas"]) == 2:
                        pred, mod, tiempo, pers = "verbo", "afirmativa", rng_fill.choice(["SIMP", "FUT"]), rng_fill.choice([1, 2, 3])
                    else:
                        pred, mod, tiempo, pers = "ser", rng_fill.choice(["afirmativa", "pregunta_si_no"]), "SIMP", 3
                else:
                    pred, mod, tiempo, pers = rng_fill.choice(estratos_lista)

                idea = gen.generar_idea_en_estrato(pred, mod, tiempo, pers, forzar_lema_id=lid)
                if idea is None:
                    continue
                usados = _lemas_usados_en_idea(idea)
                if lid not in usados:
                    continue
                h = _hash_idea(idea)
                if h in hashes:
                    continue
                hashes.add(h)
                extras.append(idea)
                for u in usados:
                    conteo[u] += 1
                faltan = cobertura_minima - conteo[lid]

    if extras:
        if exacto_n and len(extras) < len(ideas):
            # Reemplazar ideas redundantes (cuyos lemas están holgados por encima de cobertura_minima)
            reemplazadas = 0
            nuevas_ideas = []
            for idea in ideas:
                usados = _lemas_usados_en_idea(idea)
                if reemplazadas < len(extras) and all(conteo[u] > cobertura_minima for u in usados):
                    for u in usados:
                        conteo[u] -= 1
                    reemplazadas += 1
                else:
                    nuevas_ideas.append(idea)
            ideas = nuevas_ideas + extras[:reemplazadas]
            # Si aún faltan extras, añadir los restantes
            if reemplazadas < len(extras):
                ideas = (ideas + extras[reemplazadas:])[:n]
        else:
            ideas.extend(extras)

    # Mezclar (CH-404)
    rng_shuf = random.Random(semilla + 12345)
    rng_shuf.shuffle(ideas)

    # 3. Construir oraciones B, A, ES y empaquetar formato CH-1301
    pares: List[Dict[str, Any]] = []
    for idx, idea in enumerate(ideas, 1):
        idea_con_id = {"id": f"I{idx:06d}", **idea}
        res = construir_oracion(idea_con_id, lex_by_id)
        par = {
            "id": f"P{idx:08d}",
            "version_spec": VERSION_SPEC,
            "version_lexico": VERSION_LEXICO,
            "idea": idea_con_id,
            "b": res["b"],
            "a": res["a"],
            "es": res["es"],
            "palabras_b": res["palabras_b"],
            "dv": res["dv"],
        }
        pares.append(par)

    if ruta_jsonl:
        with open(ruta_jsonl, "w", encoding="utf-8") as f:
            for par in pares:
                f.write(json.dumps(par, ensure_ascii=False) + "\n")

    reporte = {}
    if verificar:
        reporte = verificar_dataset(
            pares, lexico, palabras_reales or [], modo_filtro=modo_filtro, cobertura_minima=cobertura_minima
        )

    return pares, reporte

print("Validador R1-R20, generador estratificado y verificador V01-V31 listos.")

## 10. Generar el Dataset Sintético Inicial (`N = 1.000` pares)

Parámetros principales de `generar_dataset_chulupi(...)`:
- `n=1000`: tamaño del dataset inicial (puedes subirlo luego a `10_000`, `50_000`, etc. cuando cargues el léxico completo de 3.130 entradas).
- `cobertura_minima=5`: mínimo de apariciones por entrada del léxico (`CH-404`). Para `n=1000` con 93 entradas garantiza el 100% de cobertura sin distorsionar la distribución; para `n >= 20_000` usar `20`.
- `verificar=True`: ejecuta automáticamente la batería `V01`–`V31` (`CH-1302`).

In [ ]:
RUTA_JSONL = "chulupi_dataset_1k.jsonl"

dataset_1k, reporte_1k = generar_dataset_chulupi(
    n=1000,
    lexico=LEXICO_CHULUPI,
    palabras_reales=PALABRAS_REALES_GUIA,
    modo_filtro="guia",
    semilla=42,
    cobertura_minima=5,
    exacto_n=True,
    verificar=True,
    ruta_jsonl=RUTA_JSONL,
)

print("=== REPORTE DE VERIFICACIÓN AUTOMÁTICA (V01-V31) ===")
print(f"Total de pares generados : {reporte_1k['total_pares']}")
print(f"Errores críticos (E)     : {reporte_1k['errores_count']}")
print(f"Cobertura léxica         : {reporte_1k['entradas_lexico_usadas']}/{reporte_1k['total_entradas_lexico']} entradas (mín={reporte_1k['cobertura_min_observada']} apariciones)")
print(f"Pares con [DERIVADA]     : {reporte_1k['prop_con_dv']:.1%} (tope orientativo V30: <=50%)")
print("Avisos (A)               :", reporte_1k["avisos"])
assert reporte_1k["errores_count"] == 0, f"Se encontraron errores críticos: {reporte_1k['errores_muestra']}"

print("\n--- Distribución predicado.modalidad ---")
for k, v in reporte_1k["distribucion_predicado_modalidad"].items():
    print(f"  {k:24s}: {v:6.1%}")

print("\n--- Detalle de reglas [DERIVADA] (tope V30 <=15% c/u) ---")
for dv, v in reporte_1k["dv_detalle"].items():
    print(f"  {dv}: {v:6.1%}")

# Vista previa tabular (usa pandas si está disponible en Colab)
try:
    import pandas as pd
    df_muestra = pd.DataFrame([
        {
            "id": p["id"],
            "tipo": f"{p['idea']['predicado']}.{p['idea']['modalidad']}",
            "tiempo": p["idea"]["tiempo"],
            "Modo B (aymara suf.)": p["b"],
            "Modo A (inventado)": p["a"],
            "Español": p["es"],
            "DV": ",".join(p["dv"]),
        }
        for p in dataset_1k[:15]
    ])
    try:
        from IPython.display import display
        display(df_muestra)
    except Exception:
        print(df_muestra.to_string(index=False))
except ImportError:
    for p in dataset_1k[:10]:
        print(f"[{p['id']}] B: {p['b']} | A: {p['a']} | ES: {p['es']}")

## 11. Exportar archivos listos para entrenamiento (TSV, JSONL y `lexico_chulupi.csv`)

Se generan 4 archivos en el directorio actual de Colab:
1. `chulupi_dataset_1k.jsonl`: formato completo de la especificación (`CH-1301`).
2. `chulupi_modo_B_1k.tsv`: pares `src_chulupi_B \t tgt_es` listos para NLLB / HuggingFace.
3. `chulupi_modo_A_1k.tsv`: pares `src_chulupi_A \t tgt_es` (lenguaje aglutinante completamente inventado).
4. `lexico_chulupi.csv`: diccionario fijo con las raíces generadas (`CH-309`).

In [ ]:
def exportar_archivos_entrenamiento(pares: List[Dict[str, Any]], lexico: List[Dict[str, Any]], prefijo: str = "chulupi"):
    ruta_b = f"{prefijo}_modo_B_{len(pares)}.tsv"
    ruta_a = f"{prefijo}_modo_A_{len(pares)}.tsv"
    ruta_lex = f"lexico_{prefijo}.csv"

    with open(ruta_b, "w", encoding="utf-8", newline="") as fb, open(ruta_a, "w", encoding="utf-8", newline="") as fa:
        fb.write("src\ttgt\tid\n")
        fa.write("src\ttgt\tid\n")
        for p in pares:
            fb.write(f"{p['b']}\t{p['es']}\t{p['id']}\n")
            fa.write(f"{p['a']}\t{p['es']}\t{p['id']}\n")

    campos_lex = list(lexico[0].keys())
    with open(ruta_lex, "w", encoding="utf-8", newline="") as fl:
        w = csv.DictWriter(fl, fieldnames=campos_lex)
        w.writeheader()
        w.writerows(lexico)

    print("Archivos exportados:")
    for r in [RUTA_JSONL, ruta_b, ruta_a, ruta_lex]:
        if os.path.exists(r):
            print(f"  - {r} ({os.path.getsize(r):,} bytes)")
    return [RUTA_JSONL, ruta_b, ruta_a, ruta_lex]

archivos_generados = exportar_archivos_entrenamiento(dataset_1k, LEXICO_CHULUPI)

# Descarga automática solo si se ejecuta dentro de Google Colab
try:
    from google.colab import files  # type: ignore
    for archivo in archivos_generados:
        files.download(archivo)
except ImportError:
    print("(Entorno local detectado; en Google Colab esta celda descargará los archivos automáticamente.)")